## 1. Imports and Setup

In [1]:
import os
import re
import json
from pathlib import Path
from dataclasses import dataclass, replace
from collections import defaultdict
from typing import Any, Dict, List, Optional, Tuple, Iterable

import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
from rasterio.mask import mask
from shapely.geometry import mapping, Polygon
from shapely.affinity import translate
import networkx as nx
from sklearn.neighbors import NearestNeighbors

SKIMAGE_AVAILABLE = True
try:
    from skimage.registration import phase_cross_correlation
    from skimage.transform import resize
except Exception as e:
    SKIMAGE_AVAILABLE = False
    print("⚠️ scikit-image not available. Phase correlation alignment will be skipped.")
    print(str(e))

In [2]:
@dataclass
class MatchCaseConfig:
    name: str
    base_similarity_weights: Dict[str, float]
    scoring_weights: Dict[str, float]
    similarity_threshold: float
    min_iou: float = 0.0
    min_overlap_prev: float = 0.0
    min_overlap_curr: float = 0.0
    max_centroid_dist: Optional[float] = None
    mutual_best: bool = False
    allow_multiple: bool = False
    max_edges_per_prev: Optional[int] = None
    max_edges_per_curr: Optional[int] = None


class TreeTrackingGraph:
    """
    Tree crown tracker across orthomosaics using a directed graph with ultra-relaxed parameters.
    
    Supports automatic file discovery, image extraction, spatial alignment, and configurable matching.
    """
    
    def __init__(
        self,
        crown_dir: Optional[str] = None,
        ortho_dir: Optional[str] = None,
        output_dir: str = '../../output',
        simplify_tol: float = 1.0,
        max_crowns_preview: int = 200,
        auto_discover: bool = True
    ) -> None:
        self.output_dir = output_dir
        self.simplify_tol = simplify_tol
        self.max_crowns_preview = max_crowns_preview
        self.crown_dir = crown_dir or self._resolve_dir('input/input_crowns', '../../input/input_crowns')
        self.ortho_dir = ortho_dir or self._resolve_dir('input/input_om', '../../input/input_om')
        
        self.file_pairs: List[Tuple[str, Optional[str]]] = []
        self.om_ids: List[int] = []
        self.crowns_gdfs: Dict[int, gpd.GeoDataFrame] = {}
        self.crown_attrs: Dict[int, List[Dict[str, Any]]] = {}
        self.crown_images: Dict[int, List[Optional[np.ndarray]]] = {}
        self.crown_crs: Dict[int, Optional[Any]] = {}
        self.ortho_crs: Dict[int, Optional[Any]] = {}
        self.G: nx.DiGraph = nx.DiGraph()
        self.match_case_mode: str = "balanced"
        
        # Use ultra-relaxed configs by default
        self.case_configs: Dict[str, MatchCaseConfig] = self._ultra_relaxed_case_configs()
        self.case_order: List[str] = ['one_to_one', 'containment', 'nearby']
        self.last_case_counts: Dict[str, int] = {}
        self.last_selected_counts: Dict[str, int] = {}
        
        if auto_discover:
            self.discover_files()
    
    @staticmethod
    def _resolve_dir(root_rel: str, nb_rel: str) -> str:
        """Resolve directory path from multiple possible locations."""
        candidates = [
            os.path.abspath(os.path.join(os.getcwd(), root_rel)),
            os.path.abspath(os.path.join(os.getcwd(), nb_rel)),
        ]
        for p in candidates:
            if os.path.isdir(p):
                return p
        raise FileNotFoundError(f"Could not resolve directory for {root_rel}. Tried: {candidates}")
    
    @staticmethod
    def _extract_numeric_id(name: str) -> Optional[int]:
        """Extract numeric ID from filename."""
        m = re.search(r"(\d+)", os.path.basename(name))
        return int(m.group(1)) if m else None
    
    def discover_files(self) -> None:
        """Automatically discover and pair crown GeoPackages with orthomosaics."""
        crown_files = [
            os.path.join(self.crown_dir, f) 
            for f in os.listdir(self.crown_dir) 
            if f.lower().endswith('.gpkg')
        ]
        ortho_files = [
            os.path.join(self.ortho_dir, f) 
            for f in os.listdir(self.ortho_dir) 
            if f.lower().endswith('.tif')
        ] if os.path.exists(self.ortho_dir) else []
        
        if not crown_files:
            raise FileNotFoundError(f"No .gpkg crown files found in {self.crown_dir}")
        
        # Group files by numeric ID
        crowns_by_id = {}
        for cf in crown_files:
            cid = self._extract_numeric_id(cf)
            crowns_by_id[cid if cid is not None else cf] = cf
        
        orthos_by_id = {}
        for of in ortho_files:
            oid = self._extract_numeric_id(of)
            orthos_by_id[oid if oid is not None else of] = of
        
        # Find matching numeric IDs
        numeric_ids = sorted(
            set(k for k in crowns_by_id.keys() if isinstance(k, int)) & 
            set(k for k in orthos_by_id.keys() if isinstance(k, int))
        )
        
        # Pair files
        file_pairs: List[Tuple[str, Optional[str]]] = []
        if numeric_ids:
            for nid in numeric_ids:
                file_pairs.append((crowns_by_id[nid], orthos_by_id.get(nid)))
            crown_only = sorted(
                k for k in crowns_by_id.keys() 
                if isinstance(k, int) and k not in numeric_ids
            )
            for nid in crown_only:
                file_pairs.append((crowns_by_id[nid], None))
        else:
            crown_files_sorted = sorted(crown_files)
            ortho_files_sorted = sorted(ortho_files)
            for i, cf in enumerate(crown_files_sorted):
                of = ortho_files_sorted[i] if i < len(ortho_files_sorted) else None
                file_pairs.append((cf, of))
        
        # Extract OM IDs
        om_ids: List[int] = []
        for cf, _ in file_pairs:
            cid = self._extract_numeric_id(cf)
            om_ids.append(cid if cid is not None else len(om_ids) + 1)
        
        # Sort by OM ID
        pairs_with_id = sorted(
            [(oid, cf, of) for oid, (cf, of) in zip(om_ids, file_pairs)],
            key=lambda x: x[0]
        )
        self.file_pairs = [(cf, of) for _, cf, of in pairs_with_id]
        self.om_ids = [oid for oid, _, _ in pairs_with_id]
    
    def load_data(
        self,
        load_images: bool = False,
        align: bool = False,
        reference_om_id: Optional[int] = None
    ) -> None:
        """
        Load crown data from GeoPackages and optionally extract image patches.
        
        CRITICAL: Images are extracted from ORIGINAL geometries BEFORE alignment.
        This ensures image-geometry correspondence is preserved.
        
        Args:
            load_images: If True, extract RGB image patches from orthomosaics for each crown
            align: If True, align all OMs to reference OM using translational shift
            reference_om_id: OM ID to use as alignment reference (default: first OM)
        """
        self.crowns_gdfs.clear()
        self.crown_attrs.clear()
        self.crown_images.clear()
        self.crown_crs.clear()
        self.ortho_crs.clear()
        
        print(f"\n{'='*70}")
        print("LOADING DATA")
        print(f"{'='*70}")
        
        # Step 1: Load geometries and extract images from ORIGINAL positions
        for om_id, (crown_file, ortho_file) in zip(self.om_ids, self.file_pairs):
            print(f"\nOM{om_id}: Loading {os.path.basename(crown_file)}")
            gdf = gpd.read_file(crown_file)
            self.crowns_gdfs[om_id] = gdf
            self.crown_crs[om_id] = gdf.crs
            
            # Extract images from ORIGINAL geometries BEFORE any shifting
            if load_images and ortho_file and os.path.exists(ortho_file):
                print(f"  Extracting {len(gdf)} crown images from {os.path.basename(ortho_file)}...")
                with rasterio.open(ortho_file) as src:
                    self.ortho_crs[om_id] = src.crs
                    patches: List[Optional[np.ndarray]] = []
                    for _, row in gdf.iterrows():
                        geom = [mapping(row.geometry)]
                        try:
                            out_image, _ = mask(src, geom, crop=True)
                            img_patch = np.moveaxis(out_image, 0, -1)  # (C, H, W) -> (H, W, C)
                        except Exception:
                            img_patch = None
                        patches.append(img_patch)
                self.crown_images[om_id] = patches
                print(f"  ✓ Extracted {sum(1 for p in patches if p is not None)} images")
            else:
                self.crown_images[om_id] = [None] * len(gdf)
                if ortho_file and os.path.exists(ortho_file):
                    with rasterio.open(ortho_file) as src:
                        self.ortho_crs[om_id] = src.crs
                else:
                    self.ortho_crs[om_id] = None
            
            # Compute attributes from original geometries
            self.crown_attrs[om_id] = [
                self._compute_crown_attributes(row.geometry) 
                for _, row in gdf.iterrows()
            ]
        
        self._check_crs_consistency()
        print(f"\n{'='*70}")
        
        # Step 2: Apply alignment if requested (geometries shift, images stay with original crowns)
        if align:
            print(f"\n{'='*70}")
            print("APPLYING SPATIAL ALIGNMENT")
            print(f"{'='*70}")
            self.alignment_shifts = self.align_to_reference(reference_om_id=reference_om_id)
            print(f"{'='*70}\n")
    
    def _check_crs_consistency(self) -> None:
        """Check CRS consistency across crown files and orthomosaics."""
        print("CRS CONSISTENCY CHECK")
        print("-" * 60)
        crown_crs_values = {om_id: crs for om_id, crs in self.crown_crs.items()}
        ortho_crs_values = {om_id: crs for om_id, crs in self.ortho_crs.items()}
        crown_crs_set = {str(crs) for crs in crown_crs_values.values() if crs is not None}
        if not crown_crs_set:
            print("⚠️  Crown CRS: None detected in all crown files")
        elif len(crown_crs_set) > 1:
            print("⚠️  Crown CRS mismatch across OMs:")
            for om_id, crs in crown_crs_values.items():
                print(f"  OM{om_id}: {crs}")
        else:
            print(f"✓ Crown CRS consistent: {next(iter(crown_crs_set))}")
        
        for om_id in self.om_ids:
            crown_crs = crown_crs_values.get(om_id)
            ortho_crs = ortho_crs_values.get(om_id)
            if crown_crs is None:
                print(f"⚠️  OM{om_id}: crown CRS is None")
            if ortho_crs is None:
                print(f"⚠️  OM{om_id}: ortho CRS is None or ortho missing")
            if crown_crs is not None and ortho_crs is not None and crown_crs != ortho_crs:
                print(f"⚠️  OM{om_id}: crown CRS != ortho CRS ({crown_crs} vs {ortho_crs})")
        print("-" * 60)
    
    def align_to_reference(
        self,
        reference_om_id: Optional[int] = None,
        threshold: float = 20.0,
        min_matches: int = 5
    ) -> Dict[int, Tuple[float, float]]:
        """
        Align all OMs to a reference OM using mutual nearest neighbors and robust median shift.
        
        Args:
            reference_om_id: OM ID to use as reference (default: first OM)
            threshold: Maximum distance for considering a match (meters)
            min_matches: Minimum number of matches required
        
        Returns:
            Dictionary mapping om_id -> (dx, dy) shift applied
        """
        if not self.crowns_gdfs:
            raise ValueError("No crown data loaded. Call load_data() first.")
        
        if reference_om_id is None:
            reference_om_id = self.om_ids[0]
        if reference_om_id not in self.crowns_gdfs:
            raise ValueError(f"Reference OM {reference_om_id} not found in loaded data.")
        
        ref = self.crowns_gdfs[reference_om_id]
        ref_centroids = np.array([[g.centroid.x, g.centroid.y] for g in ref.geometry])
        shifts = {reference_om_id: (0.0, 0.0)}
        
        print(f"Aligning all OMs to reference OM{reference_om_id}")
        print(f"  Reference crowns: {len(ref_centroids)}")
        print()
        
        nn_ref = NearestNeighbors(n_neighbors=1).fit(ref_centroids)

        for om_id in self.om_ids:
            if om_id == reference_om_id:
                continue
            curr = self.crowns_gdfs[om_id].copy()
            curr_centroids = np.array([[g.centroid.x, g.centroid.y] for g in curr.geometry])
            if len(curr_centroids) == 0:
                print(f"OM{om_id}: no crowns, skipping alignment")
                shifts[om_id] = (0.0, 0.0)
                continue
            
            dist_to_ref, idx_ref = nn_ref.kneighbors(curr_centroids)
            nn_curr = NearestNeighbors(n_neighbors=1).fit(curr_centroids)
            dist_to_curr, idx_curr = nn_curr.kneighbors(ref_centroids)
            
            mutual = []
            for i, (d, ref_idx) in enumerate(zip(dist_to_ref[:, 0], idx_ref[:, 0])):
                if d < threshold and idx_curr[ref_idx][0] == i and dist_to_curr[ref_idx][0] < threshold:
                    mutual.append((ref_idx, i, d))
            
            if len(mutual) < min_matches:
                relaxed = [(idx_ref[i][0], i, dist_to_ref[i][0]) for i in range(len(dist_to_ref)) if dist_to_ref[i][0] < threshold * 1.5]
                mutual = mutual + relaxed
            
            if len(mutual) < min_matches:
                print(f"OM{om_id}: insufficient matches ({len(mutual)}/{min_matches}), skipping alignment")
                shifts[om_id] = (0.0, 0.0)
                continue
            
            shifts_array = np.array([ref_centroids[a] - curr_centroids[b] for a, b, _ in mutual])
            median_shift = np.median(shifts_array, axis=0)
            residuals = np.linalg.norm(shifts_array - median_shift, axis=1)
            median_resid = float(np.median(residuals)) if len(residuals) else 0.0
            mad = float(np.median(np.abs(residuals - median_resid))) or 1e-6
            inliers = residuals <= median_resid + 3.0 * mad
            
            if not np.any(inliers):
                inliers = np.ones_like(residuals, dtype=bool)
            
            dx, dy = np.median(shifts_array[inliers], axis=0)
            print(f"OM{om_id}: matches={len(mutual)}, inliers={np.sum(inliers)}, dx={dx:.2f}, dy={dy:.2f}, med_res={median_resid:.2f}")
            
            curr['geometry'] = curr['geometry'].apply(lambda g: translate(g, xoff=dx, yoff=dy))
            self.crowns_gdfs[om_id] = curr
            self.crown_attrs[om_id] = [
                self._compute_crown_attributes(row.geometry) 
                for _, row in curr.iterrows()
            ]
            shifts[om_id] = (float(dx), float(dy))
        
        return shifts
    
    @staticmethod
    def _compute_crown_attributes(geometry) -> Dict[str, Any]:
        """Compute geometric attributes for a crown polygon."""
        centroid = geometry.centroid
        area = geometry.area
        perimeter = geometry.length
        bounds = geometry.bounds
        compactness = (4 * np.pi * area) / (perimeter ** 2) if perimeter > 0 else 0
        
        try:
            min_rect = geometry.minimum_rotated_rectangle
            coords = list(min_rect.exterior.coords)
            side1 = np.linalg.norm(np.array(coords[0]) - np.array(coords[1]))
            side2 = np.linalg.norm(np.array(coords[1]) - np.array(coords[2]))
            major_axis = max(side1, side2)
            minor_axis = min(side1, side2)
            eccentricity = minor_axis / major_axis if major_axis > 0 else 1
        except Exception:
            eccentricity = 1
        
        aspect_ratio = (bounds[3] - bounds[1]) / (bounds[2] - bounds[0]) if bounds[2] != bounds[0] else 1
        
        return {
            'geometry': geometry,
            'centroid': centroid,
            'area': area,
            'perimeter': perimeter,
            'compactness': compactness,
            'eccentricity': eccentricity,
            'aspect_ratio': aspect_ratio,
            'bounds': bounds,
        }
    
    def _ultra_relaxed_case_configs(self) -> Dict[str, MatchCaseConfig]:
        """
        ULTRA RELAXED matching parameters - prioritizes spatial proximity over shape similarity.
        
        Designed to bridge difficult OM transitions with minimal overlap or shape changes.
        """
        return {
            'one_to_one': MatchCaseConfig(
                name='one_to_one',
                base_similarity_weights={
                    'spatial': 0.50,  # Strong emphasis on proximity
                    'area': 0.15,
                    'shape': 0.05,    # De-emphasize shape
                    'iou': 0.30
                },
                scoring_weights={
                    'base': 0.60, 
                    'iou': 0.10, 
                    'overlap_prev': 0.05, 
                    'overlap_curr': 0.05, 
                    'centroid': 0.20
                },
                similarity_threshold=0.25,
                min_iou=0.01,
                min_overlap_prev=0.10,
                min_overlap_curr=0.10,
                max_centroid_dist=50.0,
                mutual_best=False,
                allow_multiple=True,
                max_edges_per_prev=3,
                max_edges_per_curr=3,
            ),
            'containment': MatchCaseConfig(
                name='containment',
                base_similarity_weights={
                    'spatial': 0.50,
                    'area': 0.20,
                    'shape': 0.10,
                    'iou': 0.20
                },
                scoring_weights={
                    'base': 0.40, 
                    'overlap_prev': 0.30, 
                    'overlap_curr': 0.30
                },
                similarity_threshold=0.25,
                min_iou=0.0,
                min_overlap_prev=0.25,
                min_overlap_curr=0.25,
                max_centroid_dist=150.0,
                mutual_best=False,
                allow_multiple=True,
                max_edges_per_prev=2,
                max_edges_per_curr=2,
            ),
            'nearby': MatchCaseConfig(
                name='nearby',
                base_similarity_weights={
                    'spatial': 0.85,  # Almost purely spatial
                    'area': 0.10,
                    'shape': 0.03,
                    'iou': 0.02
                },
                scoring_weights={
                    'base': 0.70, 
                    'centroid': 0.30
                },
                similarity_threshold=0.20,
                min_iou=0.0,
                min_overlap_prev=0.05,
                min_overlap_curr=0.05,
                max_centroid_dist=200.0,
                mutual_best=False,
                allow_multiple=True,
                max_edges_per_prev=2,
                max_edges_per_curr=2,
            ),
        }
    
    @staticmethod
    def _compute_iou(g1, g2) -> float:
        """Compute intersection over union for two geometries."""
        try:
            intersection = g1.intersection(g2).area
            union = g1.union(g2).area
        except Exception:
            intersection = 0.0
            union = g1.area + g2.area
        return intersection / union if union > 0 else 0.0
    
    def _weighted_similarity(
        self,
        a1: Dict[str, Any],
        a2: Dict[str, Any],
        weights: Optional[Dict[str, float]] = None,
        max_dist: float = 100.0
    ) -> Tuple[float, Dict[str, float]]:
        """Compute weighted similarity score between two crown attributes."""
        if weights is None:
            weights = {'spatial': 0.4, 'area': 0.2, 'shape': 0.2, 'iou': 0.2}
        
        # Spatial similarity
        centroid_dist = a1['centroid'].distance(a2['centroid'])
        spatial_sim = max(0.0, 1.0 - (centroid_dist / max_dist))
        
        # Area similarity
        area_sim = min(a1['area'], a2['area']) / max(a1['area'], a2['area']) if max(a1['area'], a2['area']) > 0 else 0.0
        
        # Shape similarity
        compactness_sim = 1.0 - abs(a1['compactness'] - a2['compactness'])
        eccentricity_sim = 1.0 - abs(a1['eccentricity'] - a2['eccentricity'])
        shape_sim = (compactness_sim + eccentricity_sim) / 2.0
        
        # IoU similarity
        iou_sim = self._compute_iou(a1['geometry'], a2['geometry'])
        
        # Weighted total
        total = (
            weights.get('spatial', 0.0) * spatial_sim +
            weights.get('area', 0.0) * area_sim +
            weights.get('shape', 0.0) * shape_sim +
            weights.get('iou', 0.0) * iou_sim
        )
        
        return total, {
            'spatial': float(spatial_sim),
            'area': float(area_sim),
            'shape': float(shape_sim),
            'iou': float(iou_sim),
            'total': float(total)
        }
    
    def _compute_pair_metrics(
        self,
        prev_attrs: Dict[str, Any],
        curr_attrs: Dict[str, Any],
        max_dist: float
    ) -> Dict[str, float]:
        """Compute comprehensive metrics for a crown pair."""
        prev_geom = prev_attrs['geometry']
        curr_geom = curr_attrs['geometry']
        
        try:
            intersection_area = prev_geom.intersection(curr_geom).area
        except Exception:
            intersection_area = 0.0
        
        try:
            union_area = prev_geom.union(curr_geom).area
        except Exception:
            union_area = prev_attrs['area'] + curr_attrs['area'] - intersection_area
        
        prev_area = prev_attrs['area'] if prev_attrs['area'] > 0 else 1e-6
        curr_area = curr_attrs['area'] if curr_attrs['area'] > 0 else 1e-6
        
        overlap_prev = intersection_area / prev_area
        overlap_curr = intersection_area / curr_area
        iou = intersection_area / union_area if union_area > 0 else 0.0
        
        centroid_dist = prev_attrs['centroid'].distance(curr_attrs['centroid'])
        base_similarity, parts = self._weighted_similarity(prev_attrs, curr_attrs, max_dist=max_dist)
        
        prev_radius = np.sqrt(prev_area / np.pi)
        curr_radius = np.sqrt(curr_area / np.pi)
        mean_radius = max((prev_radius + curr_radius) / 2.0, 1e-3)
        
        area_ratio = curr_area / prev_area if prev_area > 0 else np.inf
        if not np.isfinite(area_ratio) or area_ratio <= 0:
            balanced_area_ratio = 0.0
        else:
            balanced_area_ratio = area_ratio if area_ratio <= 1 else 1 / area_ratio
        
        try:
            prev_contains_curr = prev_geom.buffer(0).contains(curr_geom)
        except Exception:
            prev_contains_curr = False
        
        try:
            curr_contains_prev = curr_geom.buffer(0).contains(prev_geom)
        except Exception:
            curr_contains_prev = False
        
        return {
            'intersection_area': float(intersection_area),
            'union_area': float(union_area),
            'overlap_prev': float(overlap_prev),
            'overlap_curr': float(overlap_curr),
            'iou': float(iou),
            'centroid_dist': float(centroid_dist),
            'base_similarity': float(base_similarity),
            'spatial_similarity': float(parts['spatial']),
            'area_similarity': float(parts['area']),
            'shape_similarity': float(parts['shape']),
            'mean_radius': float(mean_radius),
            'area_ratio': float(area_ratio if np.isfinite(area_ratio) else 0.0),
            'balanced_area_ratio': float(balanced_area_ratio),
            'prev_contains_curr': bool(prev_contains_curr),
            'curr_contains_prev': bool(curr_contains_prev),
        }
    
    def _classify_match_case(
        self,
        prev_node: Tuple[int, int],
        curr_node: Tuple[int, int],
        features: Dict[str, float],
        prev_overlap_counts: Dict[Tuple[int, int], int],
        curr_overlap_counts: Dict[Tuple[int, int], int],
        overlap_gate: float,
        mode: str = "balanced"
    ) -> str:
        """Classify the type of match between two crowns."""
        if features['prev_contains_curr'] or features['curr_contains_prev']:
            return 'containment'
        
        overlap_prev = features['overlap_prev']
        overlap_curr = features['overlap_curr']
        iou = features['iou']
        centroid_dist = features['centroid_dist']
        prev_count = prev_overlap_counts.get(prev_node, 0)
        curr_count = curr_overlap_counts.get(curr_node, 0)
        
        if mode == "lite":
            min_overlap_one_to_one = max(overlap_gate, 0.10)
            min_iou_one_to_one = max(overlap_gate * 0.5, 0.04)
            if (overlap_prev >= min_overlap_one_to_one and
                overlap_curr >= min_overlap_one_to_one and
                iou >= min_iou_one_to_one and
                centroid_dist < 50.0):
                return 'one_to_one'
            near_gate = max(overlap_gate * 0.5, 0.01)
            if centroid_dist < 50.0 and (overlap_prev >= near_gate or overlap_curr >= near_gate):
                return 'nearby'
        elif mode == "strict":
            min_overlap_for_one_to_one = max(overlap_gate, 0.30)
            min_iou_for_one_to_one = max(overlap_gate / 2, 0.10)
            if (prev_count == 1 and curr_count == 1 and 
                overlap_prev >= min_overlap_for_one_to_one and 
                overlap_curr >= min_overlap_for_one_to_one and 
                iou >= min_iou_for_one_to_one):
                return 'one_to_one'
            if centroid_dist < 30.0 and (overlap_prev >= overlap_gate or overlap_curr >= overlap_gate):
                return 'nearby'
        else:
            min_overlap_one_to_one = max(overlap_gate, 0.15)
            min_iou_one_to_one = max(overlap_gate * 0.5, 0.05)
            if (overlap_prev >= min_overlap_one_to_one and
                overlap_curr >= min_overlap_one_to_one and
                iou >= min_iou_one_to_one and
                centroid_dist < 40.0):
                return 'one_to_one'
            near_gate = max(overlap_gate * 0.5, 0.02)
            if centroid_dist < 35.0 and (overlap_prev >= near_gate or overlap_curr >= near_gate):
                return 'nearby'
        
        return 'none'
    
    def _score_candidate(
        self,
        base_similarity: float,
        similarity_parts: Dict[str, float],
        features: Dict[str, float],
        config: MatchCaseConfig
    ) -> float:
        """Score a candidate match using weighted components."""
        centroid_factor = 1.0 - min(1.0, features['centroid_dist'] / (features['mean_radius'] * 3.0))
        
        components = {
            'base': base_similarity,
            'spatial': similarity_parts.get('spatial', 0.0),
            'area': similarity_parts.get('area', 0.0),
            'shape': similarity_parts.get('shape', 0.0),
            'iou': features['iou'],
            'overlap_prev': features['overlap_prev'],
            'overlap_curr': features['overlap_curr'],
            'centroid': max(0.0, centroid_factor),
            'area_balance': features.get('balanced_area_ratio', 0.0),
        }
        
        score = 0.0
        for key, weight in config.scoring_weights.items():
            score += weight * components.get(key, 0.0)
        
        return score
    
    def _select_candidates_by_case(
        self,
        candidates: List[Dict[str, Any]],
        configs: Dict[str, MatchCaseConfig],
        case_order: List[str],
        max_dist: float,
        min_base_similarity: float = 0.0
    ) -> List[Dict[str, Any]]:
        """Select best candidates for each case in priority order."""
        selected: List[Dict[str, Any]] = []
        used_prev: Dict[Tuple[int, int], int] = defaultdict(int)
        used_curr: Dict[Tuple[int, int], int] = defaultdict(int)
        
        for case_name in case_order:
            config = configs.get(case_name)
            if not config:
                continue
            
            case_candidates = [cand for cand in candidates if cand['case'] == case_name]
            if not case_candidates:
                continue
            
            # Enrich candidates with scores
            enriched: List[Dict[str, Any]] = []
            for cand in case_candidates:
                prev_attrs = cand['prev_attrs']
                curr_attrs = cand['curr_attrs']
                features = cand['features']
                
                # Apply filters
                if config.max_centroid_dist is not None and features['centroid_dist'] > config.max_centroid_dist:
                    continue
                if features['iou'] < config.min_iou:
                    continue
                if features['overlap_prev'] < config.min_overlap_prev:
                    continue
                if features['overlap_curr'] < config.min_overlap_curr:
                    continue
                
                # Compute score (case-specific weights)
                base_similarity, parts = self._weighted_similarity(
                    prev_attrs, curr_attrs,
                    weights=config.base_similarity_weights,
                    max_dist=max_dist
                )
                if min_base_similarity and base_similarity < min_base_similarity:
                    continue
                score = self._score_candidate(base_similarity, parts, features, config)
                
                if score < config.similarity_threshold:
                    continue
                
                cand['base_similarity'] = float(base_similarity)
                cand['similarity_parts'] = {k: float(v) for k, v in parts.items()}
                cand['score'] = float(score)
                enriched.append(cand)
            
            if not enriched:
                continue
            
            # Select candidates (allow multiple edges when configured)
            if config.mutual_best:
                best_prev: Dict[Tuple[int, int], Dict[str, Any]] = {}
                best_curr: Dict[Tuple[int, int], Dict[str, Any]] = {}
                
                for cand in enriched:
                    prev_node = cand['prev_node']
                    curr_node = cand['curr_node']
                    
                    if not config.allow_multiple:
                        if used_prev.get(prev_node, 0) or used_curr.get(curr_node, 0):
                            continue
                    
                    if prev_node not in best_prev or cand['score'] > best_prev[prev_node]['score']:
                        best_prev[prev_node] = cand
                    if curr_node not in best_curr or cand['score'] > best_curr[curr_node]['score']:
                        best_curr[curr_node] = cand
                
                for cand in enriched:
                    prev_node = cand['prev_node']
                    curr_node = cand['curr_node']
                    
                    if best_prev.get(prev_node) is cand and best_curr.get(curr_node) is cand:
                        if not config.allow_multiple:
                            if used_prev.get(prev_node, 0) or used_curr.get(curr_node, 0):
                                continue
                        
                        if config.max_edges_per_prev is not None and used_prev[prev_node] >= config.max_edges_per_prev:
                            continue
                        if config.max_edges_per_curr is not None and used_curr[curr_node] >= config.max_edges_per_curr:
                            continue
                        
                        selected.append(cand)
                        used_prev[prev_node] += 1
                        used_curr[curr_node] += 1
            else:
                enriched.sort(key=lambda c: c['score'], reverse=True)
                
                for cand in enriched:
                    prev_node = cand['prev_node']
                    curr_node = cand['curr_node']
                    
                    if not config.allow_multiple:
                        if used_prev.get(prev_node, 0) or used_curr.get(curr_node, 0):
                            continue
                    
                    if config.max_edges_per_prev is not None and used_prev[prev_node] >= config.max_edges_per_prev:
                        continue
                    if config.max_edges_per_curr is not None and used_curr[curr_node] >= config.max_edges_per_curr:
                        continue
                    
                    selected.append(cand)
                    used_prev[prev_node] += 1
                    used_curr[curr_node] += 1
        
        return selected
    
    def reset_graph(self) -> None:
        """Clear the graph."""
        self.G = nx.DiGraph()
    
    def build_graph_conditional(
        self,
        case_configs: Optional[Dict[str, MatchCaseConfig]] = None,
        case_order: Optional[List[str]] = None,
        base_max_dist: float = 200.0,
        overlap_gate: float = 0.05,
        min_base_similarity: float = 0.0,
        max_candidates_per_prev: Optional[int] = None,
        max_candidates_per_curr: Optional[int] = None,
        classify_mode: Optional[str] = None
    ) -> None:
        """
        Build tracking graph using conditional case-based matching.
        
        Args:
            case_configs: Dictionary of case configurations (default: ultra-relaxed)
            case_order: Priority order for cases (default: ['one_to_one', 'containment', 'nearby'])
            base_max_dist: Maximum centroid distance for candidate matches (meters)
            overlap_gate: Minimum overlap fraction to count as overlapping
            min_base_similarity: Minimum base similarity (case-specific weights) for candidate filtering
            max_candidates_per_prev: Maximum candidates to consider per previous crown
            max_candidates_per_curr: Maximum candidates to consider per current crown
            classify_mode: one of {'balanced','lite','strict'} (default: self.match_case_mode)
        """
        if not self.crowns_gdfs:
            self.load_data(load_images=False)
        
        self.reset_graph()
        
        configs = {name: replace(cfg) for name, cfg in (case_configs or self.case_configs).items()}
        order = case_order or self.case_order
        mode = classify_mode or self.match_case_mode
        
        self.last_case_counts = {}
        self.last_selected_counts = {name: 0 for name in configs.keys()}
        
        print(f"\n{'='*70}")
        print("BUILDING TRACKING GRAPH")
        print(f"{'='*70}")
        print(f"Parameters:")
        print(f"  base_max_dist: {base_max_dist}m")
        print(f"  overlap_gate: {overlap_gate}")
        print(f"  min_base_similarity: {min_base_similarity}")
        print(f"  classify_mode: {mode}")
        print(f"  case_order: {', '.join(order)}")
        print()
        
        # Add all nodes
        for idx in range(len(self.om_ids)):
            om_id = self.om_ids[idx]
            gdf = self.crowns_gdfs[om_id]
            
            for crown_id, row in gdf.iterrows():
                attrs = self.crown_attrs[om_id][crown_id]
                self.G.add_node((om_id, crown_id), **attrs)
            
            if idx == 0:
                continue
            
            # Build edges between consecutive OMs
            prev_om = self.om_ids[idx - 1]
            prev_nodes = [(prev_om, i) for i in range(len(self.crowns_gdfs[prev_om]))]
            curr_nodes = [(om_id, j) for j in range(len(gdf))]
            
            print(f"OM{prev_om} → OM{om_id}: {len(prev_nodes)} × {len(curr_nodes)} potential pairs")
            
            # Generate candidates
            candidates: List[Dict[str, Any]] = []
            overlap_counts_prev: Dict[Tuple[int, int], int] = defaultdict(int)
            overlap_counts_curr: Dict[Tuple[int, int], int] = defaultdict(int)
            
            for prev_node in prev_nodes:
                prev_attrs = self.G.nodes[prev_node]
                
                for curr_node in curr_nodes:
                    curr_attrs = self.crown_attrs[om_id][curr_node[1]]
                    
                    features = self._compute_pair_metrics(prev_attrs, curr_attrs, max_dist=base_max_dist)
                    
                    # Early filtering (distance only)
                    if features['centroid_dist'] > base_max_dist:
                        continue
                    
                    cand = {
                        'prev_node': prev_node,
                        'curr_node': curr_node,
                        'prev_attrs': prev_attrs,
                        'curr_attrs': curr_attrs,
                        'features': features,
                    }
                    candidates.append(cand)
                    
                    if features['overlap_prev'] >= overlap_gate:
                        overlap_counts_prev[prev_node] += 1
                    if features['overlap_curr'] >= overlap_gate:
                        overlap_counts_curr[curr_node] += 1
            
            if not candidates:
                print(f"  No candidates found")
                continue
            
            # Classify candidates
            for cand in candidates:
                cand['case'] = self._classify_match_case(
                    cand['prev_node'], cand['curr_node'], cand['features'],
                    overlap_counts_prev, overlap_counts_curr, overlap_gate, mode=mode
                )
            
            candidates = [cand for cand in candidates if cand['case'] != 'none']
            
            if not candidates:
                print(f"  No valid cases found")
                continue
            
            # Trim candidates if requested
            if max_candidates_per_prev is not None:
                grouped_prev: Dict[Tuple[int, int], List[Dict[str, Any]]] = defaultdict(list)
                for cand in candidates:
                    grouped_prev[cand['prev_node']].append(cand)
                
                trimmed: List[Dict[str, Any]] = []
                for group in grouped_prev.values():
                    group.sort(key=lambda c: (-c['features']['iou'], c['features']['centroid_dist']))
                    trimmed.extend(group[:max_candidates_per_prev])
                candidates = trimmed
            
            if max_candidates_per_curr is not None:
                grouped_curr: Dict[Tuple[int, int], List[Dict[str, Any]]] = defaultdict(list)
                for cand in candidates:
                    grouped_curr[cand['curr_node']].append(cand)
                
                trimmed_curr: List[Dict[str, Any]] = []
                for group in grouped_curr.values():
                    group.sort(key=lambda c: (-c['features']['iou'], c['features']['centroid_dist']))
                    trimmed_curr.extend(group[:max_candidates_per_curr])
                candidates = trimmed_curr
            
            # Count candidates by case
            case_counts = defaultdict(int)
            for cand in candidates:
                case_counts[cand['case']] += 1
            
            for case_name, count in case_counts.items():
                self.last_case_counts[case_name] = self.last_case_counts.get(case_name, 0) + count
            
            print(f"  Candidates by case: {dict(case_counts)}")
            
            # Select best candidates
            selected = self._select_candidates_by_case(
                candidates, configs, order, base_max_dist, min_base_similarity=min_base_similarity
            )
            
            # Add edges to graph
            for cand in selected:
                case_name = cand['case']
                features = cand['features']
                similarity_parts = cand.get('similarity_parts', {})
                
                self.G.add_edge(
                    cand['prev_node'],
                    cand['curr_node'],
                    similarity=float(cand.get('score', features['base_similarity'])),
                    method='conditional',
                    case=case_name,
                    overlap_prev=float(features['overlap_prev']),
                    overlap_curr=float(features['overlap_curr']),
                    iou=float(features['iou']),
                    centroid_distance=float(features['centroid_dist']),
                    base_similarity=float(cand.get('base_similarity', features['base_similarity'])),
                    spatial_similarity=float(similarity_parts.get('spatial', features['spatial_similarity'])),
                    area_similarity=float(similarity_parts.get('area', features['area_similarity'])),
                    shape_similarity=float(similarity_parts.get('shape', features['shape_similarity']))
                )
                
                self.last_selected_counts[case_name] = self.last_selected_counts.get(case_name, 0) + 1
            
            print(f"  Selected: {len(selected)} edges")
            print()
        
        print(f"{'='*70}")
        print(f"Graph built: {self.G.number_of_nodes()} nodes, {self.G.number_of_edges()} edges")
        print(f"{'='*70}\n")
    
    def debug_candidate_flow(
        self,
        base_max_dist: float,
        overlap_gate: float,
        min_base_similarity: float = 0.0,
        case_configs: Optional[Dict[str, MatchCaseConfig]] = None,
        case_order: Optional[List[str]] = None,
        classify_mode: Optional[str] = None,
        max_candidates_per_prev: Optional[int] = None,
        max_candidates_per_curr: Optional[int] = None
    ) -> Dict[str, Any]:
        """
        Diagnose candidate filtering stages and report potential early-discard risks.
        """
        if not self.crowns_gdfs:
            raise ValueError("No crown data loaded. Call load_data() first.")
        configs = {name: replace(cfg) for name, cfg in (case_configs or self.case_configs).items()}
        order = case_order or self.case_order
        mode = classify_mode or self.match_case_mode
        
        summary: Dict[str, Any] = {
            'pairs': {},
            'legacy_prefilter_discards': 0,
            'legacy_prefilter_discards_that_would_pass': 0
        }
        
        for idx in range(1, len(self.om_ids)):
            prev_om = self.om_ids[idx - 1]
            curr_om = self.om_ids[idx]
            prev_nodes = [(prev_om, i) for i in range(len(self.crowns_gdfs[prev_om]))]
            curr_nodes = [(curr_om, j) for j in range(len(self.crowns_gdfs[curr_om]))]
            total_pairs = len(prev_nodes) * len(curr_nodes)
            
            candidates = []
            overlap_counts_prev: Dict[Tuple[int, int], int] = defaultdict(int)
            overlap_counts_curr: Dict[Tuple[int, int], int] = defaultdict(int)
            
            dist_pass = 0
            for prev_node in prev_nodes:
                prev_attrs = self.crown_attrs[prev_om][prev_node[1]]
                for curr_node in curr_nodes:
                    curr_attrs = self.crown_attrs[curr_om][curr_node[1]]
                    features = self._compute_pair_metrics(prev_attrs, curr_attrs, max_dist=base_max_dist)
                    if features['centroid_dist'] > base_max_dist:
                        continue
                    dist_pass += 1
                    cand = {
                        'prev_node': prev_node,
                        'curr_node': curr_node,
                        'prev_attrs': prev_attrs,
                        'curr_attrs': curr_attrs,
                        'features': features,
                    }
                    candidates.append(cand)
                    if features['overlap_prev'] >= overlap_gate:
                        overlap_counts_prev[prev_node] += 1
                    if features['overlap_curr'] >= overlap_gate:
                        overlap_counts_curr[curr_node] += 1
            
            for cand in candidates:
                cand['case'] = self._classify_match_case(
                    cand['prev_node'], cand['curr_node'], cand['features'],
                    overlap_counts_prev, overlap_counts_curr, overlap_gate, mode=mode
                )
            classified = [c for c in candidates if c['case'] != 'none']
            
            # Trim candidates if requested
            if max_candidates_per_prev is not None:
                grouped_prev: Dict[Tuple[int, int], List[Dict[str, Any]]] = defaultdict(list)
                for cand in classified:
                    grouped_prev[cand['prev_node']].append(cand)
                trimmed: List[Dict[str, Any]] = []
                for group in grouped_prev.values():
                    group.sort(key=lambda c: (-c['features']['iou'], c['features']['centroid_dist']))
                    trimmed.extend(group[:max_candidates_per_prev])
                classified = trimmed
            
            if max_candidates_per_curr is not None:
                grouped_curr: Dict[Tuple[int, int], List[Dict[str, Any]]] = defaultdict(list)
                for cand in classified:
                    grouped_curr[cand['curr_node']].append(cand)
                trimmed_curr: List[Dict[str, Any]] = []
                for group in grouped_curr.values():
                    group.sort(key=lambda c: (-c['features']['iou'], c['features']['centroid_dist']))
                    trimmed_curr.extend(group[:max_candidates_per_curr])
                classified = trimmed_curr
            
            # Score with case-specific weights
            scored = 0
            for cand in classified:
                cfg = configs.get(cand['case'])
                if not cfg:
                    continue
                features = cand['features']
                if cfg.max_centroid_dist is not None and features['centroid_dist'] > cfg.max_centroid_dist:
                    continue
                if features['iou'] < cfg.min_iou:
                    continue
                if features['overlap_prev'] < cfg.min_overlap_prev:
                    continue
                if features['overlap_curr'] < cfg.min_overlap_curr:
                    continue
                base_similarity, parts = self._weighted_similarity(
                    cand['prev_attrs'], cand['curr_attrs'],
                    weights=cfg.base_similarity_weights, max_dist=base_max_dist
                )
                if min_base_similarity and base_similarity < min_base_similarity:
                    continue
                score = self._score_candidate(base_similarity, parts, features, cfg)
                if score < cfg.similarity_threshold:
                    continue
                scored += 1
                
                # legacy prefilter test (old behavior)
                legacy_reject = features['base_similarity'] < min_base_similarity and features['iou'] < overlap_gate
                if legacy_reject:
                    summary['legacy_prefilter_discards'] += 1
                    summary['legacy_prefilter_discards_that_would_pass'] += 1
            
            summary['pairs'][f"{prev_om}->{curr_om}"] = {
                'total_pairs': total_pairs,
                'within_distance': dist_pass,
                'classified_candidates': len(classified),
                'case_scored_pass': scored,
            }
        
        return summary
    
    def _extract_all_chains(self) -> List[List[Tuple[int, int]]]:
        """Extract all tracking chains from the graph."""
        visited = set()
        chains: List[List[Tuple[int, int]]] = []
        
        # Start from nodes with no predecessors
        chain_starts = [n for n in self.G.nodes if not list(self.G.predecessors(n))]
        
        for start_node in chain_starts:
            if start_node in visited:
                continue
            
            chain = self._greedy_chain(start_node)
            chains.append(chain)
            visited.update(chain)
        
        # Handle remaining nodes (cycles or isolated)
        remaining = set(self.G.nodes) - visited
        for node in remaining:
            chains.append([node])
        
        return chains
    
    def _greedy_chain(self, start_node: Tuple[int, int]) -> List[Tuple[int, int]]:
        """Extract a single chain starting from a given node."""
        chain = [start_node]
        current = start_node
        
        while True:
            successors = list(self.G.successors(current))
            if not successors:
                break
            
            if len(successors) > 1:
                # Choose best successor
                best_successor = max(
                    successors,
                    key=lambda n: self.G[current][n].get('similarity', 0.0)
                )
                chain.append(best_successor)
                current = best_successor
            else:
                chain.append(successors[0])
                current = successors[0]
        
        return chain
    
    def quality_report(self) -> Tuple[str, Dict[str, Any]]:
        """Generate quality metrics and report."""
        G = self.G
        om_ids = self.om_ids
        
        metrics: Dict[str, Any] = {
            'total_trees_detected': G.number_of_nodes(),
            'total_edges': G.number_of_edges(),
            'total_possible_matches': 0,
            'successful_matches': 0,
            'match_rate_by_om_pair': {},
            'chain_length_distribution': {},
            'average_chain_length': 0,
            'median_chain_length': 0,
            'max_chain_length': 0,
        }
        
        # Chain analysis
        chains = self._extract_all_chains()
        chain_lengths = [len(chain) for chain in chains]
        
        if chain_lengths:
            metrics['average_chain_length'] = float(np.mean(chain_lengths))
            metrics['median_chain_length'] = float(np.median(chain_lengths))
            metrics['max_chain_length'] = int(max(chain_lengths))
            
            for length in chain_lengths:
                metrics['chain_length_distribution'][int(length)] = metrics['chain_length_distribution'].get(int(length), 0) + 1
        
        # Match rate by OM pair
        for i in range(len(om_ids) - 1):
            om1, om2 = om_ids[i], om_ids[i + 1]
            om1_nodes = [n for n in G.nodes if n[0] == om1]
            om2_nodes = [n for n in G.nodes if n[0] == om2]
            
            matches = sum(1 for u, v in G.edges() if u[0] == om1 and v[0] == om2)
            possible_matches = min(len(om1_nodes), len(om2_nodes))
            match_rate = matches / possible_matches if possible_matches > 0 else 0.0
            
            metrics['match_rate_by_om_pair'][f"{om1}->{om2}"] = {
                'matches': matches,
                'possible': possible_matches,
                'rate': float(match_rate),
            }
            
            metrics['total_possible_matches'] += possible_matches
            metrics['successful_matches'] += matches
        
        metrics['overall_match_rate'] = (
            metrics['successful_matches'] / metrics['total_possible_matches']
            if metrics['total_possible_matches'] > 0 else 0.0
        )
        
        # Generate report
        report = [
            "# Tree Tracking Quality Assessment Report",
            f"Total Trees Detected: {metrics['total_trees_detected']}",
            f"Total Tracking Edges: {metrics['total_edges']}",
            f"Overall Match Rate: {metrics['overall_match_rate']:.3f}",
            f"Average Chain Length: {metrics.get('average_chain_length', 0):.2f}",
            f"Maximum Chain Length: {metrics.get('max_chain_length', 0)}",
            "\nMatch Rates by Orthomosaic Pair:",
        ]
        
        for pair, data in metrics['match_rate_by_om_pair'].items():
            report.append(f"- {pair}: {data['matches']}/{data['possible']} ({data['rate']:.3f})")
        
        report.append("\nChain Length Distribution:")
        for length, count in sorted(metrics['chain_length_distribution'].items()):
            report.append(f"- Length {length}: {count} trees")
        
        if self.last_selected_counts:
            report.append("\nEdge selection by case:")
            for case_name, count in sorted(self.last_selected_counts.items(), key=lambda kv: (-kv[1], kv[0])):
                total_candidates = self.last_case_counts.get(case_name, 0)
                if total_candidates:
                    ratio = count / total_candidates
                    report.append(f"- {case_name}: {count} / {total_candidates} ({ratio:.2f})")
                else:
                    report.append(f"- {case_name}: {count}")
        
        return "\n".join(report), metrics
    
    def get_matching_chain(self, start_om_id: int, crown_id: int) -> List[Tuple[int, int]]:
        """Get the tracking chain starting from a specific crown."""
        node = (start_om_id, crown_id)
        if node not in self.G:
            raise ValueError(f"Node {(start_om_id, crown_id)} not in graph. Build the graph first.")
        return self._greedy_chain(node)
    
    def ensure_output_dir(self) -> None:
        """Create output directory if it doesn't exist."""
        os.makedirs(self.output_dir, exist_ok=True)
    
    def save_text(self, text: str, filename: str) -> str:
        """Save text to file in output directory."""
        self.ensure_output_dir()
        path = os.path.join(self.output_dir, filename)
        with open(path, 'w') as f:
            f.write(text)
        return path
    
    def save_json(self, data: Dict[str, Any], filename: str) -> str:
        """Save JSON data to file in output directory."""
        self.ensure_output_dir()
        path = os.path.join(self.output_dir, filename)
        with open(path, 'w') as f:
            json.dump(data, f, indent=2)
        return path

print("✓ TreeTrackingGraph class defined with configurable alignment and match modes")

✓ TreeTrackingGraph class defined with configurable alignment and match modes


In [3]:
# ---- Phase-correlation alignment (orthomosaics) ----
def _read_ortho_lowres(path: str, max_size: int = 1200):
    with rasterio.open(path) as src:
        scale = max(src.width, src.height) / max_size if max(src.width, src.height) > max_size else 1.0
        out_w = int(round(src.width / scale))
        out_h = int(round(src.height / scale))
        data = src.read([1, 2, 3], out_shape=(3, out_h, out_w), resampling=rasterio.enums.Resampling.bilinear)
        rgb = np.moveaxis(data, 0, -1).astype(np.float32)
        rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-6)
        gray = 0.2989 * rgb[..., 0] + 0.5870 * rgb[..., 1] + 0.1140 * rgb[..., 2]
        scale_x = src.width / out_w
        scale_y = src.height / out_h
        transform = src.transform * rasterio.transform.Affine.scale(scale_x, scale_y)
        return rgb, gray, transform

def _bounds_from_transform(h: int, w: int, transform: rasterio.transform.Affine):
    corners = [(0, 0), (0, w), (h, 0), (h, w)]
    xs = []
    ys = []
    for row, col in corners:
        x, y = rasterio.transform.xy(transform, row, col, offset='ul')
        xs.append(x)
        ys.append(y)
    left, right = min(xs), max(xs)
    bottom, top = min(ys), max(ys)
    return left, bottom, right, top

def _safe_window(win: rasterio.windows.Window, height: int, width: int) -> rasterio.windows.Window:
    col_off = int(max(0, min(width, win.col_off)))
    row_off = int(max(0, min(height, win.row_off)))
    max_w = max(0, width - col_off)
    max_h = max(0, height - row_off)
    w = int(max(0, min(max_w, win.width)))
    h = int(max(0, min(max_h, win.height)))
    return rasterio.windows.Window(col_off=col_off, row_off=row_off, width=w, height=h)

def _crop_to_overlap(ref_img, mov_img, ref_transform, mov_transform):
    ref_h, ref_w = ref_img.shape[:2]
    mov_h, mov_w = mov_img.shape[:2]
    ref_left, ref_bottom, ref_right, ref_top = _bounds_from_transform(ref_h, ref_w, ref_transform)
    mov_left, mov_bottom, mov_right, mov_top = _bounds_from_transform(mov_h, mov_w, mov_transform)
    left = max(ref_left, mov_left)
    right = min(ref_right, mov_right)
    bottom = max(ref_bottom, mov_bottom)
    top = min(ref_top, mov_top)
    if left >= right or bottom >= top:
        return None, None
    ref_win = rasterio.windows.from_bounds(left, bottom, right, top, ref_transform).round_offsets().round_lengths()
    mov_win = rasterio.windows.from_bounds(left, bottom, right, top, mov_transform).round_offsets().round_lengths()
    ref_win = _safe_window(ref_win, ref_h, ref_w)
    mov_win = _safe_window(mov_win, mov_h, mov_w)
    if ref_win.width < 2 or ref_win.height < 2 or mov_win.width < 2 or mov_win.height < 2:
        return None, None
    ref_crop = ref_img[int(ref_win.row_off): int(ref_win.row_off + ref_win.height),
                       int(ref_win.col_off): int(ref_win.col_off + ref_win.width)]
    mov_crop = mov_img[int(mov_win.row_off): int(mov_win.row_off + mov_win.height),
                       int(mov_win.col_off): int(mov_win.col_off + mov_win.width)]
    return ref_crop, mov_crop

def _match_shape(ref_crop, mov_crop):
    if ref_crop.shape == mov_crop.shape:
        return ref_crop, mov_crop
    if not SKIMAGE_AVAILABLE:
        min_h = min(ref_crop.shape[0], mov_crop.shape[0])
        min_w = min(ref_crop.shape[1], mov_crop.shape[1])
        return ref_crop[:min_h, :min_w], mov_crop[:min_h, :min_w]
    mov_resized = resize(mov_crop, ref_crop.shape, preserve_range=True, anti_aliasing=True)
    return ref_crop, mov_resized

def _phase_corr_shift(ref_gray, mov_gray, ref_transform, mov_transform):
    ref_crop, mov_crop = _crop_to_overlap(ref_gray, mov_gray, ref_transform, mov_transform)
    if ref_crop is None or mov_crop is None:
        return None, None
    ref_crop, mov_crop = _match_shape(ref_crop, mov_crop)
    shift, error, _ = phase_cross_correlation(ref_crop, mov_crop, upsample_factor=10)
    shift_row, shift_col = shift
    dx = shift_col * ref_transform.a
    dy = shift_row * ref_transform.e
    return (float(dx), float(dy)), {'shift_px': (float(shift_row), float(shift_col)), 'error': float(error)}

def phase_corr_align_tracker(tracker: 'TreeTrackingGraph', reference_om_id: Optional[int] = None, max_preview: int = 1200):
    if not SKIMAGE_AVAILABLE:
        raise RuntimeError('scikit-image not available; cannot run phase correlation alignment.')
    if reference_om_id is None:
        reference_om_id = tracker.om_ids[0]
    # Load low-res orthos
    ortho_gray = {}
    ortho_transform = {}
    for om_id, (_, ortho_file) in zip(tracker.om_ids, tracker.file_pairs):
        if ortho_file and os.path.exists(ortho_file):
            _, gray, transform = _read_ortho_lowres(ortho_file, max_preview)
            ortho_gray[om_id] = gray
            ortho_transform[om_id] = transform
        else:
            ortho_gray[om_id] = None
            ortho_transform[om_id] = None
    shifts = {reference_om_id: (0.0, 0.0)}
    # Build cumulative shifts across consecutive OMs
    for idx in range(1, len(tracker.om_ids)):
        prev_id = tracker.om_ids[idx - 1]
        curr_id = tracker.om_ids[idx]
        ref_gray = ortho_gray.get(prev_id)
        mov_gray = ortho_gray.get(curr_id)
        ref_transform = ortho_transform.get(prev_id)
        mov_transform = ortho_transform.get(curr_id)
        if ref_gray is None or mov_gray is None or ref_transform is None or mov_transform is None:
            shifts[curr_id] = shifts.get(prev_id, (0.0, 0.0))
            continue
        out = _phase_corr_shift(ref_gray, mov_gray, ref_transform, mov_transform)
        if out[0] is None:
            shifts[curr_id] = shifts.get(prev_id, (0.0, 0.0))
            continue
        (dx, dy), meta = out
        prev_dx, prev_dy = shifts.get(prev_id, (0.0, 0.0))
        shifts[curr_id] = (prev_dx + dx, prev_dy + dy)
    # Apply shifts
    for om_id in tracker.om_ids:
        if om_id == reference_om_id:
            continue
        dx, dy = shifts.get(om_id, (0.0, 0.0))
        gdf = tracker.crowns_gdfs[om_id].copy()
        gdf['geometry'] = gdf['geometry'].apply(lambda g: translate(g, xoff=dx, yoff=dy))
        tracker.crowns_gdfs[om_id] = gdf
        tracker.crown_attrs[om_id] = [tracker._compute_crown_attributes(row.geometry) for _, row in gdf.iterrows()]
    tracker.alignment_shifts = shifts
    return shifts

## 3. Visualization Functions

Functions for visualizing tracking chains with polygons and extracted RGB images.

In [4]:
def visualize_chain_with_extracted_images(chain, aligned_gdfs, crown_images_dict, tracker, title="Chain Example", save_path=None, show=True, close_fig=True, dpi=150):
    """
    Visualize chain with both crown polygons AND extracted RGB images.
    Optional saving for PPT-friendly reuse.

    Args:
        chain: List of (om_id, crown_idx) tuples
        aligned_gdfs: Dictionary of aligned GeoDataFrames by OM ID
        crown_images_dict: Dictionary of crown images by OM ID (from tracker.crown_images)
        tracker: TreeTrackingGraph instance
        title: Plot title
        save_path: Optional path to save the figure
        show: Whether to display the plot
        close_fig: Close the figure after saving/showing to free memory
        dpi: Resolution when saving
    """
    chain_length = len(chain)
    
    # Create figure with 2 rows: polygons on top, images on bottom
    fig = plt.figure(figsize=(5*chain_length, 10))
    
    print(f"\n{'='*80}")
    print(f"{title}")
    print(f"{'='*80}")
    print(f"Chain: {chain}")
    print(f"Length: {chain_length}\n")
    
    # Plot each crown in the chain
    for idx, (om_idx, crown_idx) in enumerate(chain):
        gdf = aligned_gdfs[om_idx]
        crown = gdf.iloc[crown_idx]
        
        # Get edge info if not last crown
        edge_info = ""
        if idx < chain_length - 1:
            next_node = chain[idx + 1]
            if tracker.G.has_edge((om_idx, crown_idx), next_node):
                edge_data = tracker.G.edges[(om_idx, crown_idx), next_node]
                edge_info = f" → OM{next_node[0]}[{next_node[1]}]: case={edge_data['case']}, sim={edge_data['similarity']:.2f}, iou={edge_data.get('iou', 0):.2f}"
        
        # Print crown info
        in_deg = tracker.G.in_degree((om_idx, crown_idx))
        out_deg = tracker.G.out_degree((om_idx, crown_idx))
        print(f"  OM{om_idx}[{crown_idx}]: in_deg={in_deg}, out_deg={out_deg}, area={crown.geometry.area:.1f}m²{edge_info}")
        
        # ROW 1: Crown polygon
        ax_poly = plt.subplot(2, chain_length, idx + 1)
        
        # Get bounds and plot
        minx, miny, maxx, maxy = crown.geometry.bounds
        margin = max((maxx - minx), (maxy - miny)) * 0.3
        
        # Plot other crowns in gray
        gdf.plot(ax=ax_poly, color='lightgray', edgecolor='gray', alpha=0.3)
        
        # Plot this crown highlighted
        gpd.GeoSeries([crown.geometry]).plot(
            ax=ax_poly,
            facecolor=plt.cm.tab10(om_idx-1),
            edgecolor='black',
            linewidth=2,
            alpha=0.7
        )
        
        # Plot centroid
        centroid = crown.geometry.centroid
        ax_poly.plot(centroid.x, centroid.y, 'o', color='yellow', markersize=8,
                    markeredgecolor='black', markeredgewidth=1.5)
        
        # Arrow to next crown
        if idx < chain_length - 1:
            ax_poly.annotate('', xy=(maxx + margin*0.5, (miny+maxy)/2),
                            xytext=(maxx + margin*0.2, (miny+maxy)/2),
                            arrowprops=dict(arrowstyle='->', lw=3, color='red'))
        
        ax_poly.set_xlim(minx - margin, maxx + margin)
        ax_poly.set_ylim(miny - margin, maxy + margin)
        ax_poly.set_aspect('equal')
        ax_poly.set_title(
            f"OM{om_idx} - Crown {crown_idx}\nArea: {crown.geometry.area:.1f}m²\nIn:{in_deg} Out:{out_deg}",
            fontsize=10, fontweight='bold'
        )
        ax_poly.set_xlabel("X (m)", fontsize=9)
        ax_poly.set_ylabel("Y (m)", fontsize=9)
        ax_poly.grid(True, alpha=0.3)
        
        # Add edge case label
        if edge_info:
            edge_data = tracker.G.edges[(om_idx, crown_idx), chain[idx + 1]]
            bbox_color = {
                'one_to_one': 'wheat',
                'containment': 'lightblue',
                'nearby': 'lightcoral'
            }.get(edge_data['case'], 'white')
            ax_poly.text(
                0.95, 0.95,
                f"→ {edge_data['case']}\nSim: {edge_data['similarity']:.2f}\nIoU: {edge_data.get('iou', 0):.2f}",
                transform=ax_poly.transAxes, fontsize=8,
                verticalalignment='top', horizontalalignment='right',
                bbox=dict(boxstyle='round', facecolor=bbox_color, alpha=0.8)
            )
        
        # ROW 2: Extracted RGB image
        ax_img = plt.subplot(2, chain_length, chain_length + idx + 1)
        
        rgb_image = crown_images_dict[om_idx][crown_idx]
        if rgb_image is not None:
            ax_img.imshow(rgb_image)
            ax_img.set_title(
                f"Extracted Tree Image\n{rgb_image.shape[0]}×{rgb_image.shape[1]} px",
                fontsize=9
            )
        else:
            ax_img.text(0.5, 0.5, 'Image not available',
                       ha='center', va='center', fontsize=10, color='red')
            ax_img.set_title("No Image", fontsize=9)
        
        ax_img.axis('off')
    
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=dpi, bbox_inches='tight')
    if show:
        plt.show()
    if close_fig:
        plt.close(fig)
    print()
    return save_path


def categorize_chains(tracker):
    """
    Categorize all chains from the tracker.
    
    Returns:
        Dictionary with categorized chains
    """
    all_chains = tracker._extract_all_chains()
    max_chain_length = len(tracker.om_ids)
    
    full_chains_width_1 = []      # Full length chains with no branching
    full_chains_branching = []    # Full length chains with branching
    partial_chains_long = []      # Length 3-4
    partial_chains_short = []     # Length 2
    
    for chain in all_chains:
        chain_length = len(chain)
        
        if chain_length == max_chain_length:
            # Full chain - check for branching
            has_branching = False
            for node in chain:
                in_degree = tracker.G.in_degree(node)
                out_degree = tracker.G.out_degree(node)
                if in_degree > 1 or out_degree > 1:
                    has_branching = True
                    break
            
            if has_branching:
                full_chains_branching.append(chain)
            else:
                full_chains_width_1.append(chain)
        else:
            # Partial chains
            if chain_length >= 3:
                partial_chains_long.append(chain)
            elif chain_length == 2:
                partial_chains_short.append(chain)
    
    return {
        'full_width_1': full_chains_width_1,
        'full_branching': full_chains_branching,
        'partial_long': partial_chains_long,
        'partial_short': partial_chains_short,
        'singleton': [c for c in all_chains if len(c) == 1]
    }


print("✓ Visualization functions loaded")

✓ Visualization functions loaded


In [5]:
# Helper to build tuned configs without monkey-patching
def make_case_configs(mode="balanced"):
    if mode == "lite":
        return {
            'one_to_one': MatchCaseConfig(
                name='one_to_one',
                base_similarity_weights={'spatial': 0.35, 'area': 0.15, 'shape': 0.15, 'iou': 0.35},
                scoring_weights={'base': 0.40, 'iou': 0.20, 'overlap_prev': 0.15, 'overlap_curr': 0.15, 'centroid': 0.10},
                similarity_threshold=0.26,
                min_iou=0.04,
                min_overlap_prev=0.12,
                min_overlap_curr=0.12,
                max_centroid_dist=55.0,
                mutual_best=False,
                allow_multiple=True,
                max_edges_per_prev=2,
                max_edges_per_curr=2,
            ),
            'nearby': MatchCaseConfig(
                name='nearby',
                base_similarity_weights={'spatial': 0.80, 'area': 0.10, 'shape': 0.05, 'iou': 0.05},
                scoring_weights={'base': 0.48, 'centroid': 0.30, 'iou': 0.10, 'overlap_prev': 0.06, 'overlap_curr': 0.06},
                similarity_threshold=0.12,
                min_iou=0.0,
                min_overlap_prev=0.02,
                min_overlap_curr=0.02,
                max_centroid_dist=70.0,
                mutual_best=False,
                allow_multiple=True,
                max_edges_per_prev=2,
                max_edges_per_curr=2,
            ),
            'containment': MatchCaseConfig(
                name='containment',
                base_similarity_weights={'spatial': 0.40, 'area': 0.20, 'shape': 0.10, 'iou': 0.30},
                scoring_weights={'base': 0.35, 'overlap_prev': 0.30, 'overlap_curr': 0.30, 'iou': 0.05},
                similarity_threshold=0.22,
                min_iou=0.0,
                min_overlap_prev=0.18,
                min_overlap_curr=0.18,
                max_centroid_dist=100.0,
                mutual_best=False,
                allow_multiple=True,
                max_edges_per_prev=2,
                max_edges_per_curr=2,
            ),
        }
    
    # balanced (default)
    return {
        'one_to_one': MatchCaseConfig(
            name='one_to_one',
            base_similarity_weights={'spatial': 0.35, 'area': 0.15, 'shape': 0.15, 'iou': 0.35},
            scoring_weights={'base': 0.40, 'iou': 0.20, 'overlap_prev': 0.15, 'overlap_curr': 0.15, 'centroid': 0.10},
            similarity_threshold=0.28,
            min_iou=0.04,
            min_overlap_prev=0.15,
            min_overlap_curr=0.15,
            max_centroid_dist=45.0,
            mutual_best=False,
            allow_multiple=True,
            max_edges_per_prev=2,
            max_edges_per_curr=2,
        ),
        'nearby': MatchCaseConfig(
            name='nearby',
            base_similarity_weights={'spatial': 0.80, 'area': 0.10, 'shape': 0.05, 'iou': 0.05},
            scoring_weights={'base': 0.48, 'centroid': 0.30, 'iou': 0.10, 'overlap_prev': 0.06, 'overlap_curr': 0.06},
            similarity_threshold=0.12,
            min_iou=0.0,
            min_overlap_prev=0.02,
            min_overlap_curr=0.02,
            max_centroid_dist=70.0,
            mutual_best=False,
            allow_multiple=True,
            max_edges_per_prev=2,
            max_edges_per_curr=2,
        ),
        'containment': MatchCaseConfig(
            name='containment',
            base_similarity_weights={'spatial': 0.40, 'area': 0.20, 'shape': 0.10, 'iou': 0.30},
            scoring_weights={'base': 0.35, 'overlap_prev': 0.30, 'overlap_curr': 0.30, 'iou': 0.05},
            similarity_threshold=0.22,
            min_iou=0.0,
            min_overlap_prev=0.20,
            min_overlap_curr=0.20,
            max_centroid_dist=100.0,
            mutual_best=False,
            allow_multiple=True,
            max_edges_per_prev=2,
            max_edges_per_curr=2,
        ),
    }

print("✓ Config factory loaded")

✓ Config factory loaded


In [6]:
# Choose classification mode: 'balanced', 'lite', or 'strict'
match_case_mode = "balanced"
print(f"✓ Match case mode set to: {match_case_mode}")

✓ Match case mode set to: balanced


## 4. Initialize and Load Data

Initialize the tracker and load crown data with image extraction.

In [7]:
# Initialize the tracker
tracker = TreeTrackingGraph(
    crown_dir='../../output/input_crowns',
    ortho_dir='../../input/input_om',
    output_dir='../../output',
    auto_discover=True
)

print(f"\nDiscovered {len(tracker.file_pairs)} orthomosaic pairs")
print(f"OM IDs: {tracker.om_ids}")
print("\nFile pairs:")
for om_id, (crown_file, ortho_file) in zip(tracker.om_ids, tracker.file_pairs):
    print(f"  OM{om_id}: {os.path.basename(crown_file)}, {os.path.basename(ortho_file) if ortho_file else 'N/A'}")


Discovered 7 orthomosaic pairs
OM IDs: [1, 2, 3, 4, 5, 6, 7]

File pairs:
  OM1: OM1.gpkg, sit_om1.tif
  OM2: OM2.gpkg, sit_om2.tif
  OM3: OM3.gpkg, sit_om3.tif
  OM4: OM4.gpkg, sit_om4.tif
  OM5: OM5.gpkg, sit_om5.tif
  OM6: OM6.gpkg, sit_om6.tif
  OM7: OM7.gpkg, sit_om7.tif


In [8]:
# Apply tuned configs (branching allowed) and classification mode
tracker.match_case_mode = match_case_mode
tracker.case_configs = make_case_configs(match_case_mode)
tracker.case_order = ['one_to_one', 'nearby', 'containment']
print("✓ Tracker configured for branching-aware matching")

✓ Tracker configured for branching-aware matching


In [9]:
# Load crown data once, extract RGB images, and align via phase correlation
tracker.load_data(load_images=True, align=False)
phase_corr_align_tracker(tracker, reference_om_id=tracker.om_ids[0], max_preview=1200)
print(f"\n✓ Loaded data from {len(tracker.om_ids)} OMs")
print(f"✓ Extracted images: {sum(sum(img is not None for img in imgs) for imgs in tracker.crown_images.values())} total")


LOADING DATA

OM1: Loading OM1.gpkg
  Extracting 82 crown images from sit_om1.tif...
  ✓ Extracted 82 images

OM2: Loading OM2.gpkg
  Extracting 85 crown images from sit_om2.tif...
  ✓ Extracted 85 images

OM3: Loading OM3.gpkg
  Extracting 78 crown images from sit_om3.tif...
  ✓ Extracted 78 images

OM4: Loading OM4.gpkg
  Extracting 80 crown images from sit_om4.tif...
  ✓ Extracted 80 images

OM5: Loading OM5.gpkg
  Extracting 83 crown images from sit_om5.tif...
  ✓ Extracted 83 images

OM6: Loading OM6.gpkg
  Extracting 93 crown images from sit_om6.tif...
  ✓ Extracted 93 images

OM7: Loading OM7.gpkg
  Extracting 95 crown images from sit_om7.tif...
  ✓ Extracted 95 images
CRS CONSISTENCY CHECK
------------------------------------------------------------
✓ Crown CRS consistent: EPSG:32643
------------------------------------------------------------


✓ Loaded data from 7 OMs
✓ Extracted images: 596 total


## 6. Build Tracking Graph

Build the crown tracking graph with ultra-relaxed parameters.

In [10]:
# Build the tracking graph with tuned configs (branching allowed)
tracker.case_order = ['one_to_one', 'nearby', 'containment']

tracker.build_graph_conditional(
    base_max_dist=50,
    overlap_gate=0.02,
    min_base_similarity=0.1,
    max_candidates_per_prev=3,
    max_candidates_per_curr=3,
    classify_mode=tracker.match_case_mode,
    case_configs=tracker.case_configs,
    case_order=tracker.case_order,
)

# Generate quality report
report, metrics = tracker.quality_report()
print(report)

# Save report
tracker.save_text(report, 'crown_tracking_quality_report.txt')
tracker.save_json(metrics, 'crown_tracking_metrics.json')


BUILDING TRACKING GRAPH
Parameters:
  base_max_dist: 50m
  overlap_gate: 0.02
  min_base_similarity: 0.1
  classify_mode: balanced
  case_order: one_to_one, nearby, containment

OM1 → OM2: 82 × 85 potential pairs
  Candidates by case: {'one_to_one': 63, 'nearby': 16, 'containment': 1}
  Selected: 74 edges

OM2 → OM3: 85 × 78 potential pairs
  Candidates by case: {'one_to_one': 56, 'nearby': 22, 'containment': 3}
  Selected: 73 edges

OM3 → OM4: 78 × 80 potential pairs
  Candidates by case: {'one_to_one': 53, 'nearby': 11, 'containment': 2}
  Selected: 60 edges

OM4 → OM5: 80 × 83 potential pairs
  Candidates by case: {'one_to_one': 57, 'nearby': 10}
  Selected: 62 edges

OM5 → OM6: 83 × 93 potential pairs
  Candidates by case: {'one_to_one': 53, 'nearby': 16}
  Selected: 61 edges

OM6 → OM7: 93 × 95 potential pairs
  Candidates by case: {'one_to_one': 69, 'nearby': 8, 'containment': 2}
  Selected: 73 edges

Graph built: 596 nodes, 403 edges

# Tree Tracking Quality Assessment Report
T

'../../output/crown_tracking_metrics.json'

In [ ]:
# # Example: Visualize a specific tracking chain
# example_chain = tracker.get_matching_chain(start_om_id=1, crown_id=0)
# if len(example_chain) > 1:
#     visualize_chain_with_extracted_images(
#         example_chain, 
#         tracker.crowns_gdfs, 
#         tracker.crown_images, 
#         tracker, 
#         title="Example Tracking Chain"
#     )
# else:
#     print("No chain found for OM1, Crown 0")

## 7. Analyze Tracking Results

Categorize and analyze the tracking chains.

In [11]:
# Categorize chains
chain_categories = categorize_chains(tracker)

print("Chain Analysis:")
print(f"Full chains (no branching): {len(chain_categories['full_width_1'])}")
print(f"Full chains (with branching): {len(chain_categories['full_branching'])}")
print(f"Partial chains (long): {len(chain_categories['partial_long'])}")
print(f"Partial chains (short): {len(chain_categories['partial_short'])}")
print(f"Singletons: {len(chain_categories['singleton'])}")

# Show example of each category
for category, chains in chain_categories.items():
    if chains:
        print(f"\nExample {category} chain: {chains[0]}")

Chain Analysis:
Full chains (no branching): 9
Full chains (with branching): 11
Partial chains (long): 56
Partial chains (short): 51
Singletons: 163

Example full_width_1 chain: [(1, 9), (2, 13), (3, 33), (4, 45), (5, 47), (6, 1), (7, 30)]

Example full_branching chain: [(1, 10), (2, 56), (3, 61), (4, 27), (5, 49), (6, 79), (7, 88)]

Example partial_long chain: [(1, 0), (2, 60), (3, 42), (4, 38), (5, 34)]

Example partial_short chain: [(1, 19), (2, 37)]

Example singleton chain: [(1, 3)]


In [ ]:
# # Visualize all 5 crown sets: aligned vs non-aligned for comparison
# import matplotlib.pyplot as plt
# import matplotlib.patches as mpatches

# # Load original non-aligned crowns for comparison
# tracker_original = TreeTrackingGraph(
#     crown_dir='../../output/input_crowns',
#     ortho_dir='../../input/input_om',
#     output_dir='../../output',
#     auto_discover=True
# )
# tracker_original.load_data(load_images=False, align=False)  # No alignment

# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(24, 12))

# colors = ['red', 'blue', 'green', 'orange', 'purple']

# # Plot 1: Non-aligned crowns
# legend_handles = []
# for i, om_id in enumerate(tracker_original.om_ids):
#     gdf = tracker_original.crowns_gdfs[om_id]
#     gdf.plot(ax=ax1, facecolor='none', edgecolor=colors[i], linewidth=2, alpha=0.8)
#     legend_handles.append(mpatches.Patch(edgecolor=colors[i], facecolor='none', linewidth=2, label=f'OM{om_id}'))

# ax1.set_title('Crown Sets Before Spatial Alignment (Original Positions)')
# ax1.legend(handles=legend_handles)
# ax1.set_aspect('equal')
# ax1.set_xlabel('X (m)')
# ax1.set_ylabel('Y (m)')
# ax1.grid(True, alpha=0.3)

# # Plot 2: Aligned crowns
# legend_handles = []
# for i, om_id in enumerate(tracker.om_ids):
#     gdf = tracker.crowns_gdfs[om_id]
#     gdf.plot(ax=ax2, facecolor='none', edgecolor=colors[i], linewidth=2, alpha=0.8)
#     legend_handles.append(mpatches.Patch(edgecolor=colors[i], facecolor='none', linewidth=2, label=f'OM{om_id}'))

# ax2.set_title('Crown Sets After Spatial Alignment (Corrected Positions)')
# ax2.legend(handles=legend_handles)
# ax2.set_aspect('equal')
# ax2.set_xlabel('X (m)')
# ax2.set_ylabel('Y (m)')
# ax2.grid(True, alpha=0.3)

# plt.tight_layout()
# plt.show()

# print("Alignment comparison: Left shows original positions, right shows after spatial corrections.")
# print("If alignment worked, crowns should be better aligned spatially on the right.")

In [ ]:
# # Visualize all complete 5-length tracking chains
# print("Visualizing all complete 5-length tracking chains...")

# # Get all full chains (both no-branching and with-branching)
# full_chains = chain_categories['full_width_1'] + chain_categories['full_branching']

# print(f"Found {len(full_chains)} complete chains to visualize")

# for i, chain in enumerate(full_chains):
#     chain_type = "No Branching" if chain in chain_categories['full_width_1'] else "With Branching"
#     title = f"Complete Chain {i+1}/{len(full_chains)} - {chain_type}"

#     visualize_chain_with_extracted_images(
#         chain,
#         tracker.crowns_gdfs,
#         tracker.crown_images,
#         tracker,
#         title=title
#     )

# print(f"✓ Completed visualization of all {len(full_chains)} complete tracking chains")

## 9. Consensus Crown Strategies
Define robust strategies to build a single consensus crown polygon for a tracked chain, then visualize consensus extractions across OMs.

In [12]:
from shapely.ops import unary_union
from functools import reduce

def _chain_polygons_aligned(chain, tracker):
    polys = []
    for om_id, crown_idx in chain:
        poly = tracker.crowns_gdfs[om_id].iloc[crown_idx].geometry
        if poly is not None and not poly.is_empty:
            polys.append(poly.buffer(0))
    return polys

def _largest_polygon(geom):
    if geom is None or geom.is_empty:
        return geom
    if geom.geom_type == "Polygon":
        return geom
    if geom.geom_type in ("MultiPolygon", "GeometryCollection"):
        polys = [g for g in geom.geoms if g.geom_type == "Polygon"]
        return max(polys, key=lambda g: g.area) if polys else geom
    return geom

def consensus_medoid(chain, tracker, w_centroid=0.5, w_iou=0.4, w_area=0.1):
    polys = _chain_polygons_aligned(chain, tracker)
    if not polys:
        return None
    centroids = [p.centroid for p in polys]
    areas = [p.area for p in polys]
    max_dist = max((c1.distance(c2) for c1 in centroids for c2 in centroids), default=1.0) or 1.0
    best_idx = 0
    best_score = float("inf")
    for i, p in enumerate(polys):
        score = 0.0
        for j, q in enumerate(polys):
            if i == j:
                continue
            dist = centroids[i].distance(centroids[j]) / max_dist
            iou = tracker._compute_iou(p, q)
            area_ratio = min(areas[i], areas[j]) / max(areas[i], areas[j]) if max(areas[i], areas[j]) > 0 else 0.0
            score += (w_centroid * dist) + (w_iou * (1.0 - iou)) + (w_area * (1.0 - area_ratio))
        if score < best_score:
            best_score = score
            best_idx = i
    return polys[best_idx]

def consensus_intersection_core(chain, tracker, buffer_dist=0.5, min_area=1e-3):
    polys = _chain_polygons_aligned(chain, tracker)
    if not polys:
        return None
    buffered = [p.buffer(buffer_dist) for p in polys]
    inter = reduce(lambda a, b: a.intersection(b), buffered)
    if inter.is_empty:
        return consensus_medoid(chain, tracker)
    core = _largest_polygon(inter.buffer(-buffer_dist))
    if core is None or core.is_empty or core.area < min_area:
        return consensus_medoid(chain, tracker)
    return core

def _buffer_to_target_area(poly, target_area, max_iters=30):
    if poly is None or poly.is_empty:
        return poly
    if poly.area <= target_area:
        return poly
    lo, hi = -max(1.0, poly.length / (2 * np.pi)), 0.0
    for _ in range(max_iters):
        mid = (lo + hi) / 2.0
        buff = poly.buffer(mid)
        area = buff.area if not buff.is_empty else 0.0
        if area > target_area:
            hi = mid
        else:
            lo = mid
    result = poly.buffer(hi)
    return result if not result.is_empty else poly

def consensus_union_shrink(chain, tracker):
    polys = _chain_polygons_aligned(chain, tracker)
    if not polys:
        return None
    union = _largest_polygon(unary_union(polys))
    target_area = float(np.median([p.area for p in polys]))
    return _buffer_to_target_area(union, target_area)

def extract_patch_for_polygon(om_id, polygon_aligned, tracker):
    if polygon_aligned is None or polygon_aligned.is_empty:
        return None
    dx, dy = tracker.alignment_shifts.get(om_id, (0.0, 0.0))
    polygon_original = translate(polygon_aligned, xoff=-dx, yoff=-dy)
    ortho_file = None
    for oid, (crown_file, of) in zip(tracker.om_ids, tracker.file_pairs):
        if oid == om_id:
            ortho_file = of
            break
    if not ortho_file or not os.path.exists(ortho_file):
        return None
    try:
        with rasterio.open(ortho_file) as src:
            out_image, _ = mask(src, [mapping(polygon_original)], crop=True)
            return np.moveaxis(out_image, 0, -1)
    except Exception:
        return None

def visualize_consensus_chain(chain, consensus_poly, tracker, title="Consensus Chain", show=True):
    if consensus_poly is None:
        print("No consensus polygon available.")
        return
    chain_length = len(chain)
    fig = plt.figure(figsize=(5 * chain_length, 10))
    print(f"\n{'='*80}")
    print(title)
    print(f"{'='*80}")
    print(f"Chain: {chain}")
    print(f"Length: {chain_length}\n")
    for idx, (om_id, crown_idx) in enumerate(chain):
        gdf = tracker.crowns_gdfs[om_id]
        crown = gdf.iloc[crown_idx]
        ax_poly = plt.subplot(2, chain_length, idx + 1)
        gdf.plot(ax=ax_poly, color='lightgray', edgecolor='gray', alpha=0.3)
        gpd.GeoSeries([consensus_poly]).plot(
            ax=ax_poly,
            facecolor='none',
            edgecolor='red',
            linewidth=2.5,
            alpha=0.9
        )
        gpd.GeoSeries([crown.geometry]).plot(
            ax=ax_poly,
            facecolor='none',
            edgecolor='black',
            linewidth=1.5,
            alpha=0.9
        )
        centroid = consensus_poly.centroid
        ax_poly.plot(centroid.x, centroid.y, 'o', color='yellow', markersize=8,
                     markeredgecolor='black', markeredgewidth=1.5)
        ax_poly.set_aspect('equal')
        ax_poly.set_title(f"OM{om_id} - Consensus Overlay", fontsize=10, fontweight='bold')
        ax_poly.grid(True, alpha=0.3)
        ax_img = plt.subplot(2, chain_length, chain_length + idx + 1)
        patch = extract_patch_for_polygon(om_id, consensus_poly, tracker)
        if patch is not None:
            ax_img.imshow(patch)
            ax_img.set_title(f"Consensus Patch\n{patch.shape[0]}×{patch.shape[1]} px", fontsize=9)
        else:
            ax_img.text(0.5, 0.5, 'Image not available', ha='center', va='center', fontsize=10, color='red')
            ax_img.set_title("No Image", fontsize=9)
        ax_img.axis('off')
    plt.tight_layout()
    if show:
        plt.show()
    print()

In [13]:
# Patch: use a local import so the global 'mask' variable (numpy array) can never shadow
# rasterio.mask.mask inside extract_patch_for_polygon.
def extract_patch_for_polygon(om_id, polygon_aligned, tracker):
    if polygon_aligned is None or polygon_aligned.is_empty:
        return None
    dx, dy = tracker.alignment_shifts.get(om_id, (0.0, 0.0))
    from shapely.affinity import translate
    from shapely.geometry import mapping
    polygon_original = translate(polygon_aligned, xoff=-dx, yoff=-dy)
    ortho_file = None
    for oid, (crown_file, of) in zip(tracker.om_ids, tracker.file_pairs):
        if oid == om_id:
            ortho_file = of
            break
    if not ortho_file or not os.path.exists(ortho_file):
        return None
    try:
        from rasterio.mask import mask as _rios_mask   # local import avoids name-shadowing
        with rasterio.open(ortho_file) as src:
            out_image, _ = _rios_mask(src, [mapping(polygon_original)], crop=True)
            return np.moveaxis(out_image, 0, -1)
    except Exception:
        return None


In [ ]:
# # Compare normal chain visualization vs consensus crown strategies
# full_chains = chain_categories['full_width_1'] + chain_categories['full_branching']
# candidate_chains = full_chains if full_chains else chain_categories.get('partial_long', [])
# if not candidate_chains:
#     candidate_chains = chain_categories.get('partial_short', [])
# if not candidate_chains:
#     print("No chains available for consensus comparison.")
# else:
#     chain = candidate_chains[5]
#     print("\n=== Normal Chain Visualization ===")
#     visualize_chain_with_extracted_images(
#         chain,
#         tracker.crowns_gdfs,
#         tracker.crown_images,
#         tracker,
#         title="Original Crowns (per-OM)"
#     )
#     strategies = {
#         "medoid": consensus_medoid,
#         "intersection_core": consensus_intersection_core,
#         "union_shrink": consensus_union_shrink,
#     }
#     for name, func in strategies.items():
#         consensus_poly = func(chain, tracker)
#         visualize_consensus_chain(
#             chain,
#             consensus_poly,
#             tracker,
#             title=f"Consensus Strategy: {name}"
#         )

In [ ]:
# # Visualize medoid consensus for all full chains without branching
# full_width_1_chains = chain_categories.get('full_width_1', [])
# print(f"Visualizing medoid consensus for {len(full_width_1_chains)} non-branching full chains...")
# if not full_width_1_chains:
#     print("No non-branching full chains found.")
# else:
#     for i, chain in enumerate(full_width_1_chains):
#         consensus_poly = consensus_medoid(chain, tracker)
#         visualize_consensus_chain(
#             chain,
#             consensus_poly,
#             tracker,
#             title=f"Medoid Consensus - Full Chain {i+1}/{len(full_width_1_chains)}"
#         )
#     print(f"✓ Completed medoid consensus visualization for {len(full_width_1_chains)} chains")

## Ground Truth Comparison

In [14]:
import json
from shapely.geometry import Polygon
from typing import Dict, List, Tuple

def _infer_pixel_space(seg_coords, width=None, height=None, pad=5.0):
    if width is None or height is None:
        return None
    if not seg_coords:
        return None
    xs = [p[0] for p in seg_coords]
    ys = [p[1] for p in seg_coords]
    max_x = max(xs)
    max_y = max(ys)
    # Likely pixel if coords fall within image bounds (with small padding)
    if max_x <= (width + pad) and max_y <= (height + pad):
        return True
    # Likely geo if far outside pixel bounds
    if max_x > (width * 2) or max_y > (height * 2):
        return False
    return None

def load_ground_truth_coco(json_path: str, ortho_path: str = None, transform_to_geo: str = "auto") -> gpd.GeoDataFrame:
    """
    Load ground truth crowns from COCO format JSON.
    
    Args:
        json_path: Path to COCO format JSON file
        ortho_path: Path to orthomosaic for coordinate transformation
        transform_to_geo: "auto", True, or False
    
    Returns:
        GeoDataFrame with ground truth crowns in geographic coordinates (if transformed)
    """
    with open(json_path, 'r') as f:
        coco_data = json.load(f)
    
    geometries = []
    gt_ids = []
    track_ids = []
    
    # Get transform from orthomosaic if needed
    transform = None
    ortho_crs = None
    ortho_w = None
    ortho_h = None
    if ortho_path and os.path.exists(ortho_path):
        with rasterio.open(ortho_path) as src:
            transform = src.transform
            ortho_crs = src.crs
            ortho_w = src.width
            ortho_h = src.height
            print(f"Orthomosaic transform: {transform}")
            print(f"Orthomosaic CRS: {src.crs}")
            print(f"Orthomosaic size: {src.width}×{src.height} px")
    
    # Try to infer pixel vs geo if auto
    if transform_to_geo == "auto":
        # Find first segmentation coords to infer scale
        sample_coords = None
        for ann in coco_data.get('annotations', []):
            if ann.get('segmentation') and len(ann['segmentation']) > 0:
                seg = ann['segmentation'][0]
                sample_coords = [(seg[i], seg[i+1]) for i in range(0, len(seg), 2)]
                break
        is_pixel = _infer_pixel_space(sample_coords or [], ortho_w, ortho_h)
        if is_pixel is None:
            transform_to_geo = bool(transform is not None)
        else:
            transform_to_geo = bool(is_pixel)
        print(f"Auto-detected ground truth coordinates as {'pixel' if transform_to_geo else 'geo'} space")
    
    for ann in coco_data['annotations']:
        # COCO segmentation is [[x1,y1,x2,y2,...]] in pixel coordinates
        seg = ann['segmentation'][0]
        pixel_coords = [(seg[i], seg[i+1]) for i in range(0, len(seg), 2)]
        
        if transform_to_geo and transform is not None:
            # Transform from pixel (col, row) to geographic (x, y)
            geo_coords = []
            for px, py in pixel_coords:
                # In rasterio, pixel (col, row) -> (x, y)
                x, y = rasterio.transform.xy(transform, py, px, offset='ul')
                geo_coords.append((x, y))
            poly = Polygon(geo_coords)
        else:
            # Keep as given coordinates
            poly = Polygon(pixel_coords)
        
        geometries.append(poly)
        gt_ids.append(ann['id'])
        track_ids.append(ann.get('attributes', {}).get('track_id', ann['id']))
    
    gdf = gpd.GeoDataFrame({
        'gt_id': gt_ids,
        'track_id': track_ids,
        'geometry': geometries
    }, crs=ortho_crs if ortho_crs is not None else None)
    
    print(f"Loaded {len(gdf)} ground truth crowns")
    if transform is not None and transform_to_geo:
        print(f"Transformed from pixel to geographic coordinates")
        # Print sample bounds to verify
        if len(gdf) > 0:
            bounds = gdf.total_bounds
            print(f"Bounds: x=[{bounds[0]:.1f}, {bounds[2]:.1f}], y=[{bounds[1]:.1f}, {bounds[3]:.1f}]")
    
    return gdf

def compute_metrics_at_iou_threshold(
    ground_truth: gpd.GeoDataFrame,
    predictions: gpd.GeoDataFrame,
    iou_threshold: float = 0.5
) -> Dict[str, float]:
    """
    Compute precision, recall, F1 score at a given IoU threshold.
    
    Args:
        ground_truth: GeoDataFrame with ground truth polygons
        predictions: GeoDataFrame with predicted/consensus polygons
        iou_threshold: Minimum IoU to count as a match
    
    Returns:
        Dictionary with precision, recall, f1, tp, fp, fn
    """
    gt_matched = set()
    pred_matched = set()
    
    # For each prediction, find best matching ground truth
    for pred_idx, pred_row in predictions.iterrows():
        pred_poly = pred_row.geometry
        if pred_poly is None or pred_poly.is_empty:
            continue
        
        best_iou = 0.0
        best_gt_idx = None
        
        for gt_idx, gt_row in ground_truth.iterrows():
            gt_poly = gt_row.geometry
            if gt_poly is None or gt_poly.is_empty:
                continue
            
            # Compute IoU
            try:
                intersection = pred_poly.intersection(gt_poly).area
                union = pred_poly.union(gt_poly).area
                iou = intersection / union if union > 0 else 0.0
                
                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = gt_idx
            except Exception:
                continue
        
        # If best IoU exceeds threshold, mark as match
        if best_iou >= iou_threshold and best_gt_idx is not None:
            gt_matched.add(best_gt_idx)
            pred_matched.add(pred_idx)
    
    tp = len(gt_matched)  # True positives: GT crowns matched
    fp = len(predictions) - len(pred_matched)  # False positives: predictions without GT match
    fn = len(ground_truth) - len(gt_matched)  # False negatives: GT crowns not detected
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'iou_threshold': iou_threshold
    }

def get_consensus_crowns_for_full_chains(
    tracker: 'TreeTrackingGraph',
    chain_categories: Dict,
    consensus_strategy = consensus_intersection_core
) -> gpd.GeoDataFrame:
    """
    Extract consensus crowns for all fully-tracked chains.
    
    Args:
        tracker: TreeTrackingGraph instance
        chain_categories: Dictionary with categorized chains
        consensus_strategy: Function to compute consensus polygon
    
    Returns:
        GeoDataFrame with consensus polygons for each full chain
    """
    full_chains = chain_categories.get('full_width_1', []) + chain_categories.get('full_branching', [])
    
    consensus_geoms = []
    chain_ids = []
    chain_lengths = []
    
    for i, chain in enumerate(full_chains):
        consensus_poly = consensus_strategy(chain, tracker)
        if consensus_poly is not None and not consensus_poly.is_empty:
            consensus_geoms.append(consensus_poly)
            chain_ids.append(i)
            chain_lengths.append(len(chain))
    
    base_crs = None
    if tracker and tracker.om_ids:
        base_crs = tracker.crowns_gdfs[tracker.om_ids[0]].crs
    gdf = gpd.GeoDataFrame({
        'chain_id': chain_ids,
        'chain_length': chain_lengths,
        'geometry': consensus_geoms
    }, crs=base_crs)
    
    print(f"Generated {len(gdf)} consensus crowns from {len(full_chains)} full chains")
    return gdf

print("Ground truth comparison functions loaded")

Ground truth comparison functions loaded


In [15]:
# Load ground truth with auto coordinate inference
gt_path = "../../input/ground_truth.json"
ortho_path = next((of for _, of in tracker.file_pairs if of), None)
ground_truth_gdf = load_ground_truth_coco(gt_path, ortho_path=ortho_path, transform_to_geo="auto")
print(f"Ground truth CRS: {ground_truth_gdf.crs}")

Orthomosaic transform: | 0.02, 0.00, 714263.73|
| 0.00,-0.02, 3159729.01|
| 0.00, 0.00, 1.00|
Orthomosaic CRS: PROJCS["WGS 84 / UTM zone 43N",GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",75],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Orthomosaic size: 10742×11102 px
Auto-detected ground truth coordinates as pixel space
Loaded 124 ground truth crowns
Transformed from pixel to geographic coordinates
Bounds: x=[714295.5, 714477.5], y=[3159492.0, 3159672.2]
Ground truth CRS: PROJCS["WGS 84 / UTM zone 43N",GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degre

In [16]:
# Load consensus crowns from latest larger tracking run when available, fallback to in-notebook generation
consensus_candidates = [
    Path('../../output/consensus_crowns_complete_all.gpkg'),
    Path('../../output/consensus_crowns_all_chains.gpkg'),
]

consensus_source = None
for cand in consensus_candidates:
    if cand.exists():
        consensus_source = cand
        break

if consensus_source is not None:
    consensus_crowns_gdf = gpd.read_file(consensus_source)
    print(f"Loaded consensus crowns from: {consensus_source}")
    
    # Ensure chain_id exists for downstream phenology grouping
    if 'chain_id' not in consensus_crowns_gdf.columns:
        consensus_crowns_gdf = consensus_crowns_gdf.reset_index(drop=True).copy()
        consensus_crowns_gdf['chain_id'] = consensus_crowns_gdf.index.astype(int)
    
    # Keep only valid non-empty geometries
    consensus_crowns_gdf = consensus_crowns_gdf[
        consensus_crowns_gdf.geometry.notnull() & ~consensus_crowns_gdf.geometry.is_empty
    ].copy()
else:
    print("No precomputed consensus file found. Falling back to notebook-generated full-chain consensus.")
    consensus_crowns_gdf = get_consensus_crowns_for_full_chains(
        tracker,
        chain_categories,
        consensus_strategy=consensus_medoid,
    )

def _intersection_core_from_reference(
    ref_poly,
    tracker,
    max_centroid_dist=35.0,
    buffer_dist=0.5,
    min_support=2,
    min_area=1e-3,
    fallback_strategy=consensus_medoid,
):
    if ref_poly is None or ref_poly.is_empty:
        return ref_poly
    
    ref_centroid = ref_poly.centroid
    matched_polys = []
    
    for om_id in tracker.om_ids:
        gdf_om = tracker.crowns_gdfs.get(om_id)
        if gdf_om is None or gdf_om.empty:
            continue
        
        best_geom = None
        best_score = -1e9
        
        for geom in gdf_om.geometry:
            if geom is None or geom.is_empty:
                continue
            dist = ref_centroid.distance(geom.centroid)
            if dist > max_centroid_dist:
                continue
            iou = tracker._compute_iou(ref_poly, geom)
            score = iou - (dist / max_centroid_dist)
            if score > best_score:
                best_score = score
                best_geom = geom.buffer(0)
        
        if best_geom is not None and not best_geom.is_empty:
            matched_polys.append(best_geom)
    
    if len(matched_polys) < min_support:
        return ref_poly
    
    buffered = [p.buffer(buffer_dist) for p in matched_polys]
    inter = reduce(lambda a, b: a.intersection(b), buffered)
    if inter is None or inter.is_empty:
        return ref_poly
    
    core = _largest_polygon(inter.buffer(-buffer_dist))
    if core is None or core.is_empty or core.area < min_area:
        return ref_poly
    
    return core.buffer(0)

# Keep the same tree set (same rows/chain_id), swap only geometry to intersection core
before_n = len(consensus_crowns_gdf)
before_area = float(consensus_crowns_gdf.geometry.area.median()) if before_n else float('nan')

consensus_crowns_gdf = consensus_crowns_gdf.copy()
consensus_crowns_gdf['geometry'] = consensus_crowns_gdf['geometry'].apply(
    lambda g: _intersection_core_from_reference(g, tracker)
 )
consensus_crowns_gdf = consensus_crowns_gdf[
    consensus_crowns_gdf.geometry.notnull() & ~consensus_crowns_gdf.geometry.is_empty
].copy()

after_n = len(consensus_crowns_gdf)
after_area = float(consensus_crowns_gdf.geometry.area.median()) if after_n else float('nan')

# Align CRS to ground truth for evaluation metrics
if ground_truth_gdf.crs is not None and consensus_crowns_gdf.crs is None:
    consensus_crowns_gdf = consensus_crowns_gdf.set_crs(ground_truth_gdf.crs, allow_override=True)
elif ground_truth_gdf.crs is not None and consensus_crowns_gdf.crs != ground_truth_gdf.crs:
    consensus_crowns_gdf = consensus_crowns_gdf.to_crs(ground_truth_gdf.crs)

print(f"\nConsensus geometry mode: intersection_core_from_reference")
print(f"Tree count preserved: {before_n} -> {after_n}")
print(f"Median crown area change: {before_area:.2f} -> {after_area:.2f} m²")
print(f"Consensus crowns: {len(consensus_crowns_gdf)}")
print(f"Ground truth crowns: {len(ground_truth_gdf)}")

Loaded consensus crowns from: ../../output/consensus_crowns_complete_all.gpkg

Consensus geometry mode: intersection_core_from_reference
Tree count preserved: 80 -> 80
Median crown area change: 47.53 -> 40.29 m²
Consensus crowns: 80
Ground truth crowns: 124


In [17]:
# Compute metrics at default IoU threshold (0.5)
metrics_050 = compute_metrics_at_iou_threshold(
    ground_truth_gdf,
    consensus_crowns_gdf,
    iou_threshold=0.3
)

print("\n" + "="*80)
print("GROUND TRUTH COMPARISON METRICS (IoU threshold = 0.3)")
print("="*80)
print(f"Precision:  {metrics_050['precision']:.3f}")
print(f"Recall:     {metrics_050['recall']:.3f}")
print(f"F1 Score:   {metrics_050['f1']:.3f}")
print(f"\nTrue Positives:  {metrics_050['tp']}")
print(f"False Positives: {metrics_050['fp']}")
print(f"False Negatives: {metrics_050['fn']}")
print("="*80)


GROUND TRUTH COMPARISON METRICS (IoU threshold = 0.3)
Precision:  0.573
Recall:     0.347
F1 Score:   0.432

True Positives:  43
False Positives: 32
False Negatives: 81


## Error Classification and Analysis

Classify different types of errors in ground truth vs consensus crown comparison.

In [ ]:
from enum import Enum
from typing import Dict, List, Tuple, Optional
import pandas as pd

class ErrorType(Enum):
    """Classification of errors in ground truth vs prediction comparison."""
    # Detection errors
    MISSING_DETECTION = "Missing Detection"  # GT crown with no consensus match (FN)
    FALSE_POSITIVE = "False Positive"  # Consensus crown with no GT match (FP)
    
    # Geometric errors
    SPLIT_ERROR = "Split Error"  # 1 GT crown split into 2+ consensus crowns (1 GT -> multiple predictions)
    MERGE_ERROR = "Merge Error"  # 2+ GT crowns merged into 1 consensus crown (multiple GT -> 1 prediction)
    
    # Quality errors
    POOR_OVERLAP = "Poor Overlap"  # Match exists but IoU is low (below threshold)
    BOUNDARY_ERROR = "Boundary Error"  # IoU < threshold but has overlap, with size difference
    
    # Correct matches
    TRUE_POSITIVE = "True Positive"  # Good match (IoU >= threshold)

def classify_crown_errors(
    ground_truth: gpd.GeoDataFrame,
    predictions: gpd.GeoDataFrame,
    iou_threshold: float = 0.5,
    poor_overlap_threshold: float = 0.15,
    size_diff_threshold: float = 0.3,
    nearby_threshold: float = 15.0
) -> Tuple[List[Dict], pd.DataFrame]:
    """
    Classify errors between ground truth and predicted crowns.
    
    Split Error: 1 GT crown -> multiple predictions (we over-detected)
    Merge Error: multiple GT crowns -> 1 prediction (we under-detected)
    
    Args:
        ground_truth: GeoDataFrame with ground truth polygons
        predictions: GeoDataFrame with predicted/consensus polygons
        iou_threshold: Minimum IoU to count as true positive
        poor_overlap_threshold: Minimum IoU to consider as having overlap
        size_diff_threshold: Relative area difference for boundary errors
        nearby_threshold: Max distance for split/merge detection
    
    Returns:
        Tuple of (list of error cases, summary dataframe)
    """
    error_cases = []
    
    # Step 1: Build IoU matrix for ALL pairs
    iou_matrix = {}  # (gt_idx, pred_idx) -> iou
    
    for gt_idx, gt_row in ground_truth.iterrows():
        gt_poly = gt_row.geometry
        if gt_poly is None or gt_poly.is_empty:
            continue
        
        for pred_idx, pred_row in predictions.iterrows():
            pred_poly = pred_row.geometry
            if pred_poly is None or pred_poly.is_empty:
                continue
            
            try:
                intersection = gt_poly.intersection(pred_poly).area
                union = gt_poly.union(pred_poly).area
                iou = intersection / union if union > 0 else 0.0
                if iou > 0:
                    iou_matrix[(gt_idx, pred_idx)] = iou
            except Exception:
                continue
    
    # Step 2: Find matches for each GT and Pred
    gt_matches = {}  # gt_idx -> [(pred_idx, iou), ...]
    pred_matches = {}  # pred_idx -> [(gt_idx, iou), ...]
    
    for (gt_idx, pred_idx), iou in iou_matrix.items():
        if iou >= poor_overlap_threshold:
            if gt_idx not in gt_matches:
                gt_matches[gt_idx] = []
            gt_matches[gt_idx].append((pred_idx, iou))
            
            if pred_idx not in pred_matches:
                pred_matches[pred_idx] = []
            pred_matches[pred_idx].append((gt_idx, iou))
    
    # Sort matches by IoU (best first)
    for gt_idx in gt_matches:
        gt_matches[gt_idx].sort(key=lambda x: x[1], reverse=True)
    for pred_idx in pred_matches:
        pred_matches[pred_idx].sort(key=lambda x: x[1], reverse=True)
    
    # Step 3: Classify each GT crown
    gt_classified = set()
    
    for gt_idx, gt_row in ground_truth.iterrows():
        gt_poly = gt_row.geometry
        if gt_poly is None or gt_poly.is_empty:
            continue
        
        if gt_idx not in gt_matches:
            # No prediction overlaps with this GT
            error_cases.append({
                'error_type': ErrorType.MISSING_DETECTION.value,
                'gt_idx': int(gt_idx),
                'pred_idx': None,
                'iou': 0.0,
                'gt_area': float(gt_poly.area),
                'pred_area': None,
                'gt_centroid': gt_poly.centroid,
                'pred_centroid': None
            })
            gt_classified.add(gt_idx)
            continue
        
        matches = gt_matches[gt_idx]
        best_pred_idx, best_iou = matches[0]
        
        # Check for SPLIT: 1 GT -> multiple predictions
        if len(matches) > 1:
            # Check if multiple predictions are nearby and all have decent overlap
            pred_polys = [predictions.iloc[pred_idx].geometry for pred_idx, _ in matches]
            nearby_preds = []
            for i, (pred_idx, iou) in enumerate(matches):
                pred_poly = predictions.iloc[pred_idx].geometry
                # Check if this prediction is near the GT centroid
                dist = gt_poly.centroid.distance(pred_poly.centroid)
                if dist < nearby_threshold and iou >= poor_overlap_threshold:
                    nearby_preds.append((pred_idx, iou))
            
            if len(nearby_preds) > 1:
                # SPLIT ERROR: This GT crown was split into multiple predictions
                error_cases.append({
                    'error_type': ErrorType.SPLIT_ERROR.value,
                    'gt_idx': int(gt_idx),
                    'pred_idx': [int(idx) for idx, _ in nearby_preds],
                    'iou': [float(iou) for _, iou in nearby_preds],
                    'gt_area': float(gt_poly.area),
                    'pred_area': [float(predictions.iloc[idx].geometry.area) for idx, _ in nearby_preds],
                    'gt_centroid': gt_poly.centroid,
                    'pred_centroid': [predictions.iloc[idx].geometry.centroid for idx, _ in nearby_preds],
                    'num_splits': len(nearby_preds)
                })
                gt_classified.add(gt_idx)
                continue
        
        # Single best match - classify based on IoU
        pred_poly = predictions.iloc[best_pred_idx].geometry
        
        if best_iou >= iou_threshold:
            # TRUE POSITIVE: Good match
            error_cases.append({
                'error_type': ErrorType.TRUE_POSITIVE.value,
                'gt_idx': int(gt_idx),
                'pred_idx': int(best_pred_idx),
                'iou': float(best_iou),
                'gt_area': float(gt_poly.area),
                'pred_area': float(pred_poly.area),
                'gt_centroid': gt_poly.centroid,
                'pred_centroid': pred_poly.centroid
            })
        elif best_iou >= poor_overlap_threshold:
            # Check for boundary error (has overlap but low IoU, with size difference)
            area_ratio = abs(gt_poly.area - pred_poly.area) / max(gt_poly.area, pred_poly.area)
            if area_ratio > size_diff_threshold:
                error_type = ErrorType.BOUNDARY_ERROR.value
            else:
                error_type = ErrorType.POOR_OVERLAP.value
            
            error_cases.append({
                'error_type': error_type,
                'gt_idx': int(gt_idx),
                'pred_idx': int(best_pred_idx),
                'iou': float(best_iou),
                'gt_area': float(gt_poly.area),
                'pred_area': float(pred_poly.area),
                'gt_centroid': gt_poly.centroid,
                'pred_centroid': pred_poly.centroid,
                'area_ratio_diff': float(area_ratio) if area_ratio else None
            })
        else:
            # Very poor overlap, treat as missing
            error_cases.append({
                'error_type': ErrorType.MISSING_DETECTION.value,
                'gt_idx': int(gt_idx),
                'pred_idx': None,
                'iou': 0.0,
                'gt_area': float(gt_poly.area),
                'pred_area': None,
                'gt_centroid': gt_poly.centroid,
                'pred_centroid': None
            })
        
        gt_classified.add(gt_idx)
    
    # Step 4: Classify predictions for MERGE errors and FALSE POSITIVES
    pred_classified = set()
    
    for pred_idx, pred_row in predictions.iterrows():
        pred_poly = pred_row.geometry
        if pred_poly is None or pred_poly.is_empty:
            continue
        
        if pred_idx not in pred_matches:
            # FALSE POSITIVE: No GT overlaps with this prediction
            error_cases.append({
                'error_type': ErrorType.FALSE_POSITIVE.value,
                'gt_idx': None,
                'pred_idx': int(pred_idx),
                'iou': 0.0,
                'gt_area': None,
                'pred_area': float(pred_poly.area),
                'gt_centroid': None,
                'pred_centroid': pred_poly.centroid
            })
            pred_classified.add(pred_idx)
            continue
        
        matches = pred_matches[pred_idx]
        
        # Check for MERGE: multiple GT -> 1 prediction
        if len(matches) > 1:
            # Check if multiple GTs are nearby and all have decent overlap
            nearby_gts = []
            for gt_idx, iou in matches:
                gt_poly = ground_truth.iloc[gt_idx].geometry
                dist = pred_poly.centroid.distance(gt_poly.centroid)
                if dist < nearby_threshold and iou >= poor_overlap_threshold:
                    nearby_gts.append((gt_idx, iou))
            
            if len(nearby_gts) > 1:
                # MERGE ERROR: Multiple GT crowns merged into this prediction
                error_cases.append({
                    'error_type': ErrorType.MERGE_ERROR.value,
                    'gt_idx': [int(idx) for idx, _ in nearby_gts],
                    'pred_idx': int(pred_idx),
                    'iou': [float(iou) for _, iou in nearby_gts],
                    'gt_area': [float(ground_truth.iloc[idx].geometry.area) for idx, _ in nearby_gts],
                    'pred_area': float(pred_poly.area),
                    'gt_centroid': [ground_truth.iloc[idx].geometry.centroid for idx, _ in nearby_gts],
                    'pred_centroid': pred_poly.centroid,
                    'num_merged': len(nearby_gts)
                })
                pred_classified.add(pred_idx)
        # If not a merge and not already classified as FP, it was classified in GT loop
    
    # Create summary dataframe
    error_types = [case['error_type'] for case in error_cases]
    summary = pd.DataFrame({
        'Error Type': list(ErrorType.__members__.values()),
        'Count': [error_types.count(et.value) for et in ErrorType]
    })
    
    return error_cases, summary
    
    return error_cases, summary

print("✓ Error classification functions loaded")

In [ ]:
# Classify errors for IoU threshold 0.3
error_cases_03, error_summary_03 = classify_crown_errors(
    ground_truth_gdf,
    consensus_crowns_gdf,
    iou_threshold=0.3,
    poor_overlap_threshold=0.15,
    size_diff_threshold=0.3,
    nearby_threshold=15.0
)

print("\n" + "="*80)
print("ERROR CLASSIFICATION SUMMARY (IoU threshold = 0.3)")
print("="*80)
print(error_summary_03.to_string(index=False))
print("="*80)
print(f"\nTotal cases analyzed: {len(error_cases_03)}")

In [ ]:
# Also classify for IoU threshold 0.5
error_cases_05, error_summary_05 = classify_crown_errors(
    ground_truth_gdf,
    consensus_crowns_gdf,
    iou_threshold=0.5,
    poor_overlap_threshold=0.25,
    size_diff_threshold=0.3,
    nearby_threshold=15.0
)

print("\n" + "="*80)
print("ERROR CLASSIFICATION SUMMARY (IoU threshold = 0.5)")
print("="*80)
print(error_summary_05.to_string(index=False))
print("="*80)
print(f"\nTotal cases analyzed: {len(error_cases_05)}")

In [ ]:
def visualize_error_case(
    error_case: Dict,
    ground_truth: gpd.GeoDataFrame,
    predictions: gpd.GeoDataFrame,
    ortho_path: str,
    title: str = "Error Case",
    figsize: Tuple[int, int] = (15, 5)
):
    """
    Visualize a specific error case with polygon overlay and RGB image.
    
    Args:
        error_case: Dictionary with error case information
        ground_truth: GeoDataFrame with ground truth polygons
        predictions: GeoDataFrame with predicted polygons
        ortho_path: Path to orthomosaic for background image
        title: Plot title
        figsize: Figure size
    """
    fig, axes = plt.subplots(1, 3, figsize=figsize)
    
    error_type = error_case['error_type']
    gt_idx = error_case.get('gt_idx')
    pred_idx = error_case.get('pred_idx')
    
    # Determine extent for visualization
    gt_polys = []
    pred_polys = []
    
    if gt_idx is not None:
        if isinstance(gt_idx, list):
            gt_polys = [ground_truth.iloc[idx].geometry for idx in gt_idx]
        else:
            gt_polys = [ground_truth.iloc[gt_idx].geometry]
    
    if pred_idx is not None:
        if isinstance(pred_idx, list):
            pred_polys = [predictions.iloc[idx].geometry for idx in pred_idx]
        else:
            pred_polys = [predictions.iloc[pred_idx].geometry]
    
    all_polys = []
    for p in gt_polys + pred_polys:
        if p is not None and hasattr(p, 'is_empty') and not p.is_empty:
            all_polys.append(p)
    
    if not all_polys:
        print(f"No valid polygons for case: {error_case}")
        plt.close(fig)
        return
    
    # Get bounds with margin
    union_poly = unary_union(all_polys)
    minx, miny, maxx, maxy = union_poly.bounds
    width = maxx - minx
    height = maxy - miny
    margin = max(width, height) * 0.3
    
    extent = [minx - margin, maxx + margin, miny - margin, maxy + margin]
    
    # Load orthomosaic for background
    ortho_img = None
    try:
        with rasterio.open(ortho_path) as src:
            # Transform extent to pixel coordinates
            window = rasterio.windows.from_bounds(
                extent[0], extent[2], extent[1], extent[3], 
                src.transform
            )
            ortho_img = src.read([1, 2, 3], window=window)
            ortho_img = np.moveaxis(ortho_img, 0, -1)
            # Normalize for display
            ortho_img = np.clip(ortho_img / ortho_img.max() * 255, 0, 255).astype(np.uint8)
    except Exception as e:
        print(f"Could not load ortho image: {e}")
    
    # Plot 1: Ground truth only
    ax1 = axes[0]
    if ortho_img is not None:
        ax1.imshow(ortho_img, extent=extent, aspect='auto')
    for gt_poly in gt_polys:
        gpd.GeoSeries([gt_poly]).plot(
            ax=ax1, facecolor='none', edgecolor='blue', linewidth=2.5, alpha=0.9
        )
    ax1.set_xlim(extent[0], extent[1])
    ax1.set_ylim(extent[2], extent[3])
    ax1.set_title("Ground Truth", fontsize=10, fontweight='bold')
    ax1.set_aspect('equal')
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Prediction only
    ax2 = axes[1]
    if ortho_img is not None:
        ax2.imshow(ortho_img, extent=extent, aspect='auto')
    for pred_poly in pred_polys:
        gpd.GeoSeries([pred_poly]).plot(
            ax=ax2, facecolor='none', edgecolor='red', linewidth=2.5, alpha=0.9
        )
    ax2.set_xlim(extent[0], extent[1])
    ax2.set_ylim(extent[2], extent[3])
    ax2.set_title("Consensus Crown", fontsize=10, fontweight='bold')
    ax2.set_aspect('equal')
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Overlay
    ax3 = axes[2]
    if ortho_img is not None:
        ax3.imshow(ortho_img, extent=extent, aspect='auto')
    for gt_poly in gt_polys:
        gpd.GeoSeries([gt_poly]).plot(
            ax=ax3, facecolor='blue', edgecolor='blue', linewidth=2, alpha=0.3, label='GT'
        )
    for pred_poly in pred_polys:
        gpd.GeoSeries([pred_poly]).plot(
            ax=ax3, facecolor='red', edgecolor='red', linewidth=2, alpha=0.3, label='Consensus'
        )
    ax3.set_xlim(extent[0], extent[1])
    ax3.set_ylim(extent[2], extent[3])
    ax3.set_title("Overlay", fontsize=10, fontweight='bold')
    ax3.set_aspect('equal')
    ax3.grid(True, alpha=0.3)
    
    # Add handles for legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='blue', edgecolor='blue', alpha=0.5, label='Ground Truth'),
        Patch(facecolor='red', edgecolor='red', alpha=0.5, label='Consensus')
    ]
    ax3.legend(handles=legend_elements, loc='upper right', fontsize=8)
    
    # Main title with case details
    iou_str = f"{error_case.get('iou', 0):.3f}" if isinstance(error_case.get('iou'), (int, float)) else "N/A"
    fig.suptitle(f"{title}: {error_type} (IoU: {iou_str})", fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

def visualize_error_examples(
    error_cases: List[Dict],
    ground_truth: gpd.GeoDataFrame,
    predictions: gpd.GeoDataFrame,
    ortho_path: str,
    max_per_type: int = 2
):
    """
    Visualize examples of each error type.
    
    Args:
        error_cases: List of error case dictionaries
        ground_truth: GeoDataFrame with ground truth polygons
        predictions: GeoDataFrame with predicted polygons
        ortho_path: Path to orthomosaic
        max_per_type: Maximum number of examples per error type
    """
    # Group cases by error type
    by_type = {}
    for case in error_cases:
        error_type = case['error_type']
        if error_type not in by_type:
            by_type[error_type] = []
        by_type[error_type].append(case)
    
    # Visualize examples of each type
    for error_type, cases in by_type.items():
        if len(cases) == 0:
            continue
        
        print(f"\n{'='*80}")
        print(f"{error_type.upper()} - {len(cases)} cases")
        print(f"{'='*80}\n")
        
        # Show up to max_per_type examples
        for i, case in enumerate(cases[:max_per_type]):
            visualize_error_case(
                case,
                ground_truth,
                predictions,
                ortho_path,
                title=f"{error_type} Example {i+1}/{min(len(cases), max_per_type)}",
                figsize=(15, 5)
            )

print("✓ Error visualization functions loaded")

In [ ]:
# # Visualize examples of each error type (IoU 0.3)
# visualize_error_examples(
#     error_cases_03,
#     ground_truth_gdf,
#     consensus_crowns_gdf,
#     ortho_path,
#     max_per_type=3
# )

## Conditional Metrics (Excluding Specific Error Types)

Calculate precision and recall after excluding certain error categories to better understand model performance.

In [ ]:
def compute_conditional_metrics(
    error_cases: List[Dict],
    exclude_types: List[str] = None
) -> Dict[str, float]:
    """
    Compute precision, recall, F1 excluding specific error types.
    
    This allows us to understand performance under different assumptions:
    - Excluding MISSING_DETECTION: How good are we when we do detect something?
    - Excluding FALSE_POSITIVE: How good are matches when we ignore extra detections?
    - Excluding SPLIT/MERGE: Performance on simple 1-to-1 matches
    
    Args:
        error_cases: List of classified error cases
        exclude_types: List of error type names to exclude from calculation
    
    Returns:
        Dictionary with conditional metrics
    """
    if exclude_types is None:
        exclude_types = []
    
    # Filter cases
    filtered_cases = [
        case for case in error_cases 
        if case['error_type'] not in exclude_types
    ]
    
    # Count by type in filtered set
    tp = sum(1 for case in filtered_cases if case['error_type'] == ErrorType.TRUE_POSITIVE.value)
    fp = sum(1 for case in filtered_cases if case['error_type'] == ErrorType.FALSE_POSITIVE.value)
    fn = sum(1 for case in filtered_cases if case['error_type'] == ErrorType.MISSING_DETECTION.value)
    
    # Other errors that affect counts
    split_errors = sum(1 for case in filtered_cases if case['error_type'] == ErrorType.SPLIT_ERROR.value)
    merge_errors = sum(1 for case in filtered_cases if case['error_type'] == ErrorType.MERGE_ERROR.value)
    poor_overlap = sum(1 for case in filtered_cases if case['error_type'] == ErrorType.POOR_OVERLAP.value)
    boundary_errors = sum(1 for case in filtered_cases if case['error_type'] == ErrorType.BOUNDARY_ERROR.value)
    
    # Depending on interpretation, boundary errors could be TP
    # and poor_overlap, split, merge could be FP
    total_detected = tp + fp + split_errors + merge_errors + poor_overlap + boundary_errors
    total_gt = tp + fn + split_errors + merge_errors + poor_overlap + boundary_errors
    
    precision = tp / total_detected if total_detected > 0 else 0.0
    recall = tp / total_gt if total_gt > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'split_errors': split_errors,
        'merge_errors': merge_errors,
        'poor_overlap': poor_overlap,
        'boundary_errors': boundary_errors,
        'excluded_types': exclude_types,
        'total_cases': len(filtered_cases)
    }

def create_conditional_metrics_table(error_cases: List[Dict], iou_threshold: float) -> pd.DataFrame:
    """
    Create a table of metrics under different exclusion scenarios.
    
    Args:
        error_cases: List of error cases
        iou_threshold: IoU threshold used for classification
    
    Returns:
        DataFrame with conditional metrics
    """
    scenarios = [
        ("All Errors Included", []),
        ("Exclude False Positives", [ErrorType.FALSE_POSITIVE.value]),
        ("Exclude Split Errors", [ErrorType.SPLIT_ERROR.value]),
        ("Exclude Merge Errors", [ErrorType.MERGE_ERROR.value]),
        ("Exclude Poor Overlap", [ErrorType.POOR_OVERLAP.value]),
        ("Exclude Boundary Errors", [ErrorType.BOUNDARY_ERROR.value]),
        ("Exclude Split & Merge", [ErrorType.SPLIT_ERROR.value, ErrorType.MERGE_ERROR.value]),
        ("Exclude Detection Errors", [ErrorType.MISSING_DETECTION.value, ErrorType.FALSE_POSITIVE.value]),
        ("Exclude All Geometric Errors", [ErrorType.SPLIT_ERROR.value, ErrorType.MERGE_ERROR.value, 
                                          ErrorType.POOR_OVERLAP.value, ErrorType.BOUNDARY_ERROR.value]),
    ]
    
    rows = []
    for scenario_name, exclude_types in scenarios:
        metrics = compute_conditional_metrics(error_cases, exclude_types)
        rows.append({
            'Scenario': scenario_name,
            'Precision': f"{metrics['precision']:.3f}",
            'Recall': f"{metrics['recall']:.3f}",
            'F1': f"{metrics['f1']:.3f}",
            'TP': metrics['tp'],
            'FP': metrics['fp'],
            'FN': metrics['fn'],
            'Cases': metrics['total_cases']
        })
    
    df = pd.DataFrame(rows)
    return df

print("✓ Conditional metrics functions loaded")

In [ ]:
# Create conditional metrics table for IoU 0.3
conditional_metrics_03 = create_conditional_metrics_table(error_cases_03, iou_threshold=0.3)

print("\n" + "="*80)
print("CONDITIONAL METRICS TABLE (IoU threshold = 0.3)")
print("="*80)
print(conditional_metrics_03.to_string(index=False))
print("="*80)

In [ ]:
# Create conditional metrics table for IoU 0.5
conditional_metrics_05 = create_conditional_metrics_table(error_cases_05, iou_threshold=0.5)

print("\n" + "="*80)
print("CONDITIONAL METRICS TABLE (IoU threshold = 0.5)")
print("="*80)
print(conditional_metrics_05.to_string(index=False))
print("="*80)

## Detailed Error Type Examples

Now let's visualize specific examples of each major error type to understand what's happening.

In [ ]:
# # Visualize specific examples of key error types
# key_error_types = [
#     ErrorType.TRUE_POSITIVE.value,
#     ErrorType.SPLIT_ERROR.value,
#     ErrorType.MERGE_ERROR.value,
#     ErrorType.MISSING_DETECTION.value,
# ]

# for error_type in key_error_types:
#     type_cases = [case for case in error_cases_03 if case['error_type'] == error_type]
#     if len(type_cases) > 0:
#         print(f"\n{'='*80}")
#         print(f"EXAMPLES: {error_type.upper()} ({len(type_cases)} total cases)")
#         print(f"{'='*80}\n")
        
#         # Show 2 examples of each type
#         for i, case in enumerate(type_cases[:2]):
#             visualize_error_case(
#                 case,
#                 ground_truth_gdf,
#                 consensus_crowns_gdf,
#                 ortho_path,
#                 title=f"{error_type} - Example {i+1}",
#                 figsize=(15, 5)
#             )

In [ ]:
# # Visualize overlap: consensus crowns vs ground truth (OM1 coordinate system)
# import matplotlib.pyplot as plt

# def _sample_for_plot(gdf, max_items=200, seed=7):
#     if len(gdf) <= max_items:
#         return gdf
#     return gdf.sample(n=max_items, random_state=seed)

# def plot_gt_vs_consensus_overlap(gt_gdf, cons_gdf, title="GT vs Consensus Overlap", max_items=200):
#     if gt_gdf is None or cons_gdf is None or gt_gdf.empty or cons_gdf.empty:
#         print("No data to plot.")
#         return
#     gt = _sample_for_plot(gt_gdf, max_items=max_items)
#     cons = _sample_for_plot(cons_gdf, max_items=max_items)
#     fig, ax = plt.subplots(1, 1, figsize=(10, 10))
#     gt.plot(ax=ax, facecolor='none', edgecolor='blue', linewidth=1.0, alpha=0.6, label='Ground Truth')
#     cons.plot(ax=ax, facecolor='none', edgecolor='red', linewidth=1.2, alpha=0.7, label='Consensus')
#     ax.set_aspect('equal')
#     ax.set_title(title)
#     ax.grid(True, alpha=0.2)
#     # Build legend manually for geopandas
#     from matplotlib.lines import Line2D
#     legend_elements = [
#         Line2D([0], [0], color='blue', lw=2, label='Ground Truth'),
#         Line2D([0], [0], color='red', lw=2, label='Consensus'),
#     ]
#     ax.legend(handles=legend_elements, loc='best')
#     plt.tight_layout()
#     plt.show()

# plot_gt_vs_consensus_overlap(ground_truth_gdf, consensus_crowns_gdf, title="GT vs Consensus Crowns (OM1/UTM)")

## 10. Phenology Signal Analysis on Consensus Crowns

Extract robust RGB-based phenology signals from fixed consensus crowns across all orthomosaics.
This section computes raw features, quality-control flags, date-wise normalization, and quick diagnostics for temporal patterns.

In [18]:
import cv2
import matplotlib.colors as mcolors

# Optional texture features (if available)
SKIMAGE_TEXTURE_AVAILABLE = True
try:
    from skimage.feature import graycomatrix, graycoprops
except Exception:
    SKIMAGE_TEXTURE_AVAILABLE = False

PHENO_OUTPUT_DIR = Path('../../output')
PHENO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional: replace with real acquisition dates if available
OM_DATE_MAP = {om_id: f"OM{om_id}" for om_id in tracker.om_ids}


def _safe_div(num, den, eps=1e-8):
    return num / (den + eps)


def _shannon_entropy(values, bins=64):
    if values.size == 0:
        return np.nan
    hist, _ = np.histogram(values, bins=bins, range=(0, 1), density=True)
    hist = hist[hist > 0]
    if hist.size == 0:
        return np.nan
    return float(-np.sum(hist * np.log2(hist)))


def _compute_patch_features(patch: np.ndarray) -> Dict[str, float]:
    if patch is None or patch.size == 0:
        return {}

    if patch.ndim != 3 or patch.shape[2] < 3:
        return {}

    rgb = patch[..., :3].astype(np.float32)

    # Handle no-data pixels (all channels zero) and invalid numeric values
    finite_mask = np.isfinite(rgb).all(axis=2)
    nonzero_mask = (rgb.sum(axis=2) > 0)
    valid_mask = finite_mask & nonzero_mask

    total_px = rgb.shape[0] * rgb.shape[1]
    valid_px = int(valid_mask.sum())
    if total_px == 0 or valid_px < 20:
        return {
            'valid_pixel_fraction': valid_px / max(total_px, 1),
        }

    rgb = np.clip(rgb, 0, 255) / 255.0
    r = rgb[..., 0]
    g = rgb[..., 1]
    b = rgb[..., 2]

    denom = (r + g + b)
    gcc_map = _safe_div(g, denom)
    rcc_map = _safe_div(r, denom)
    exg_map = (2 * g - r - b)

    # HSV for vegetation-like masking
    hsv = mcolors.rgb_to_hsv(rgb)
    h, s, v = hsv[..., 0], hsv[..., 1], hsv[..., 2]

    dark_mask = (v < 0.12)
    veg_mask = (
        valid_mask
        & (h >= 0.18) & (h <= 0.48)
        & (s >= 0.15)
        & (v >= 0.12)
    )

    gray = (0.299 * r + 0.587 * g + 0.114 * b)
    gray_smooth = cv2.GaussianBlur(gray, (5, 5), 0)

    valid_gray = gray[valid_mask]
    valid_gray_smooth = gray_smooth[valid_mask]
    valid_r = r[valid_mask]
    valid_g = g[valid_mask]
    valid_b = b[valid_mask]
    valid_gcc = gcc_map[valid_mask]
    valid_rcc = rcc_map[valid_mask]
    valid_exg = exg_map[valid_mask]

    # Blur metric (larger = sharper)
    gray_u8 = np.clip(gray * 255, 0, 255).astype(np.uint8)
    lap_var = float(cv2.Laplacian(gray_u8, cv2.CV_64F).var())

    features = {
        'valid_pixel_fraction': valid_px / total_px,
        'shadow_fraction': float((dark_mask & valid_mask).sum() / max(valid_px, 1)),
        'veg_fraction_hsv': float(veg_mask.sum() / max(valid_px, 1)),
        'gcc_mean': float(np.mean(valid_gcc)),
        'rcc_mean': float(np.mean(valid_rcc)),
        'exg_mean': float(np.mean(valid_exg)),
        'r_mean': float(np.mean(valid_r)),
        'g_mean': float(np.mean(valid_g)),
        'b_mean': float(np.mean(valid_b)),
        'r_std': float(np.std(valid_r)),
        'g_std': float(np.std(valid_g)),
        'b_std': float(np.std(valid_b)),
        'r_cv': float(_safe_div(np.std(valid_r), np.mean(valid_r))),
        'g_cv': float(_safe_div(np.std(valid_g), np.mean(valid_g))),
        'b_cv': float(_safe_div(np.std(valid_b), np.mean(valid_b))),
        'gray_std': float(np.std(valid_gray)),
        'gray_std_smooth': float(np.std(valid_gray_smooth)),
        'gray_entropy': _shannon_entropy(valid_gray),
        'laplacian_var': lap_var,
    }

    if SKIMAGE_TEXTURE_AVAILABLE:
        try:
            quant = np.clip((gray * 31).astype(np.uint8), 0, 31)
            glcm = graycomatrix(
                quant,
                distances=[1],
                angles=[0, np.pi / 4, np.pi / 2, 3 * np.pi / 4],
                levels=32,
                symmetric=True,
                normed=True,
            )
            features['glcm_contrast'] = float(graycoprops(glcm, 'contrast').mean())
            features['glcm_homogeneity'] = float(graycoprops(glcm, 'homogeneity').mean())
            features['glcm_energy'] = float(graycoprops(glcm, 'energy').mean())
        except Exception:
            features['glcm_contrast'] = np.nan
            features['glcm_homogeneity'] = np.nan
            features['glcm_energy'] = np.nan
    else:
        features['glcm_contrast'] = np.nan
        features['glcm_homogeneity'] = np.nan
        features['glcm_energy'] = np.nan

    # QC flag
    features['is_bad_observation'] = bool(
        (features['valid_pixel_fraction'] < 0.60)
        or (features['shadow_fraction'] > 0.55)
        or (features['laplacian_var'] < 25)
    )

    return features


print(f"Texture features available: {SKIMAGE_TEXTURE_AVAILABLE}")
print("✓ Phenology feature functions loaded")

Texture features available: True
✓ Phenology feature functions loaded


In [19]:
# Extract per-date phenology features from consensus crowns (prefer larger external set)
import pandas as pd

tracker_crs = tracker.crowns_gdfs[tracker.om_ids[0]].crs if tracker.om_ids else None
if tracker_crs is not None and consensus_crowns_gdf.crs is not None and consensus_crowns_gdf.crs != tracker_crs:
    consensus_for_extract = consensus_crowns_gdf.to_crs(tracker_crs).copy()
else:
    consensus_for_extract = consensus_crowns_gdf.copy()

# Ensure a stable chain_id
if 'chain_id' not in consensus_for_extract.columns:
    consensus_for_extract = consensus_for_extract.reset_index(drop=True)
    consensus_for_extract['chain_id'] = consensus_for_extract.index.astype(int)

records = []
consensus_polys = {}

for _, row_cons in consensus_for_extract.iterrows():
    chain_id = int(row_cons['chain_id'])
    consensus_poly = row_cons.geometry
    if consensus_poly is None or consensus_poly.is_empty:
        continue
    consensus_polys[chain_id] = consensus_poly

    for om_id in tracker.om_ids:
        patch = extract_patch_for_polygon(om_id, consensus_poly, tracker)
        feats = _compute_patch_features(patch)

        row = {
            'chain_id': chain_id,
            'om_id': int(om_id),
            'date_label': OM_DATE_MAP.get(om_id, f"OM{om_id}"),
            'patch_h': int(patch.shape[0]) if isinstance(patch, np.ndarray) else np.nan,
            'patch_w': int(patch.shape[1]) if isinstance(patch, np.ndarray) else np.nan,
            'patch_exists': bool(isinstance(patch, np.ndarray) and patch.size > 0),
        }
        row.update(feats)
        records.append(row)

phenology_df = pd.DataFrame(records).sort_values(['chain_id', 'om_id']).reset_index(drop=True)

if phenology_df.empty:
    raise RuntimeError("No phenology features were extracted. Check consensus polygons and orthomosaic paths.")

# Ensure all expected columns exist, even if extraction failed for some rows
for col in [
    'valid_pixel_fraction', 'shadow_fraction', 'veg_fraction_hsv', 'gcc_mean', 'rcc_mean', 'exg_mean',
    'r_std', 'g_std', 'b_std', 'r_cv', 'g_cv', 'b_cv', 'gray_std', 'gray_std_smooth', 'gray_entropy',
    'laplacian_var', 'glcm_contrast', 'glcm_homogeneity', 'glcm_energy', 'is_bad_observation'
 ]:
    if col not in phenology_df.columns:
        phenology_df[col] = np.nan

phenology_df['is_bad_observation'] = phenology_df['is_bad_observation'].fillna(True).astype(bool)

raw_csv = PHENO_OUTPUT_DIR / 'consensus_phenology_features_raw.csv'
phenology_df.to_csv(raw_csv, index=False)

print(f"Consensus crowns used: {len(consensus_polys)}")
print(f"Rows extracted (trees × dates): {len(phenology_df)}")
print(f"Bad-observation rate: {phenology_df['is_bad_observation'].mean():.2%}")
print(f"Saved raw features: {raw_csv}")

display(
    phenology_df.groupby('om_id')[['patch_exists', 'is_bad_observation', 'veg_fraction_hsv', 'gcc_mean', 'rcc_mean']]
    .agg({
        'patch_exists': 'mean',
        'is_bad_observation': 'mean',
        'veg_fraction_hsv': 'median',
        'gcc_mean': 'median',
        'rcc_mean': 'median',
    })
    .rename(columns={
        'patch_exists': 'patch_exists_rate',
        'is_bad_observation': 'bad_obs_rate',
    })
)

Consensus crowns used: 80
Rows extracted (trees × dates): 560
Bad-observation rate: 5.18%
Saved raw features: ../../output/consensus_phenology_features_raw.csv


,patch_exists_rate,bad_obs_rate,veg_fraction_hsv,gcc_mean,rcc_mean
om_id,,,,,
1,1.0,0.0500,0.394783,0.405028,0.356774
2,1.0,0.1250,0.432669,0.414256,0.354627
3,1.0,0.0375,0.647011,0.431039,0.388921
4,1.0,0.0375,0.885409,0.483722,0.381114
5,1.0,0.0375,0.913650,0.502656,0.357358
6,1.0,0.0375,0.849477,0.397137,0.334393
7,1.0,0.0375,0.833480,0.398629,0.333288


In [20]:
# Entropy fix: use probability histogram (non-negative Shannon entropy)
def _shannon_entropy(values, bins=64):
    if values.size == 0:
        return np.nan
    hist, _ = np.histogram(values, bins=bins, range=(0, 1), density=False)
    total = hist.sum()
    if total <= 0:
        return np.nan
    probs = hist.astype(np.float64) / total
    probs = probs[probs > 0]
    return float(-np.sum(probs * np.log2(probs)))

print('✓ Updated entropy function to probability-based Shannon entropy')

✓ Updated entropy function to probability-based Shannon entropy


In [21]:
# Date-wise normalization to reduce illumination effects
feature_cols = [
    'gcc_mean', 'rcc_mean', 'exg_mean', 'veg_fraction_hsv',
    'gray_std', 'gray_std_smooth', 'gray_entropy',
    'r_cv', 'g_cv', 'b_cv', 'glcm_contrast', 'glcm_homogeneity',
]

norm_df = phenology_df.copy()

for col in feature_cols:
    if col not in norm_df.columns:
        continue

    # Robust z-score by date (median/MAD)
    med = norm_df.groupby('om_id')[col].transform('median')
    mad = norm_df.groupby('om_id')[col].transform(lambda x: np.median(np.abs(x - np.median(x))))
    scale = 1.4826 * mad.replace(0, np.nan)
    norm_df[f'{col}_rz_date'] = (norm_df[col] - med) / (scale + 1e-8)

    # Date-wise percentile rank
    norm_df[f'{col}_pct_date'] = norm_df.groupby('om_id')[col].rank(pct=True)

# Derive per-tree temporal descriptors (on non-bad observations)
clean_df = norm_df[~norm_df['is_bad_observation']].copy()

summary_rows = []
for chain_id, g in clean_df.groupby('chain_id'):
    g = g.sort_values('om_id')
    if g.empty:
        continue

    row = {
        'chain_id': int(chain_id),
        'n_obs_clean': int(len(g)),
        'om_start': int(g['om_id'].min()),
        'om_end': int(g['om_id'].max()),
    }

    for col in ['gcc_mean', 'rcc_mean', 'veg_fraction_hsv', 'gray_std_smooth', 'gray_entropy']:
        vals = g[col].dropna().values
        row[f'{col}_amplitude'] = float(np.nanmax(vals) - np.nanmin(vals)) if len(vals) > 1 else np.nan

    # Simple slope wrt OM index
    for col in ['gcc_mean', 'rcc_mean', 'veg_fraction_hsv', 'gray_std_smooth']:
        gg = g[['om_id', col]].dropna()
        if len(gg) >= 2:
            slope = np.polyfit(gg['om_id'].values, gg[col].values, 1)[0]
            row[f'{col}_slope_per_om'] = float(slope)
        else:
            row[f'{col}_slope_per_om'] = np.nan

    summary_rows.append(row)

phenology_tree_summary = pd.DataFrame(summary_rows).sort_values('chain_id').reset_index(drop=True)

norm_csv = PHENO_OUTPUT_DIR / 'consensus_phenology_features_normalized.csv'
summary_csv = PHENO_OUTPUT_DIR / 'consensus_phenology_tree_summary.csv'

norm_df.to_csv(norm_csv, index=False)
phenology_tree_summary.to_csv(summary_csv, index=False)

print(f"Saved normalized features: {norm_csv}")
print(f"Saved per-tree summary:  {summary_csv}")
print(f"Clean observations retained: {len(clean_df)}/{len(norm_df)} ({len(clean_df)/max(len(norm_df),1):.1%})")

display(phenology_tree_summary.head(10))

Saved normalized features: ../../output/consensus_phenology_features_normalized.csv
Saved per-tree summary:  ../../output/consensus_phenology_tree_summary.csv
Clean observations retained: 531/560 (94.8%)


,chain_id,n_obs_clean,om_start,om_end,gcc_mean_amplitude,rcc_mean_amplitude,veg_fraction_hsv_amplitude,gray_std_smooth_amplitude,gray_entropy_amplitude,gcc_mean_slope_per_om,rcc_mean_slope_per_om,veg_fraction_hsv_slope_per_om,gray_std_smooth_slope_per_om
0,1,7,1,7,0.286883,0.118101,0.506926,0.052827,41.942365,0.017916,-0.016512,0.080396,-0.005830
1,2,7,1,7,0.127832,0.054443,0.713547,0.037454,13.575440,0.006010,-0.004754,0.078477,-0.005035
2,3,7,1,7,0.149058,0.059627,0.923030,0.064357,17.991173,0.003215,-0.010485,0.148207,-0.008932
3,4,7,1,7,0.088442,0.068502,0.147007,0.061362,34.055269,-0.003729,-0.007506,0.024560,-0.007667
4,5,7,1,7,0.113706,0.054217,0.868710,0.076057,40.649243,0.008749,-0.008961,0.106213,-0.004458
5,6,7,1,7,0.092442,0.079146,0.823975,0.055536,43.943160,0.004765,-0.006000,0.089945,-0.002944
6,7,7,1,7,0.177303,0.050620,0.099070,0.077671,49.759620,-0.002794,-0.003677,0.014147,-0.011654
7,8,7,1,7,0.190631,0.073049,0.980169,0.071480,41.536432,0.002264,-0.011885,0.172120,-0.006334
8,9,7,1,7,0.097117,0.033619,0.654471,0.050144,24.899441,0.003694,-0.004899,0.073864,-0.002464
9,10,7,1,7,0.087313,0.121881,0.586059,0.044591,46.171832,-0.004665,0.006531,0.065079,0.001149


In [ ]:
# # Quick diagnostics: temporal medians and example tree trajectories
# plot_features = ['gcc_mean', 'rcc_mean', 'veg_fraction_hsv', 'gray_std_smooth', 'gray_entropy']

# date_stats = (
#     norm_df.groupby('om_id')[plot_features]
#     .agg(['median', 'mean', 'std'])
# )

# fig, axes = plt.subplots(2, 3, figsize=(16, 8))
# axes = axes.flatten()

# for i, feat in enumerate(plot_features):
#     ax = axes[i]
#     med = norm_df.groupby('om_id')[feat].median()
#     ax.plot(med.index, med.values, marker='o', linewidth=2)
#     ax.set_title(f'{feat} (median across trees)')
#     ax.set_xlabel('OM index')
#     ax.grid(alpha=0.3)

# axes[-1].axis('off')
# plt.tight_layout()

# overview_png = PHENO_OUTPUT_DIR / 'consensus_phenology_overview.png'
# plt.savefig(overview_png, dpi=180, bbox_inches='tight')
# plt.show()

# # Plot a few best-covered trees
# coverage = (
#     clean_df.groupby('chain_id')['om_id']
#     .nunique()
#     .sort_values(ascending=False)
# )
# example_chains = coverage.head(20).index.tolist()

# if example_chains:
#     fig, axes = plt.subplots(len(example_chains), 1, figsize=(12, 2.2 * len(example_chains)), sharex=True)
#     if len(example_chains) == 1:
#         axes = [axes]

#     for ax, chain_id in zip(axes, example_chains):
#         g = clean_df[clean_df['chain_id'] == chain_id].sort_values('om_id')
#         ax.plot(g['om_id'], g['gcc_mean'], marker='o', label='GCC')
#         ax.plot(g['om_id'], g['rcc_mean'], marker='s', label='RCC')
#         ax.plot(g['om_id'], g['veg_fraction_hsv'], marker='^', label='VegFraction')
#         ax.set_ylabel(f'Tree {chain_id}')
#         ax.grid(alpha=0.3)
#         ax.legend(loc='upper right', ncol=3, fontsize=8)

#     axes[-1].set_xlabel('OM index')
#     plt.tight_layout()
#     tree_png = PHENO_OUTPUT_DIR / 'consensus_phenology_example_trees.png'
#     plt.savefig(tree_png, dpi=180, bbox_inches='tight')
#     plt.show()
# else:
#     tree_png = None

# print(f'Saved overview plot: {overview_png}')
# if tree_png is not None:
#     print(f'Saved tree trajectories: {tree_png}')

# print('\nMedian phenology signals by OM:')
# display(norm_df.groupby('om_id')[plot_features].median().round(4))

In [ ]:
# # Plot all trees together: one graph per metric, different color per tree
# metrics_for_all_trees = ['gcc_mean', 'rcc_mean', 'veg_fraction_hsv'
#                         #  , 'gray_std_smooth', 
#                         #  'gray_entropy'
#                          ]

# all_tree_df = clean_df.copy().sort_values(['chain_id', 'om_id'])
# all_tree_ids = sorted(all_tree_df['chain_id'].unique())

# if len(all_tree_ids) == 0:
#     print('No clean tree observations available for plotting.')
# else:
#     n_metrics = len(metrics_for_all_trees)
#     fig, axes = plt.subplots(n_metrics, 1, figsize=(14, 3.0 * n_metrics), sharex=True)
#     if n_metrics == 1:
#         axes = [axes]

#     cmap = plt.cm.get_cmap('tab20', max(len(all_tree_ids), 1))
#     tree_to_color = {tree_id: cmap(i) for i, tree_id in enumerate(all_tree_ids)}

#     for ax, metric_name in zip(axes, metrics_for_all_trees):
#         for tree_id in all_tree_ids:
#             tree_ts = all_tree_df[all_tree_df['chain_id'] == tree_id]
#             ax.plot(
#                 tree_ts['om_id'],
#                 tree_ts[metric_name],
#                 marker='o',
#                 linewidth=1.4,
#                 alpha=0.9,
#                 color=tree_to_color[tree_id],
#                 label=f'Tree {tree_id}'
#             )
#         ax.set_title(f'{metric_name} (all trees)')
#         ax.set_ylabel(metric_name)
#         ax.grid(alpha=0.3)

#     axes[-1].set_xlabel('OM index')

#     handles, labels = axes[0].get_legend_handles_labels()
#     fig.legend(handles, labels, loc='center left', bbox_to_anchor=(1.01, 0.5), title='Trees', fontsize=8)

#     plt.tight_layout(rect=[0, 0, 0.86, 1])

#     all_trees_png = PHENO_OUTPUT_DIR / 'consensus_phenology_all_trees_per_metric.png'
#     plt.savefig(all_trees_png, dpi=180, bbox_inches='tight')
#     plt.show()

#     print(f'Saved all-trees per-metric plot: {all_trees_png}')

In [ ]:
# # Plot all trees together using robust date-normalized metrics (rz_date)
# metrics_rz_for_all_trees = [
#     'gcc_mean_rz_date',
#     # 'rcc_mean_rz_date',
#     'veg_fraction_hsv_rz_date',
#     # 'gray_std_smooth_rz_date',
#     # 'gray_entropy_rz_date',
# ]

# all_tree_df_rz = clean_df.copy().sort_values(['chain_id', 'om_id'])
# all_tree_ids_rz = sorted(all_tree_df_rz['chain_id'].unique())

# available_rz = [m for m in metrics_rz_for_all_trees if m in all_tree_df_rz.columns]

# if len(all_tree_ids_rz) == 0 or len(available_rz) == 0:
#     print('No robust-normalized metrics available for plotting.')
# else:
#     n_metrics = len(available_rz)
#     fig, axes = plt.subplots(n_metrics, 1, figsize=(14, 3.0 * n_metrics), sharex=True)
#     if n_metrics == 1:
#         axes = [axes]

#     cmap = plt.get_cmap('tab20', max(len(all_tree_ids_rz), 1))
#     tree_to_color_rz = {tree_id: cmap(i) for i, tree_id in enumerate(all_tree_ids_rz)}

#     for ax, metric_name in zip(axes, available_rz):
#         for tree_id in all_tree_ids_rz:
#             tree_ts = all_tree_df_rz[all_tree_df_rz['chain_id'] == tree_id]
#             ax.plot(
#                 tree_ts['om_id'],
#                 tree_ts[metric_name],
#                 marker='o',
#                 linewidth=1.4,
#                 alpha=0.9,
#                 color=tree_to_color_rz[tree_id],
#                 label=f'Tree {tree_id}'
#             )
#         ax.axhline(0.0, color='black', linewidth=1.0, linestyle='--', alpha=0.6)
#         ax.set_title(f'{metric_name} (all trees, robust date-normalized)')
#         ax.set_ylabel(metric_name)
#         ax.grid(alpha=0.3)

#     axes[-1].set_xlabel('OM index')

#     handles, labels = axes[0].get_legend_handles_labels()
#     fig.legend(handles, labels, loc='center left', bbox_to_anchor=(1.01, 0.5), title='Trees', fontsize=8)

#     plt.tight_layout(rect=[0, 0, 0.86, 1])

#     all_trees_rz_png = PHENO_OUTPUT_DIR / 'consensus_phenology_all_trees_per_metric_rz_date.png'
#     plt.savefig(all_trees_rz_png, dpi=180, bbox_inches='tight')
#     plt.show()

#     print(f'Saved all-trees robust-normalized plot: {all_trees_rz_png}')

In [ ]:
# Spatial view of all consensus crowns + outlier highlighting from rz_date metrics
rz_cols_for_outlier = [
    'gcc_mean_rz_date',
    'rcc_mean_rz_date',
    'veg_fraction_hsv_rz_date',
    'gray_std_smooth_rz_date',
    'gray_entropy_rz_date',
]

available_rz_cols = [c for c in rz_cols_for_outlier if c in norm_df.columns]
if len(available_rz_cols) == 0:
    print('No rz_date columns found in norm_df. Run normalization cell first.')
else:
    # Aggregate per tree: robust outlier score from median absolute rz across dates/metrics
    outlier_scores = []
    for tree_id, g in norm_df.groupby('chain_id'):
        vals = g[available_rz_cols].to_numpy(dtype=float)
        vals = vals[np.isfinite(vals)]
        if vals.size == 0:
            score = np.nan
            max_abs = np.nan
        else:
            abs_vals = np.abs(vals)
            score = float(np.nanmedian(abs_vals))
            max_abs = float(np.nanmax(abs_vals))
        outlier_scores.append({
            'chain_id': int(tree_id),
            'outlier_score_median_abs_rz': score,
            'outlier_score_max_abs_rz': max_abs,
        })

    outlier_df = pd.DataFrame(outlier_scores)

    # Build consensus GeoDataFrame from already-computed polygons
    geoms = []
    chain_ids = []
    for cid, geom in consensus_polys.items():
        if geom is not None and not geom.is_empty:
            chain_ids.append(int(cid))
            geoms.append(geom)

    consensus_vis_gdf = gpd.GeoDataFrame(
        {'chain_id': chain_ids, 'geometry': geoms},
        crs=tracker.crowns_gdfs[tracker.om_ids[0]].crs if tracker.om_ids else None,
    )

    consensus_vis_gdf = consensus_vis_gdf.merge(outlier_df, on='chain_id', how='left')

    # Determine top outliers for annotation
    n_label = 20
    top_outliers = consensus_vis_gdf.nlargest(n_label, 'outlier_score_median_abs_rz')

    fig, ax = plt.subplots(1, 1, figsize=(12, 10))

    # Base outlines
    consensus_vis_gdf.plot(
        ax=ax,
        facecolor='none',
        edgecolor='lightgray',
        linewidth=1.0,
        alpha=0.8,
    )

    # Outlier-colored fill
    consensus_vis_gdf.plot(
        ax=ax,
        column='outlier_score_median_abs_rz',
        cmap='plasma',
        linewidth=0.8,
        edgecolor='black',
        alpha=0.65,
        legend=True,
        legend_kwds={'label': 'Median |rz_date| outlier score', 'shrink': 0.8},
    )

    # Label strongest outliers
    for _, row in top_outliers.iterrows():
        c = row.geometry.centroid
        ax.text(
            c.x,
            c.y,
            f"T{int(row['chain_id'])}\n{row['outlier_score_median_abs_rz']:.2f}",
            fontsize=8,
            ha='center',
            va='center',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8, edgecolor='black')
        )

    ax.set_title('Consensus Crowns Spatial Map with Phenology Outlier Scores', fontsize=13, fontweight='bold')
    ax.set_aspect('equal')
    ax.grid(alpha=0.25)

    spatial_outlier_png = PHENO_OUTPUT_DIR / 'consensus_crowns_spatial_outliers.png'
    plt.tight_layout()
    plt.savefig(spatial_outlier_png, dpi=180, bbox_inches='tight')
    plt.show()

    print(f'Saved spatial outlier map: {spatial_outlier_png}')
    print('\nTop outlier trees (by median absolute rz_date):')
    display(top_outliers[['chain_id', 'outlier_score_median_abs_rz', 'outlier_score_max_abs_rz']].round(3))

In [ ]:
# # Visual audit: show all 7 consensus-crown chips per tree (for trees with metrics)
# # This helps inspect whether metric changes match visual crown changes.

# if 'consensus_polys' not in globals() or len(consensus_polys) == 0:
#     print('consensus_polys not found; rebuilding from full chains...')
#     full_chains = chain_categories.get('full_width_1', []) + chain_categories.get('full_branching', [])
#     consensus_polys = {}
#     for chain_id, chain in enumerate(full_chains):
#         poly = consensus_medoid(chain, tracker)
#         if poly is not None and not poly.is_empty:
#             consensus_polys[chain_id] = poly

# if 'norm_df' not in globals() or norm_df.empty:
#     raise RuntimeError('norm_df is not available. Run phenology extraction and normalization cells first.')

# metric_tree_ids = sorted(norm_df['chain_id'].dropna().astype(int).unique().tolist())
# plot_tree_ids = [tree_id for tree_id in metric_tree_ids if tree_id in consensus_polys]

# if len(plot_tree_ids) == 0:
#     print('No trees available with both metrics and consensus polygons.')
# else:
#     seq_dir = PHENO_OUTPUT_DIR / 'consensus_crown_sequences'
#     seq_dir.mkdir(parents=True, exist_ok=True)

#     print(f'Generating visual sequences for {len(plot_tree_ids)} trees × {len(tracker.om_ids)} OMs...')

#     for tree_id in plot_tree_ids:
#         consensus_poly = consensus_polys[tree_id]
#         fig, axes = plt.subplots(1, len(tracker.om_ids), figsize=(3.0 * len(tracker.om_ids), 3.8), sharex=False, sharey=False)

#         if len(tracker.om_ids) == 1:
#             axes = [axes]

#         for ax, om_id in zip(axes, tracker.om_ids):
#             patch = extract_patch_for_polygon(om_id, consensus_poly, tracker)
#             row = norm_df[(norm_df['chain_id'] == tree_id) & (norm_df['om_id'] == om_id)]

#             if patch is not None and isinstance(patch, np.ndarray) and patch.size > 0:
#                 ax.imshow(np.clip(patch[..., :3], 0, 255).astype(np.uint8))
#             else:
#                 ax.text(0.5, 0.5, 'No patch', ha='center', va='center', fontsize=9, color='red')

#             if len(row) > 0:
#                 gcc = row['gcc_mean'].iloc[0] if 'gcc_mean' in row else np.nan
#                 rcc = row['rcc_mean'].iloc[0] if 'rcc_mean' in row else np.nan
#                 veg = row['veg_fraction_hsv'].iloc[0] if 'veg_fraction_hsv' in row else np.nan
#                 bad = row['is_bad_observation'].iloc[0] if 'is_bad_observation' in row else False
#                 bad_tag = '⚠' if bool(bad) else ''
#                 ax.set_title(
#                     f"OM{om_id}{bad_tag}\nG:{gcc:.3f} R:{rcc:.3f}\nV:{veg:.3f}",
#                     fontsize=8
#                 )
#             else:
#                 ax.set_title(f'OM{om_id}\n(no metrics)', fontsize=8)

#             ax.axis('off')

#         fig.suptitle(f'Tree {tree_id} - Consensus Crown Across OMs', fontsize=12, fontweight='bold')
#         plt.tight_layout()

#         out_png = seq_dir / f'tree_{tree_id:02d}_consensus_sequence.png'
#         fig.savefig(out_png, dpi=160, bbox_inches='tight')
#         plt.show()
#         plt.close(fig)

#     print(f'Saved per-tree sequence figures to: {seq_dir}')

In [ ]:

# # Visual audit: texture/heterogeneity + normalised (rz_date) metrics per-tree chip sequence
# # Each chip title shows: (1) raw texture metrics, (2) rz_date normalised colour+texture metrics

# if 'consensus_polys' not in globals() or len(consensus_polys) == 0:
#     print('consensus_polys not found; rebuilding from full chains...')
#     full_chains = chain_categories.get('full_width_1', []) + chain_categories.get('full_branching', [])
#     consensus_polys = {}
#     for chain_id, chain in enumerate(full_chains):
#         poly = consensus_medoid(chain, tracker)
#         if poly is not None and not poly.is_empty:
#             consensus_polys[chain_id] = poly

# if 'norm_df' not in globals() or norm_df.empty:
#     raise RuntimeError('norm_df not available. Run phenology extraction and normalisation cells first.')

# metric_tree_ids = sorted(norm_df['chain_id'].dropna().astype(int).unique().tolist())
# plot_tree_ids = [tid for tid in metric_tree_ids if tid in consensus_polys]

# tex_seq_dir = PHENO_OUTPUT_DIR / 'consensus_crown_sequences_texture_norm'
# tex_seq_dir.mkdir(parents=True, exist_ok=True)

# print(f'Generating texture+norm sequences for {len(plot_tree_ids)} trees...')

# def _get(row, col):
#     """Safely retrieve a scalar from a filtered DataFrame row."""
#     return row[col].iloc[0] if (len(row) > 0 and col in row.columns) else np.nan

# for tree_id in plot_tree_ids:
#     consensus_poly = consensus_polys[tree_id]
#     n_oms = len(tracker.om_ids)
#     fig, axes = plt.subplots(1, n_oms, figsize=(3.2 * n_oms, 4.6), sharex=False, sharey=False)
#     if n_oms == 1:
#         axes = [axes]

#     for ax, om_id in zip(axes, tracker.om_ids):
#         patch = extract_patch_for_polygon(om_id, consensus_poly, tracker)
#         row = norm_df[(norm_df['chain_id'] == tree_id) & (norm_df['om_id'] == om_id)]

#         if patch is not None and isinstance(patch, np.ndarray) and patch.size > 0:
#             ax.imshow(np.clip(patch[..., :3], 0, 255).astype(np.uint8))
#         else:
#             ax.text(0.5, 0.5, 'No patch', ha='center', va='center', fontsize=8, color='red')

#         bad      = bool(_get(row, 'is_bad_observation')) if len(row) > 0 else False
#         bad_tag  = '⚠' if bad else ''

#         # raw texture metrics
#         gstd     = _get(row, 'gray_std_smooth')
#         ent      = _get(row, 'gray_entropy')
#         glcm_c   = _get(row, 'glcm_contrast')
#         lap      = _get(row, 'laplacian_var')

#         # rz_date normalised colour + texture metrics
#         gcc_rz   = _get(row, 'gcc_mean_rz_date')
#         rcc_rz   = _get(row, 'rcc_mean_rz_date')
#         veg_rz   = _get(row, 'veg_fraction_hsv_rz_date')
#         gstd_rz  = _get(row, 'gray_std_smooth_rz_date')
#         ent_rz   = _get(row, 'gray_entropy_rz_date')

#         def _fmt(v, fmt='.2f'):
#             return f'{v:{fmt}}' if not (isinstance(v, float) and np.isnan(v)) else 'nan'

#         title = (
#             f"OM{om_id}{bad_tag}\n"
#             f"Gstd:{_fmt(gstd,'.1f')} Ent:{_fmt(ent,'.2f')}\n"
#             f"GLCM:{_fmt(glcm_c,'.1f')} Lap:{_fmt(lap,'.0f')}\n"
#             f"Grz:{_fmt(gcc_rz)} Rrz:{_fmt(rcc_rz)} Vrz:{_fmt(veg_rz)}\n"
#             f"GSrz:{_fmt(gstd_rz)} Erz:{_fmt(ent_rz)}"
#         )
#         ax.set_title(title, fontsize=7, linespacing=1.3)
#         ax.axis('off')

#     fig.suptitle(f'Tree {tree_id} – Texture & Normalised Metrics', fontsize=11, fontweight='bold')
#     plt.tight_layout()

#     out_png = tex_seq_dir / f'tree_{tree_id:02d}_texture_norm_sequence.png'
#     fig.savefig(out_png, dpi=160, bbox_inches='tight')
#     plt.show()
#     plt.close(fig)

# print(f'Saved texture+norm sequences to: {tex_seq_dir}')


## 11. Phenological Pattern Discovery — Clustering

With 7 orthomosaics, we work in a low-temporal-density regime.  
The strategy here is **feature-first clustering based on season shape**:

1. **Phenometrics** — compact per-tree descriptors (amplitude, peak timing, directional trend)  
2. **Normalized time series feature matrix** — per-tree min–max normalization per metric (shape, not level) → concatenated into a feature vector  
3. **Hierarchical clustering** (Ward linkage) — identify groups of trees with similar seasonal patterns  
4. **Cluster visualization** — median trajectories, spatial layout, and PCA biplot

The goal is to see whether trees that are spatially close (same species niche?) or that respond similarly to the season cluster together naturally.

In [22]:
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster, leaves_list
from scipy.interpolate import interp1d
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
import warnings

# ── Configuration ──────────────────────────────────────────────────────────────
CLUST_METRICS = [
    # colour / vegetation signal
    'gcc_mean', 'rcc_mean', 'veg_fraction_hsv',
    # spatial texture & heterogeneity
    'gray_std_smooth', 'gray_entropy',
    'glcm_contrast', 'glcm_homogeneity', 'glcm_energy',
    'laplacian_var',
]
N_CLUSTERS = 6          # elbow plot in next cell helps pick the right k
OM_IDS     = sorted(norm_df['om_id'].unique())   # [1..7]
T          = len(OM_IDS)

# ── Step 1: pivot each metric to (tree × OM), NaN-out bad observations ─────────
raw_series: Dict[str, pd.DataFrame] = {}

for metric in CLUST_METRICS:
    if metric not in norm_df.columns:
        print(f"  ⚠ '{metric}' not in norm_df — skipping")
        continue
    wide = (
        norm_df[['chain_id', 'om_id', metric, 'is_bad_observation']]
        .copy()
        .assign(**{metric: lambda d, m=metric: d[m].where(~d['is_bad_observation'].astype(bool))})
        .pivot(index='chain_id', columns='om_id', values=metric)
        .reindex(columns=OM_IDS)
    )
    raw_series[metric] = wide

# Keep only metrics that were actually found
CLUST_METRICS = [m for m in CLUST_METRICS if m in raw_series]
print(f"Active metrics ({len(CLUST_METRICS)}): {CLUST_METRICS}")

# ── Step 2: linear interpolation within each tree's series ─────────────────────
def _interp_series(row: np.ndarray, om_ids: List[int]) -> np.ndarray:
    """Linear interpolation over NaN gaps; edge-fill at boundaries."""
    xi = np.array(om_ids, dtype=float)
    yi = row.astype(float)
    valid = np.isfinite(yi)
    if valid.sum() < 2:
        fillval = yi[valid][0] if valid.sum() == 1 else 0.0
        return np.where(valid, yi, fillval)
    f = interp1d(xi[valid], yi[valid], kind='linear', bounds_error=False,
                 fill_value=(yi[valid][0], yi[valid][-1]))
    return f(xi)

interp_series: Dict[str, np.ndarray] = {}
tree_ids_all = None

for metric in CLUST_METRICS:
    wide = raw_series[metric]
    if tree_ids_all is None:
        tree_ids_all = wide.index.tolist()
    mat = np.vstack([_interp_series(row.values, OM_IDS) for _, row in wide.iterrows()])
    interp_series[metric] = mat   # (N_trees, T)

tree_ids_arr = np.array(tree_ids_all)
N_trees = len(tree_ids_arr)

# ── Step 3: per-tree min–max normalisation (captures season shape, not level) ──
normed_series: Dict[str, np.ndarray] = {}

for metric, mat in interp_series.items():
    normed = np.zeros_like(mat)
    for i in range(N_trees):
        mn, mx = mat[i].min(), mat[i].max()
        rng = mx - mn
        normed[i] = (mat[i] - mn) / rng if rng > 1e-9 else np.zeros(T)
    normed_series[metric] = normed

# ── Step 4: scalar phenometrics (used only for post-hoc labelling) ─────────────
pheno_records = []
for i, tree_id in enumerate(tree_ids_arr):
    rec = {'chain_id': int(tree_id)}
    for metric in CLUST_METRICS:
        raw  = interp_series[metric][i]
        norm = normed_series[metric][i]
        amp       = raw.max() - raw.min()
        peak_om   = OM_IDS[int(np.argmax(raw))]
        trough_om = OM_IDS[int(np.argmin(raw))]
        slope     = float(np.polyfit(np.arange(T), raw, 1)[0])
        diffs     = np.diff(norm)
        signs     = np.sign(diffs[np.abs(diffs) > 0.05])
        dir_changes = int(np.sum(np.diff(signs) != 0)) if len(signs) > 1 else 0
        rec[f'{metric}_amplitude']   = float(amp)
        rec[f'{metric}_peak_om']     = float(peak_om)
        rec[f'{metric}_trough_om']   = float(trough_om)
        rec[f'{metric}_slope']       = float(slope)
        rec[f'{metric}_dir_changes'] = float(dir_changes)
    pheno_records.append(rec)

phenometrics_df = pd.DataFrame(pheno_records).set_index('chain_id')

# ── Step 5: flat feature matrix ────────────────────────────────────────────────
feature_blocks = [normed_series[m] for m in CLUST_METRICS]  # each (N_trees, T)
X = np.hstack(feature_blocks)   # (N_trees, T * len(CLUST_METRICS))

# Impute any residual NaN with column medians
col_meds = np.nanmedian(X, axis=0)
for col_idx in range(X.shape[1]):
    nan_mask = ~np.isfinite(X[:, col_idx])
    X[nan_mask, col_idx] = col_meds[col_idx]

print(f"\nTrees:           {N_trees}")
print(f"OM dates:        {OM_IDS}")
print(f"Feature matrix:  {X.shape}  ({T} time-steps × {len(CLUST_METRICS)} metrics)")
print()
print("Phenometrics — amplitudes (first 5 trees):")
display(phenometrics_df[[f'{m}_amplitude' for m in CLUST_METRICS]].head())


Active metrics (9): ['gcc_mean', 'rcc_mean', 'veg_fraction_hsv', 'gray_std_smooth', 'gray_entropy', 'glcm_contrast', 'glcm_homogeneity', 'glcm_energy', 'laplacian_var']

Trees:           80
OM dates:        [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]
Feature matrix:  (80, 63)  (7 time-steps × 9 metrics)

Phenometrics — amplitudes (first 5 trees):


,gcc_mean_amplitude,rcc_mean_amplitude,veg_fraction_hsv_amplitude,gray_std_smooth_amplitude,gray_entropy_amplitude,glcm_contrast_amplitude,glcm_homogeneity_amplitude,glcm_energy_amplitude,laplacian_var_amplitude
chain_id,,,,,,,,,
1,0.286883,0.118101,0.506926,0.052827,41.942365,1.303027,0.062802,0.085694,520.148766
2,0.127832,0.054443,0.713547,0.037454,13.575440,1.989915,0.092635,0.018138,555.507261
3,0.149058,0.059627,0.923030,0.064357,17.991173,5.418418,0.053817,0.007442,963.788841
4,0.088442,0.068502,0.147007,0.061362,34.055269,3.063260,0.065448,0.013785,519.977528
5,0.113706,0.054217,0.868710,0.076057,40.649243,3.920206,0.122475,0.038152,543.510563


In [ ]:
# ── Hierarchical clustering (Ward linkage) ────────────────────────────────────
Z = linkage(X, method='ward', metric='euclidean')

# ── Elbow plot: last 15 merge distances ───────────────────────────────────────
# The "elbow" (biggest jump in merge distance going right-to-left) suggests
# the natural number of clusters. Red dashed line = current N_CLUSTERS cut.
last_n = 15
merge_dists = Z[-last_n:, 2][::-1]   # sorted from k=2 upward
fig, ax = plt.subplots(figsize=(8, 3.5))
ks = np.arange(2, last_n + 2)
ax.plot(ks, merge_dists, 'o-', color='steelblue', linewidth=2)
ax.axvline(N_CLUSTERS, color='tomato', linestyle='--', linewidth=1.5,
           label=f'Current N_CLUSTERS = {N_CLUSTERS}')
# Annotate the biggest gap
gaps = np.diff(merge_dists)
best_k = int(ks[np.argmax(gaps) + 1])
ax.axvline(best_k, color='green', linestyle=':', linewidth=1.5,
           label=f'Largest gap at k={best_k}')
ax.set_xlabel('Number of clusters k')
ax.set_ylabel('Merge distance (Ward)')
ax.set_title('Cluster count elbow plot — pick k at the biggest jump', fontsize=11)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f"Suggested k by largest merge-gap: {best_k}")
print(f"Using N_CLUSTERS = {N_CLUSTERS}  (change in feature matrix cell to explore)")

# ── Full dendrogram ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
dend = dendrogram(
    Z,
    labels=[f'T{tid}' for tid in tree_ids_arr],
    leaf_rotation=90,
    leaf_font_size=9,
    color_threshold=0.7 * Z[-N_CLUSTERS + 1, 2],
    ax=ax,
)
ax.set_title(
    f'Hierarchical Clustering Dendrogram\n'
    f'(Ward linkage · Euclidean · {len(CLUST_METRICS)} metrics × {T} time-steps)',
    fontsize=12, fontweight='bold'
)
ax.set_xlabel('Tree ID', fontsize=10)
ax.set_ylabel('Linkage distance', fontsize=10)
ax.axhline(y=Z[-N_CLUSTERS + 1, 2], color='red', linestyle='--',
           linewidth=1.4, alpha=0.7, label=f'Cut for k={N_CLUSTERS}')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
dend_png = PHENO_OUTPUT_DIR / 'phenology_clustering_dendrogram.png'
plt.savefig(dend_png, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved: {dend_png}')

# ── Cut the tree ───────────────────────────────────────────────────────────────
cluster_labels = fcluster(Z, N_CLUSTERS, criterion='maxclust')   # 1-indexed

cluster_df = pd.DataFrame({
    'chain_id': tree_ids_arr,
    'cluster':  cluster_labels,
}).set_index('chain_id')

# Attach cluster to phenometrics
phenometrics_df = phenometrics_df.join(cluster_df)

# Summary
print(f'\nCluster assignments (k={N_CLUSTERS}):')
print(cluster_df['cluster'].value_counts().sort_index().rename('n_trees').to_frame())
print()
print('Per-cluster medians (key phenometrics):')
colour_cols = [f'{m}_amplitude' for m in ['gcc_mean', 'veg_fraction_hsv']] + \
              [f'{m}_peak_om'   for m in ['gcc_mean', 'veg_fraction_hsv']] + \
              [f'{m}_slope'     for m in ['gcc_mean', 'rcc_mean']]
texture_cols = [f'{m}_amplitude' for m in
                ['gray_std_smooth', 'gray_entropy', 'glcm_contrast', 'glcm_energy', 'laplacian_var']
                if f'{m}_amplitude' in phenometrics_df.columns]
display(
    phenometrics_df[['cluster'] + colour_cols + texture_cols]
    .groupby('cluster')
    .median()
    .round(4)
)


In [ ]:
# ── Per-cluster seasonal trajectories ─────────────────────────────────────────
# Rows = clusters, columns = metrics.
# Each panel: faint individual-tree lines + bold cluster-median + IQR shading.
# Both RAW (absolute level) and MIN-MAX NORMALISED (shape only) versions saved.

CLUSTER_PALETTE = plt.colormaps.get_cmap('tab10')
unique_clusts   = sorted(cluster_df['cluster'].unique())
cluster_colors  = {c: CLUSTER_PALETTE(i / 10.0) for i, c in enumerate(unique_clusts)}

for mode, series_dict, ylabel_sfx in [
    ('Raw values',         interp_series, ''),
    ('Min–max normalised', normed_series, ' (0–1 per tree)'),
]:
    n_c = len(unique_clusts)
    n_m = len(CLUST_METRICS)
    fig, axes = plt.subplots(n_c, n_m, figsize=(3.2 * n_m, 2.5 * n_c), sharex=True)
    if n_c == 1:
        axes = axes[np.newaxis, :]

    for row_idx, clust in enumerate(unique_clusts):
        tree_mask    = (cluster_df['cluster'] == clust).values
        tree_indices = np.where(tree_mask)[0]
        color        = cluster_colors[clust]

        for col_idx, metric in enumerate(CLUST_METRICS):
            ax  = axes[row_idx, col_idx]
            mat = series_dict[metric]

            for t_idx in tree_indices:
                ax.plot(OM_IDS, mat[t_idx], color=color, alpha=0.25, linewidth=1.0)

            if len(tree_indices) > 0:
                cluster_med = np.median(mat[tree_indices], axis=0)
                ax.plot(OM_IDS, cluster_med, color=color, linewidth=2.5,
                        marker='o', markersize=4,
                        label=f'C{clust} (n={len(tree_indices)})')

            if len(tree_indices) > 1:
                q25 = np.percentile(mat[tree_indices], 25, axis=0)
                q75 = np.percentile(mat[tree_indices], 75, axis=0)
                ax.fill_between(OM_IDS, q25, q75, color=color, alpha=0.12)

            if row_idx == 0:
                ax.set_title(metric, fontsize=8, fontweight='bold')
            if col_idx == 0:
                ax.set_ylabel(f'C{clust}\n{ylabel_sfx}', fontsize=8)
            ax.set_xticks(OM_IDS)
            ax.set_xticklabels([f'OM{o}' for o in OM_IDS], fontsize=6)
            ax.grid(alpha=0.25)

    fig.suptitle(
        f'Phenological Clusters — {mode}\n'
        f'(k={N_CLUSTERS}, Ward linkage, shading = IQR)',
        fontsize=12, fontweight='bold', y=1.01
    )
    plt.tight_layout()
    suffix   = 'raw' if 'Raw' in mode else 'norm'
    out_path = PHENO_OUTPUT_DIR / f'phenology_cluster_trajectories_{suffix}.png'
    plt.savefig(out_path, dpi=180, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out_path}')


In [ ]:
# ── Spatial map of phenological clusters ──────────────────────────────────────
#
# Key question: do trees of similar phenological pattern cluster spatially?
# (Same microhabitat / species patch) or are patterns distributed randomly?

from matplotlib.patches import Patch as MPatch

geoms_c, cids_c, clusts_c = [], [], []
for cid, geom in consensus_polys.items():
    if geom is None or geom.is_empty:
        continue
    cid_int = int(cid)
    if cid_int not in cluster_df.index:
        continue
    geoms_c.append(geom)
    cids_c.append(cid_int)
    clusts_c.append(int(cluster_df.loc[cid_int, 'cluster']))

cluster_spatial_gdf = gpd.GeoDataFrame(
    {'chain_id': cids_c, 'cluster': clusts_c, 'geometry': geoms_c},
    crs=tracker.crowns_gdfs[tracker.om_ids[0]].crs,
)

fig, ax = plt.subplots(figsize=(12, 10))

for clust, color in cluster_colors.items():
    sub = cluster_spatial_gdf[cluster_spatial_gdf['cluster'] == clust]
    n   = len(sub)
    sub.plot(ax=ax, color=color, edgecolor='black', linewidth=0.7, alpha=0.75)

    # Label each crown with its tree ID
    for _, row in sub.iterrows():
        cx, cy = row.geometry.centroid.x, row.geometry.centroid.y
        ax.text(cx, cy, str(int(row['chain_id'])),
                ha='center', va='center', fontsize=7, fontweight='bold',
                color='white',
                bbox=dict(boxstyle='round,pad=0.15', facecolor=color,
                          edgecolor='none', alpha=0.85))

legend_handles = [
    MPatch(color=cluster_colors[c], label=f'Cluster {c} (n={int((cluster_spatial_gdf["cluster"] == c).sum())})')
    for c in sorted(cluster_colors)
]
ax.legend(handles=legend_handles, loc='upper right', fontsize=10, title='Phenology Cluster')
ax.set_title('Spatial Distribution of Phenological Clusters', fontsize=13, fontweight='bold')
ax.set_aspect('equal')
ax.grid(alpha=0.25)
plt.tight_layout()

spatial_clust_png = PHENO_OUTPUT_DIR / 'phenology_cluster_spatial_map.png'
plt.savefig(spatial_clust_png, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved: {spatial_clust_png}')

# ── Phenometric heatmap: trees × summary features (sorted by cluster) ─────────
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

heat_cols = (
    [f'{m}_amplitude'   for m in CLUST_METRICS] +
    [f'{m}_peak_om'     for m in ['gcc_mean', 'veg_fraction_hsv']] +
    [f'{m}_slope'       for m in ['gcc_mean', 'rcc_mean', 'veg_fraction_hsv']] +
    [f'{m}_dir_changes' for m in ['gcc_mean', 'veg_fraction_hsv']]
)
heat_df = phenometrics_df[['cluster'] + heat_cols].sort_values('cluster')
heat_mat = MinMaxScaler().fit_transform(heat_df[heat_cols].values)  # 0–1 across trees per feature

row_colors_list = [cluster_colors[c] for c in heat_df['cluster']]

fig, axes = plt.subplots(1, 2, figsize=(16, max(5, N_trees * 0.38)),
                          gridspec_kw={'width_ratios': [0.04, 1]})

# Cluster color bar on left
ax_cb   = axes[0]
cb_arr  = np.array([[c] for c in heat_df['cluster'].values])
ax_cb.imshow([[cluster_colors[c]] for c in heat_df['cluster'].values],
             aspect='auto', interpolation='nearest')
ax_cb.set_xticks([])
ax_cb.set_yticks(range(N_trees))
ax_cb.set_yticklabels([f'T{tid}' for tid in heat_df.index], fontsize=8)
ax_cb.set_title('C', fontsize=8)

# Heatmap
ax_hm = axes[1]
im = ax_hm.imshow(heat_mat, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1, interpolation='nearest')
ax_hm.set_xticks(range(len(heat_cols)))
ax_hm.set_xticklabels(heat_cols, rotation=45, ha='right', fontsize=8)
ax_hm.set_yticks([])
ax_hm.set_title('Phenometrics Heatmap (min–max scaled, sorted by cluster)', fontsize=10, fontweight='bold')
plt.colorbar(im, ax=ax_hm, shrink=0.6, label='Value (0=min, 1=max across trees)')

plt.tight_layout()
heatmap_png = PHENO_OUTPUT_DIR / 'phenology_phenometrics_heatmap.png'
plt.savefig(heatmap_png, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved: {heatmap_png}')


In [ ]:
# ── PCA biplot: what drives the cluster separation? ───────────────────────────
#
# PCA on the flat feature matrix (min-max normalised time series).
# Biplot loadings point to which (metric, time-step) combinations vary most
# between trees — these are the phenological dimensions that matter.

pca  = PCA(n_components=min(6, X.shape[1], X.shape[0]))
X_pc = pca.fit_transform(X)
ev   = pca.explained_variance_ratio_ * 100

# ── PC1 vs PC2 scatter ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Panel A: PC1 vs PC2
ax = axes[0]
for clust, color in cluster_colors.items():
    mask = cluster_df['cluster'].values == clust
    ax.scatter(X_pc[mask, 0], X_pc[mask, 1],
               c=[color], s=90, edgecolors='black', linewidths=0.6,
               label=f'Cluster {clust}', zorder=3, alpha=0.9)
    for t_idx in np.where(mask)[0]:
        ax.annotate(f'T{tree_ids_arr[t_idx]}',
                    (X_pc[t_idx, 0], X_pc[t_idx, 1]),
                    fontsize=7, ha='left', va='bottom',
                    xytext=(3, 3), textcoords='offset points', alpha=0.75)

ax.set_xlabel(f'PC1 ({ev[0]:.1f}% var)', fontsize=10)
ax.set_ylabel(f'PC2 ({ev[1]:.1f}% var)', fontsize=10)
ax.set_title('PCA — PC1 vs PC2 (coloured by cluster)', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.axhline(0, color='grey', linewidth=0.5, linestyle='--')
ax.axvline(0, color='grey', linewidth=0.5, linestyle='--')
ax.grid(alpha=0.2)

# Panel B: PC loadings (biplot) — label by metric+OM
ax2 = axes[1]

# Build feature names: (metric × OM)
feature_names = [f'{m[:4]}_OM{o}' for m in CLUST_METRICS for o in OM_IDS]
loadings = pca.components_[:2].T   # (n_features, 2)

# Plot loading arrows — show only top N by PC1/PC2 magnitude for readability
top_n = 14
magnitudes = np.sqrt(loadings[:, 0]**2 + loadings[:, 1]**2)
top_idx = np.argsort(magnitudes)[-top_n:]

scale = 2.5
for i in top_idx:
    ax2.annotate(
        '', xy=(loadings[i, 0] * scale, loadings[i, 1] * scale),
        xytext=(0, 0),
        arrowprops=dict(arrowstyle='->', color='steelblue', lw=1.4)
    )
    ax2.text(
        loadings[i, 0] * scale * 1.12,
        loadings[i, 1] * scale * 1.12,
        feature_names[i],
        fontsize=7.5, color='steelblue', ha='center', va='center'
    )

ax2.scatter(X_pc[:, 0], X_pc[:, 1],
            c=[cluster_colors[c] for c in cluster_df['cluster'].values],
            s=60, edgecolors='black', linewidths=0.5, alpha=0.55, zorder=3)
ax2.set_xlim(-3.5, 3.5)
ax2.set_ylim(-3.5, 3.5)
ax2.set_xlabel(f'PC1 ({ev[0]:.1f}%)', fontsize=10)
ax2.set_ylabel(f'PC2 ({ev[1]:.1f}%)', fontsize=10)
ax2.set_title('Biplot — top-14 feature loadings\n(arrows = which metric×date drives separation)',
              fontsize=10, fontweight='bold')
ax2.axhline(0, color='grey', linewidth=0.5, linestyle='--')
ax2.axvline(0, color='grey', linewidth=0.5, linestyle='--')
ax2.grid(alpha=0.2)

plt.tight_layout()
pca_png = PHENO_OUTPUT_DIR / 'phenology_pca_biplot.png'
plt.savefig(pca_png, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved: {pca_png}')

# ── Scree + cumulative variance ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(range(1, len(ev) + 1), ev, color='steelblue', alpha=0.8)
ax.plot(range(1, len(ev) + 1), np.cumsum(ev), 'o-', color='tomato', label='Cumulative')
ax.axhline(80, color='grey', linestyle='--', linewidth=0.8, label='80% threshold')
ax.set_xlabel('PC')
ax.set_ylabel('% variance explained')
ax.set_title('PCA Scree Plot')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print('\nVariance explained per PC:',
      '  '.join([f'PC{i+1}={v:.1f}%' for i, v in enumerate(ev)]))


In [ ]:
# ── DTW distance matrix ────────────────────────────────────────────────────────
#
# With T=7, DTW is unstable for fine timing offsets, but it still usefully
# captures whether two trees are "shape-similar but shifted by a few dates".
# We compute it per-metric on normalised series and average across metrics.
#
# No external library needed — hand-rolled vanilla DTW with Sakoe-Chiba band.

def _dtw(s1: np.ndarray, s2: np.ndarray, window: int = 2) -> float:
    """Vanilla DTW with Sakoe-Chiba band (window in time-steps)."""
    n, m = len(s1), len(s2)
    D = np.full((n + 1, m + 1), np.inf)
    D[0, 0] = 0.0
    for i in range(1, n + 1):
        j_lo = max(1, i - window)
        j_hi = min(m, i + window)
        for j in range(j_lo, j_hi + 1):
            cost = (s1[i - 1] - s2[j - 1]) ** 2
            D[i, j] = cost + min(D[i - 1, j], D[i, j - 1], D[i - 1, j - 1])
    return float(D[n, m] ** 0.5)   # RMSE-normalised DTW distance

# Average DTW over all metrics (on min–max normalised series)
dtw_dist = np.zeros((N_trees, N_trees))
for metric in CLUST_METRICS:
    mat = normed_series[metric]
    for i in range(N_trees):
        for j in range(i + 1, N_trees):
            d = _dtw(mat[i], mat[j], window=2)
            dtw_dist[i, j] += d
            dtw_dist[j, i] += d
dtw_dist /= len(CLUST_METRICS)

# ── Plot DTW matrix reordered by Ward dendrogram leaves ───────────────────────
order   = leaves_list(Z)
dtw_ord = dtw_dist[np.ix_(order, order)]
labels_ord = [f'T{tree_ids_arr[i]}' for i in order]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

im = axes[0].imshow(dtw_ord, cmap='viridis_r', aspect='auto', interpolation='nearest')
axes[0].set_xticks(range(N_trees))
axes[0].set_xticklabels(labels_ord, rotation=90, fontsize=8)
axes[0].set_yticks(range(N_trees))
axes[0].set_yticklabels(labels_ord, fontsize=8)
axes[0].set_title('Mean DTW Distance Matrix\n(reordered by Ward dendrogram)', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=axes[0], shrink=0.8, label='Avg DTW distance (Sakoe-Chiba w=2)')

# Overlay cluster block borders
cumcount = 0
clust_order_vals = [int(cluster_df.loc[tree_ids_arr[o], 'cluster']) for o in order]
boundaries = []
for i in range(1, len(clust_order_vals)):
    if clust_order_vals[i] != clust_order_vals[i - 1]:
        boundaries.append(i - 0.5)
for b in boundaries:
    axes[0].axhline(b, color='white', linewidth=1.5)
    axes[0].axvline(b, color='white', linewidth=1.5)

# ── DTW-based clustering comparison ───────────────────────────────────────────
# Re-cluster on DTW distances; compare with Ward clusters
from scipy.cluster.hierarchy import linkage as sc_linkage

Z_dtw         = sc_linkage(squareform(dtw_dist), method='average')
dtw_labels_4  = fcluster(Z_dtw, N_CLUSTERS, criterion='maxclust')

# Contingency table: Ward cluster vs DTW cluster
ward_labels = cluster_df.loc[tree_ids_arr, 'cluster'].values
contingency  = pd.crosstab(
    pd.Series(ward_labels,  name='Ward cluster'),
    pd.Series(dtw_labels_4, name='DTW cluster'),
)
axes[1].axis('off')
tbl = axes[1].table(
    cellText  = contingency.values,
    rowLabels = [f'Ward C{r}' for r in contingency.index],
    colLabels = [f'DTW C{c}'  for c in contingency.columns],
    cellLoc   = 'center',
    loc       = 'center',
)
tbl.auto_set_font_size(True)
tbl.scale(1.4, 1.8)
axes[1].set_title(
    'Ward vs DTW Cluster Agreement\n(off-diagonal = trees assigned differently)',
    fontsize=11, fontweight='bold', y=0.85
)

plt.tight_layout()
dtw_png = PHENO_OUTPUT_DIR / 'phenology_dtw_distance_matrix.png'
plt.savefig(dtw_png, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved: {dtw_png}')

# Agreement summary
n_agree = int(np.sum(
    [1 for wc, dc in zip(ward_labels, dtw_labels_4)
     if wc == dc or True]  # always count — value is in contingency table
))
print('\nWard vs DTW contingency:')
display(contingency)
print('\nNote: DTW cluster labels are arbitrary — compare the block patterns, not numbers.')


In [ ]:

# ── Section 11.7  Tree crown image gallery, grouped by cluster ─────────────────
import warnings
import numpy as np
import matplotlib.pyplot as plt

PATCH_SIZE = 80
OM_DISPLAY = [1, 2, 3, 4, 5, 6, 7]

# Auto-build cluster titles from phenometrics median (gcc peak OM + veg amplitude)
def _auto_clust_name(clust_id):
    if 'cluster' not in phenometrics_df.columns:
        return f'Cluster {clust_id}'
    sub = phenometrics_df[phenometrics_df['cluster'] == clust_id]
    if sub.empty:
        return f'Cluster {clust_id}  (n=0)'
    n = len(sub)
    gcc_peak = sub['gcc_mean_peak_om'].median() if 'gcc_mean_peak_om' in sub else float('nan')
    veg_amp  = sub['veg_fraction_hsv_amplitude'].median() if 'veg_fraction_hsv_amplitude' in sub else float('nan')
    tex      = sub['gray_entropy_amplitude'].median() if 'gray_entropy_amplitude' in sub else float('nan')
    parts = [f'n={n}']
    if not np.isnan(gcc_peak): parts.append(f'GCC-peak OM{int(gcc_peak)}')
    if not np.isnan(veg_amp):  parts.append(f'veg-amp {veg_amp:.2f}')
    if not np.isnan(tex):      parts.append(f'ent-amp {tex:.2f}')
    return f'Cluster {clust_id}  —  ' + '  |  '.join(parts)

def _resize_chip(patch, size=PATCH_SIZE):
    if patch is None or patch.size == 0:
        return np.zeros((size, size, 3), dtype=np.uint8)
    try:
        import cv2
        return cv2.resize(patch, (size, size), interpolation=cv2.INTER_LINEAR)
    except Exception:
        return np.zeros((size, size, 3), dtype=np.uint8)

def _get_rgb_chip(om_id, poly, tracker):
    try:
        patch = extract_patch_for_polygon(om_id, poly, tracker)
        if patch is None or patch.size == 0:
            return None
        if patch.ndim == 3 and patch.shape[0] in (1, 3, 4):
            patch = np.moveaxis(patch[:3], 0, -1)
        patch = patch[..., :3].astype(float)
        lo, hi = np.percentile(patch, [2, 98])
        if hi > lo:
            patch = np.clip((patch - lo) / (hi - lo), 0, 1)
        else:
            patch = np.zeros_like(patch)
        return (patch * 255).astype(np.uint8)
    except Exception:
        return None

n_cols = len(OM_DISPLAY)
gallery_paths = []
CLUSTER_PALETTE = plt.colormaps.get_cmap('tab10')

for clust_id in sorted(cluster_df['cluster'].unique()):
    trees = cluster_df[cluster_df['cluster'] == clust_id].index.tolist()
    n_rows = len(trees)

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(n_cols * 1.7, n_rows * 2.0),
        squeeze=False,
        gridspec_kw={'hspace': 0.08, 'wspace': 0.06},
    )
    bg = '#111122'
    fig.patch.set_facecolor(bg)

    for r, cid in enumerate(trees):
        poly = consensus_polys.get(cid)
        for c, om_id in enumerate(OM_DISPLAY):
            ax = axes[r, c]
            ax.set_facecolor(bg)
            ax.set_xticks([])
            ax.set_yticks([])
            for sp in ax.spines.values():
                sp.set_edgecolor('#334')
                sp.set_linewidth(0.5)
            if poly is not None:
                chip = _get_rgb_chip(om_id, poly, tracker)
                if chip is not None:
                    ax.imshow(_resize_chip(chip, PATCH_SIZE))
                else:
                    ax.text(0.5, 0.5, '✗', ha='center', va='center',
                            color='#888', fontsize=12, transform=ax.transAxes)
            else:
                ax.text(0.5, 0.5, '?', ha='center', va='center',
                        color='#f44', fontsize=12, transform=ax.transAxes)
            if r == 0:
                ax.set_title(f'OM{om_id}', color='white', fontsize=8, pad=4)
            if c == 0:
                ax.set_ylabel(f'T{cid}', rotation=0, labelpad=32,
                              va='center', color='white', fontsize=9, fontweight='bold')

    clust_color = CLUSTER_PALETTE((clust_id - 1) / max(N_CLUSTERS - 1, 1))
    title_str   = _auto_clust_name(clust_id)
    fig.suptitle(title_str, color=clust_color, fontsize=11, fontweight='bold', y=1.02)

    out_png = PHENO_OUTPUT_DIR.parent / f'phenology_cluster{clust_id}_tree_images.png'
    plt.savefig(out_png, dpi=130, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()
    gallery_paths.append(out_png)
    print(f'  ✓ Cluster {clust_id}  ({n_rows} trees)  →  {out_png.name}')

print(f'\nAll {len(gallery_paths)} cluster galleries saved.')


### Interpreting the clustering results

**What to look for**

| Plot | What it tells you |
|---|---|
| **Dendrogram** | How many natural groupings exist; how far apart clusters are (y-axis = dissimilarity). A long gap before the last merge → k is confident. |
| **Cluster trajectories (raw)** | Absolute level differences — e.g. one cluster might have consistently high GCC (dense canopy), another low. |
| **Cluster trajectories (normalised)** | Seasonal *shape* — one cluster might peak early (OM3), another late (OM6), or show double-peak (leaf flush + flower). |
| **Spatial map** | If clusters are spatially contiguous → trees of the same species/stand behave phenologically alike. Random scatter → individual variation dominates. |
| **PCA biplot** | Arrows pointing in the same direction = correlated features. Long arrows = high-variance features that split the clusters. `gcc_OMx` arrows at different angles = the split is *when* GCC peaks, not *how high* it gets. |
| **DTW matrix** | Dark diagonal blocks = trees that are self-similar in season shape. Agreement with Ward clustering validates that the structure is real. |

**Next steps**

- If spatial clustering is found → link to species layer / canopy structure data (LiDAR, field survey)
- If only 2–3 meaningful clusters emerge → use cluster label as a covariate in satellite calibration (different phenological response curves per cluster)
- With more flights (denser temporal sampling), replace the `_interp_series` step with TIMESAT-style smooth curve fitting (double-logistic) and derive SOS / EOS / peak-date with confidence intervals
- Add `flower_fraction` as a sixth metric (requires a simple HSV thresholding for white/yellow petals) — this is likely the highest-contrast variable for seasonal classes

## 12. Leaf Shed Classifier — Deciduousness Detection and Phenophase Timing

**Goal**: answer two questions for every tracked tree, using only its time series of spectral/texture signals:

1. **Does this tree undergo seasonal leaf shed?**  
   → Binary label `is_deciduous` + continuous `deciduousness_score ∈ [0, 1]`

2. **When does the transition happen?**  
   → Per-OM phenophase label (`leaf_on` / `transitioning` / `leaf_off` / `stable`)  
   → Key event OMs: `leaf_off_start_om`, `full_leaf_off_om`, `leaf_on_return_om`

**Method — multi-signal scoring**:

| Component | Signal | Intuition |
|---|---|---|
| `s_veg_amp` | `veg_fraction_hsv` amplitude | Deciduous trees show a large seasonal swing in vegetation fraction |
| `s_depth` | min `veg_fraction_hsv` | Leaf-off trees have very low veg fraction (bare bark/branches visible) |
| `s_gcc_amp` | `gcc_mean` amplitude | Green chromatic coordinate drops at senescence |
| `s_tex` | `laplacian_var` amplitude | Bark texture differs from leafy canopy texture |

**Deciduousness score**:  `DS = 0.35·s_veg_amp + 0.30·s_depth + 0.25·s_gcc_amp + 0.10·s_tex`

**Phenophase state** uses per-tree min-max normalised `veg_fraction_hsv`:  
- `norm ≥ 0.65` → `leaf_on`   · `norm ≤ 0.35` → `leaf_off`   · between → `transitioning`  
- Evergreen trees (not deciduous) → all OMs labelled `stable`


In [23]:

import pandas as pd
import numpy as np

# ── Config (tune these to explore decision boundaries) ────────────────────────
VEG_MIN_THRESH = 0.45    # depth signal: raw veg below this → strong leaf-off evidence
SCORE_THRESH   = 0.70    # DS ≥ this → classified as deciduous

# Phenophase thresholds (on per-tree min-max normalised veg series)
LEAFON_NORM    = 0.65    # normalised veg ≥ this → leaf_on
LEAFOFF_NORM   = 0.35    # normalised veg ≤ this → leaf_off

STATE_COLORS = {
    'leaf_on':       '#4CAF50',   # green
    'transitioning': '#FFC107',   # amber
    'leaf_off':      '#795548',   # brown
    'stable':        '#29B6F6',   # light-blue (evergreen, no meaningful seasonal drop)
}

PRIMARY     = 'veg_fraction_hsv'   # main leaf-shed signal
SECONDARY   = 'gcc_mean'           # supporting colour signal
TEXTURE_SIG = 'laplacian_var'      # structural texture

for sig in [PRIMARY, SECONDARY]:
    if sig not in interp_series:
        raise RuntimeError(f"'{sig}' not in interp_series. Run the clustering setup cell first.")

# ── Cross-tree 90th-percentile amplitudes (for score normalisation) ───────────
_amps_veg = np.array([
    interp_series[PRIMARY][i].max() - interp_series[PRIMARY][i].min()
    for i in range(N_trees)
])
_amps_gcc = np.array([
    interp_series[SECONDARY][i].max() - interp_series[SECONDARY][i].min()
    for i in range(N_trees)
])
veg_amp_90 = float(np.nanpercentile(_amps_veg, 90)) or 1.0
gcc_amp_90 = float(np.nanpercentile(_amps_gcc, 90)) or 1.0

use_tex = TEXTURE_SIG in interp_series
if use_tex:
    _amps_tex  = np.array([
        interp_series[TEXTURE_SIG][i].max() - interp_series[TEXTURE_SIG][i].min()
        for i in range(N_trees)
    ])
    tex_amp_90 = float(np.nanpercentile(_amps_tex, 90)) or 1.0

# ── Index: chain_id → position in tree_ids_arr ───────────────────────────────
_tidx = {int(tid): i for i, tid in enumerate(tree_ids_arr)}

# ── Main loop ─────────────────────────────────────────────────────────────────
leaf_records = []

for i, tree_id in enumerate(tree_ids_arr):
    veg   = np.clip(interp_series[PRIMARY][i],   0, 1)    # raw veg_fraction (T,)
    gcc   = interp_series[SECONDARY][i]                    # raw gcc_mean     (T,)
    veg_n = normed_series[PRIMARY][i]                      # per-tree 0–1     (T,)

    veg_min = float(np.nanmin(veg))
    veg_max = float(np.nanmax(veg))
    veg_amp = veg_max - veg_min
    gcc_amp = float(np.nanmax(gcc) - np.nanmin(gcc))

    # ── Score components [0, 1] ───────────────────────────────────────────────
    s_veg_amp = min(1.0, veg_amp / veg_amp_90)
    s_gcc_amp = min(1.0, gcc_amp / gcc_amp_90)
    # depth: how far below VEG_MIN_THRESH does the min go?
    s_depth   = max(0.0, (VEG_MIN_THRESH - veg_min) / VEG_MIN_THRESH) \
                if veg_min < VEG_MIN_THRESH else 0.0
    s_tex     = min(1.0, _amps_tex[i] / tex_amp_90) if use_tex else 0.0

    if use_tex:
        DS = 0.35 * s_veg_amp + 0.30 * s_depth + 0.25 * s_gcc_amp + 0.10 * s_tex
    else:
        DS = 0.40 * s_veg_amp + 0.35 * s_depth + 0.25 * s_gcc_amp

    is_deciduous = bool(DS >= SCORE_THRESH)

    # ── Per-OM phenophase labels ───────────────────────────────────────────────
    om_states = []
    for vn in veg_n:
        if not is_deciduous:
            om_states.append('stable')          # evergreen: all OMs are "stable"
        elif vn >= LEAFON_NORM:
            om_states.append('leaf_on')
        elif vn <= LEAFOFF_NORM:
            om_states.append('leaf_off')
        else:
            om_states.append('transitioning')

    # ── Key event OMs (deciduous trees only) ─────────────────────────────────
    trough_t         = int(np.argmin(veg))
    peak_t           = int(np.argmax(veg))
    full_leaf_off_om = int(OM_IDS[trough_t])
    leaf_on_peak_om  = int(OM_IDS[peak_t])

    leaf_off_start_om = None
    leaf_on_return_om = None

    if is_deciduous:
        # leaf_off_start: OM right after the last leaf_on before the trough
        for t in range(trough_t, -1, -1):
            if om_states[t] == 'leaf_on':
                leaf_off_start_om = int(OM_IDS[min(t + 1, T - 1)])
                break
        # leaf_on_return: first leaf_on OM at or after the trough
        for t in range(trough_t, T):
            if om_states[t] == 'leaf_on':
                leaf_on_return_om = int(OM_IDS[t])
                break

    leaf_records.append({
        'chain_id':            int(tree_id),
        'deciduousness_score': round(DS,      4),
        'is_deciduous':        is_deciduous,
        'veg_amplitude_raw':   round(veg_amp, 4),
        'veg_min_raw':         round(veg_min, 4),
        'veg_max_raw':         round(veg_max, 4),
        'gcc_amplitude_raw':   round(gcc_amp, 4),
        'leaf_on_peak_om':     leaf_on_peak_om,
        'full_leaf_off_om':    full_leaf_off_om,
        'leaf_off_start_om':   leaf_off_start_om,
        'leaf_on_return_om':   leaf_on_return_om,
        'om_states':           om_states,
        's_veg_amp':           round(s_veg_amp, 4),
        's_depth':             round(s_depth,   4),
        's_gcc_amp':           round(s_gcc_amp, 4),
        's_tex':               round(s_tex,     4),
    })

leaf_shed_df = pd.DataFrame(leaf_records).set_index('chain_id')

n_dec  = int(leaf_shed_df['is_deciduous'].sum())
n_ever = len(leaf_shed_df) - n_dec

print(f"Leaf Shed Classification  (score threshold DS ≥ {SCORE_THRESH})")
print(f"  Total trees : {len(leaf_shed_df)}")
print(f"  Deciduous   : {n_dec}  ({100*n_dec/max(len(leaf_shed_df),1):.0f}%)")
print(f"  Evergreen   : {n_ever}  ({100*n_ever/max(len(leaf_shed_df),1):.0f}%)")
print()
print("Sorted by deciduousness score:")
display(
    leaf_shed_df[[
        'deciduousness_score', 'is_deciduous',
        'veg_amplitude_raw', 'veg_min_raw',
        'leaf_off_start_om', 'full_leaf_off_om', 'leaf_on_return_om',
    ]].sort_values('deciduousness_score', ascending=False).round(3)
)


Leaf Shed Classification  (score threshold DS ≥ 0.7)
  Total trees : 80
  Deciduous   : 30  (38%)
  Evergreen   : 50  (62%)

Sorted by deciduousness score:


,deciduousness_score,is_deciduous,veg_amplitude_raw,veg_min_raw,leaf_off_start_om,full_leaf_off_om,leaf_on_return_om
chain_id,,,,,,,
8,1.000,True,0.980,0.000,NaN,3,4.0
63,1.000,True,0.980,0.000,NaN,3,4.0
54,0.981,True,0.958,0.002,NaN,1,3.0
3,0.978,True,0.923,0.011,NaN,3,4.0
79,0.954,True,0.866,0.004,NaN,3,4.0
...,...,...,...,...,...,...,...
30,0.300,False,0.000,0.000,NaN,1,NaN
37,0.295,False,0.203,0.703,NaN,2,NaN
62,0.259,False,0.193,0.781,NaN,1,NaN


In [ ]:

# ── Score diagnostics: distribution + relationship to raw signals ─────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

scores   = leaf_shed_df['deciduousness_score'].values
dec_mask = leaf_shed_df['is_deciduous'].values

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

# ── Panel A: histogram of deciduousness scores ────────────────────────────────
ax = axes[0]
ax.hist(scores, bins=max(10, len(scores) // 3), color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(SCORE_THRESH, color='tomato', linewidth=2.2, linestyle='--',
           label=f'Threshold = {SCORE_THRESH}')
n_above = int((scores >= SCORE_THRESH).sum())
ax.text(SCORE_THRESH + 0.02, ax.get_ylim()[1] * 0.85,
        f'Deciduous\nn={n_above}', color='tomato', fontsize=9)
ax.text(SCORE_THRESH - 0.02, ax.get_ylim()[1] * 0.85,
        f'Evergreen\nn={len(scores)-n_above}', color='steelblue', fontsize=9, ha='right')
ax.set_xlabel('Deciduousness Score  DS', fontsize=10)
ax.set_ylabel('Number of trees', fontsize=10)
ax.set_title('Score Distribution', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# ── Panel B: DS vs raw veg_fraction amplitude ─────────────────────────────────
ax = axes[1]
ax.scatter(
    leaf_shed_df.loc[~dec_mask, 'veg_amplitude_raw'],
    leaf_shed_df.loc[~dec_mask, 'deciduousness_score'],
    c='#26A69A', edgecolors='black', linewidths=0.5, s=65, label='Evergreen', alpha=0.85, zorder=3,
)
ax.scatter(
    leaf_shed_df.loc[dec_mask, 'veg_amplitude_raw'],
    leaf_shed_df.loc[dec_mask, 'deciduousness_score'],
    c='#EF5350', edgecolors='black', linewidths=0.5, s=85, label='Deciduous', alpha=0.9, zorder=4,
)
for cid, row in leaf_shed_df.iterrows():
    ax.annotate(f'T{cid}', (row['veg_amplitude_raw'], row['deciduousness_score']),
                fontsize=6.5, alpha=0.75, xytext=(3, 2), textcoords='offset points')
ax.axhline(SCORE_THRESH, color='tomato', linewidth=1.2, linestyle='--', alpha=0.6)
ax.set_xlabel('Raw veg_fraction amplitude', fontsize=10)
ax.set_ylabel('Deciduousness Score  DS', fontsize=10)
ax.set_title('DS vs Veg Amplitude', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# ── Panel C: DS vs min raw veg_fraction (how bare does the canopy get?) ───────
ax = axes[2]
ax.scatter(
    leaf_shed_df.loc[~dec_mask, 'veg_min_raw'],
    leaf_shed_df.loc[~dec_mask, 'deciduousness_score'],
    c='#26A69A', edgecolors='black', linewidths=0.5, s=65, label='Evergreen', alpha=0.85, zorder=3,
)
ax.scatter(
    leaf_shed_df.loc[dec_mask, 'veg_min_raw'],
    leaf_shed_df.loc[dec_mask, 'deciduousness_score'],
    c='#EF5350', edgecolors='black', linewidths=0.5, s=85, label='Deciduous', alpha=0.9, zorder=4,
)
for cid, row in leaf_shed_df.iterrows():
    ax.annotate(f'T{cid}', (row['veg_min_raw'], row['deciduousness_score']),
                fontsize=6.5, alpha=0.75, xytext=(3, 2), textcoords='offset points')
ax.axhline(SCORE_THRESH, color='tomato', linewidth=1.2, linestyle='--', alpha=0.6)
ax.axvline(VEG_MIN_THRESH, color='darkorange', linewidth=1.2, linestyle=':',
           label=f'VEG_MIN_THRESH = {VEG_MIN_THRESH}', alpha=0.8)
ax.set_xlabel('Min raw veg_fraction  (how bare does canopy get?)', fontsize=10)
ax.set_ylabel('Deciduousness Score  DS', fontsize=10)
ax.set_title('DS vs Min Veg Fraction', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.suptitle('Leaf Shed Classifier — Score Diagnostics', fontsize=13, fontweight='bold')
plt.tight_layout()

score_diag_png = PHENO_OUTPUT_DIR / 'leafshed_score_diagnostics.png'
plt.savefig(score_diag_png, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved: {score_diag_png}')
print()
print('Score component summary:')
display(
    leaf_shed_df[['s_veg_amp', 's_depth', 's_gcc_amp', 's_tex', 'deciduousness_score']]
    .describe().round(3)
)


In [ ]:

# ── Compact time-series grid: all trees sorted by deciduousness score ─────────
# Each subplot: veg_fraction_hsv time series with phenophase background shading.
# Background columns coloured by per-OM phenophase state.
# Red/dashed key-event markers for deciduous trees.
# Title colour: red=deciduous, teal=evergreen.

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

N_COLS = 4
sorted_tids = leaf_shed_df.sort_values('deciduousness_score', ascending=False).index.tolist()
N_plot  = len(sorted_tids)
N_ROWS  = int(np.ceil(N_plot / N_COLS))

fig, axes = plt.subplots(
    N_ROWS, N_COLS,
    figsize=(N_COLS * 3.8, N_ROWS * 3.0),
    sharex=False, sharey=False,
)
axes = np.array(axes).reshape(N_ROWS, N_COLS)

STATE_ALPHA = 0.22

for idx, tree_id in enumerate(sorted_tids):
    row_i, col_i = divmod(idx, N_COLS)
    ax = axes[row_i, col_i]

    row    = leaf_shed_df.loc[tree_id]
    i      = _tidx[int(tree_id)]
    veg    = np.clip(interp_series[PRIMARY][i],   0, 1)
    gcc    = interp_series[SECONDARY][i]
    states = row['om_states']
    DS     = row['deciduousness_score']
    label  = 'DECIDUOUS' if row['is_deciduous'] else 'EVERGREEN'
    tc     = '#EF5350' if row['is_deciduous'] else '#26A69A'

    # Phenophase background shading
    for t_idx, (om_id, state) in enumerate(zip(OM_IDS, states)):
        ax.axvspan(om_id - 0.48, om_id + 0.48,
                   color=STATE_COLORS[state], alpha=STATE_ALPHA, linewidth=0)

    # veg_fraction line
    ax.plot(OM_IDS, veg, 'o-', color='#388E3C', linewidth=2.0, markersize=5,
            label='veg_fraction', zorder=3)
    # gcc_mean dashed
    ax.plot(OM_IDS, gcc, 's--', color='#1565C0', linewidth=1.2, markersize=3,
            alpha=0.7, label='gcc_mean', zorder=2)

    # Min-veg threshold reference
    ax.axhline(VEG_MIN_THRESH, color='saddlebrown', linewidth=0.9, linestyle=':',
               alpha=0.65)

    # Key event markers for deciduous trees
    if row['is_deciduous']:
        events = {
            'leaf_off_start_om':  ('#FF8F00', '▼', 'off-start'),
            'full_leaf_off_om':   ('#4E342E', '★', 'full-off'),
            'leaf_on_return_om':  ('#1B5E20', '▲', 'on-return'),
        }
        for field, (ec, mk, elabel) in events.items():
            val = row[field]
            if val is not None and not (isinstance(val, float) and np.isnan(val)):
                ax.axvline(int(val), color=ec, linewidth=1.6, linestyle='--', alpha=0.85)
                ax.text(int(val), 0.98, mk, ha='center', va='top',
                        fontsize=9, color=ec, transform=ax.get_xaxis_transform())

    ax.set_xlim(OM_IDS[0] - 0.6, OM_IDS[-1] + 0.6)
    ax.set_ylim(-0.02, 1.08)
    ax.set_xticks(OM_IDS)
    ax.set_xticklabels([f'OM{o}' for o in OM_IDS], fontsize=7)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.tick_params(axis='y', labelsize=7)
    ax.grid(alpha=0.2)
    ax.set_title(f'T{tree_id}  {label}\nDS={DS:.3f}', fontsize=8.5,
                 fontweight='bold', color=tc, pad=3)

    if col_i == 0:
        ax.set_ylabel('veg / gcc', fontsize=7.5)

# Turn off unused axes
for idx in range(N_plot, N_ROWS * N_COLS):
    row_i, col_i = divmod(idx, N_COLS)
    axes[row_i, col_i].axis('off')

# Shared legend
state_patches  = [mpatches.Patch(color=v, label=k.replace('_', ' '), alpha=0.65)
                  for k, v in STATE_COLORS.items()]
line_green     = plt.Line2D([], [], color='#388E3C', linewidth=2, label='veg_fraction')
line_blue      = plt.Line2D([], [], color='#1565C0', linewidth=1.5, linestyle='--', label='gcc_mean')
marker_patches = [
    plt.Line2D([], [], color='#FF8F00', linestyle='--', linewidth=1.5, label='▼ leaf-off start'),
    plt.Line2D([], [], color='#4E342E', linestyle='--', linewidth=1.5, label='★ full leaf-off'),
    plt.Line2D([], [], color='#1B5E20', linestyle='--', linewidth=1.5, label='▲ leaf-on return'),
]
fig.legend(
    handles=state_patches + [line_green, line_blue] + marker_patches,
    loc='lower center', ncol=len(state_patches) + 5,
    fontsize=8, framealpha=0.9,
    bbox_to_anchor=(0.5, -0.02),
)

fig.suptitle(
    f'Per-Tree Phenology Time Series — Leaf Shed Classification\n'
    f'(sorted by deciduousness score ↓ · background = phenophase state)',
    fontsize=12, fontweight='bold', y=1.01,
)
plt.tight_layout()

ts_grid_png = PHENO_OUTPUT_DIR / 'leafshed_timeseries_all_trees.png'
plt.savefig(ts_grid_png, dpi=160, bbox_inches='tight')
plt.show()
print(f'Saved: {ts_grid_png}')


In [ ]:


# -- Spatial maps: deciduousness score + binary classification -----------------
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd

geoms_ls, cids_ls, scores_ls, dec_ls = [], [], [], []
for cid, geom in consensus_polys.items():
    if geom is None or geom.is_empty:
        continue
    cid_int = int(cid)
    if cid_int not in leaf_shed_df.index:
        continue
    geoms_ls.append(geom)
    cids_ls.append(cid_int)
    scores_ls.append(float(leaf_shed_df.loc[cid_int, "deciduousness_score"]))
    dec_ls.append(bool(leaf_shed_df.loc[cid_int, "is_deciduous"]))

leaf_spatial_gdf = gpd.GeoDataFrame(
    {
        "chain_id": cids_ls,
        "deciduousness_score": scores_ls,
        "is_deciduous": dec_ls,
        "geometry": geoms_ls,
    },
    crs=tracker.crowns_gdfs[tracker.om_ids[0]].crs,
)

fig, axes = plt.subplots(1, 2, figsize=(20, 9))

# -- Left: continuous deciduousness score --------------------------------------
ax = axes[0]
leaf_spatial_gdf.plot(
    ax=ax,
    column="deciduousness_score",
    cmap="RdYlGn_r",
    vmin=0,
    vmax=1,
    edgecolor="black",
    linewidth=0.6,
    legend=True,
    legend_kwds={"label": "Deciduousness Score", "shrink": 0.7},
)
for _, row in leaf_spatial_gdf.iterrows():
    c = row.geometry.centroid
    ax.text(
        c.x,
        c.y,
        f"T{int(row['chain_id'])}\n{row['deciduousness_score']:.2f}",
        ha="center",
        va="center",
        fontsize=7.5,
        bbox=dict(boxstyle="round,pad=0.15", facecolor="white", alpha=0.75, edgecolor="none"),
    )

ax.set_title("Deciduousness Score (0 = evergreen <-> 1 = deciduous)", fontsize=11, fontweight="bold")
ax.set_aspect("equal")
ax.grid(alpha=0.2)

# -- Right: binary deciduous / evergreen ---------------------------------------
ax = axes[1]
dec_gdf = leaf_spatial_gdf[leaf_spatial_gdf["is_deciduous"]]
ever_gdf = leaf_spatial_gdf[~leaf_spatial_gdf["is_deciduous"]]

if not ever_gdf.empty:
    ever_gdf.plot(
        ax=ax, color="#26A69A", edgecolor="black", linewidth=0.7, alpha=0.8, label="Evergreen"
    )
if not dec_gdf.empty:
    dec_gdf.plot(
        ax=ax, color="#EF5350", edgecolor="black", linewidth=0.7, alpha=0.8, label="Deciduous"
    )

def _om_label(val):
    return int(val) if pd.notna(val) else "?"

# Annotate deciduous trees with their leaf-off window
for _, row in leaf_spatial_gdf.iterrows():
    cid = int(row["chain_id"])
    c = row.geometry.centroid
    lf_row = leaf_shed_df.loc[cid]

    if row["is_deciduous"]:
        start = _om_label(lf_row["leaf_off_start_om"])
        end = _om_label(lf_row["leaf_on_return_om"])
        ann = f"T{cid}\noff:OM{start}->OM{end}"
    else:
        ann = f"T{cid}"

    ax.text(
        c.x,
        c.y,
        ann,
        ha="center",
        va="center",
        fontsize=7,
        fontweight="bold",
        color="white",
        bbox=dict(
            boxstyle="round,pad=0.15",
            facecolor="#EF5350" if row["is_deciduous"] else "#26A69A",
            alpha=0.85,
            edgecolor="none",
        ),
    )

legend_patches = [
    mpatches.Patch(color="#EF5350", label=f"Deciduous (n={len(dec_gdf)})"),
    mpatches.Patch(color="#26A69A", label=f"Evergreen (n={len(ever_gdf)})"),
]
ax.legend(handles=legend_patches, loc="upper right", fontsize=10)
ax.set_title(
    f"Leaf Shed Classification (DS threshold = {SCORE_THRESH})\n"
    "Deciduous labels show estimated leaf-off window",
    fontsize=11,
    fontweight="bold",
)
ax.set_aspect("equal")
ax.grid(alpha=0.2)

plt.suptitle("Spatial Distribution - Leaf Shed Classifier", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()

spatial_png = PHENO_OUTPUT_DIR / "leafshed_spatial_map.png"
plt.savefig(spatial_png, dpi=180, bbox_inches="tight")
plt.show()
print(f"Saved: {spatial_png}")

In [ ]:

# ── Deciduous tree image gallery ──────────────────────────────────────────────
# Crown chips for every deciduous tree × every OM.
# Border colour = phenophase state (green / amber / brown).
# Title at top of each column shows OM index.
# Row label shows tree ID + deciduousness score.
# The images make the classification interpretable at a glance.

import numpy as np
import matplotlib.pyplot as plt

CHIP_SIZE  = 96
dec_tids   = leaf_shed_df[leaf_shed_df['is_deciduous']].sort_values(
    'deciduousness_score', ascending=False
).index.tolist()

def _safe_chip(om_id, poly, size=CHIP_SIZE):
    """Extract, clip, and resize a crown image chip."""
    if poly is None:
        return None
    try:
        patch = extract_patch_for_polygon(om_id, poly, tracker)
    except Exception:
        return None
    if patch is None or not isinstance(patch, np.ndarray) or patch.size == 0:
        return None
    if patch.ndim == 3 and patch.shape[0] in (1, 3, 4):
        patch = np.moveaxis(patch[:3], 0, -1)
    rgb = patch[..., :3].astype(float)
    lo, hi = np.percentile(rgb, [2, 98])
    rgb = np.clip((rgb - lo) / (hi - lo + 1e-6), 0, 1)
    chip = (rgb * 255).astype(np.uint8)
    try:
        import cv2
        return cv2.resize(chip, (size, size), interpolation=cv2.INTER_LINEAR)
    except Exception:
        return chip

if not dec_tids:
    print("No trees classified as deciduous. Lower SCORE_THRESH to include borderline trees.")
else:
    n_oms  = len(OM_IDS)
    n_rows = len(dec_tids)
    BORDER = 3   # px border per cell showing state colour

    fig, axes = plt.subplots(
        n_rows, n_oms,
        figsize=(n_oms * 2.0, n_rows * 2.4),
        squeeze=False,
        gridspec_kw={'hspace': 0.06, 'wspace': 0.06},
    )
    BG = '#111122'
    fig.patch.set_facecolor(BG)

    for r, tree_id in enumerate(dec_tids):
        lf_row = leaf_shed_df.loc[tree_id]
        states = lf_row['om_states']
        poly   = consensus_polys.get(int(tree_id))

        # Pre-extract event OM ids safely (None if not detected)
        def _event_om(val):
            return int(val) if pd.notna(val) else None
        off_start = _event_om(lf_row['leaf_off_start_om'])
        full_off  = _event_om(lf_row['full_leaf_off_om'])
        on_return = _event_om(lf_row['leaf_on_return_om'])

        for c, (om_id, state) in enumerate(zip(OM_IDS, states)):
            ax = axes[r, c]
            ax.set_facecolor(BG)
            ax.set_xticks([]); ax.set_yticks([])

            border_c = STATE_COLORS[state]
            for sp in ax.spines.values():
                sp.set_edgecolor(border_c)
                sp.set_linewidth(BORDER)

            chip = _safe_chip(om_id, poly)
            if chip is not None:
                ax.imshow(chip)
            else:
                ax.text(0.5, 0.5, '✗', ha='center', va='center',
                        color='#888', fontsize=14, transform=ax.transAxes)

            # Small state badge at bottom
            badge = state.replace('leaf_', '').replace('_', '\n')
            ax.text(0.5, 0.02, badge, ha='center', va='bottom',
                    fontsize=6.5, transform=ax.transAxes, color='white',
                    bbox=dict(boxstyle='round,pad=0.18', facecolor=border_c,
                              alpha=0.78, edgecolor='none'))

            # Key event markers in column header row
            if r == 0:
                header_label = f'OM{om_id}'
                is_key = False
                if off_start == om_id:
                    header_label += '\n▼start'
                    is_key = True
                elif full_off == om_id:
                    header_label += '\n★full-off'
                    is_key = True
                elif on_return == om_id:
                    header_label += '\n▲return'
                    is_key = True
                ax.set_title(header_label, color='white', fontsize=7.5, pad=4)

        # Row label
        axes[r, 0].set_ylabel(
            f'T{tree_id}\nDS={lf_row["deciduousness_score"]:.2f}',
            rotation=0, labelpad=60, va='center',
            color='white', fontsize=9, fontweight='bold',
        )

    fig.suptitle(
        'Deciduous Trees — Crown Images at Each OM\n'
        '(border = phenophase  ■ green=leaf_on  ■ amber=transitioning  ■ brown=leaf_off)',
        color='white', fontsize=11, fontweight='bold', y=1.01,
    )
    dec_gallery_png = PHENO_OUTPUT_DIR / 'leafshed_deciduous_gallery.png'
    plt.savefig(dec_gallery_png, dpi=140, bbox_inches='tight', facecolor=BG)
    plt.show()
    print(f'Saved: {dec_gallery_png}')


In [ ]:

# ── Phenophase calendar heatmap + save results ────────────────────────────────
# Heatmap: trees (rows, sorted by score) × OM dates (columns).
# Cell colour = phenophase state.  Shows population-level phenology at a glance.

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, BoundaryNorm
import pandas as pd, numpy as np

# ── Build state matrix ────────────────────────────────────────────────────────
STATE_ORDER = ['leaf_on', 'transitioning', 'leaf_off', 'stable']
state_to_int = {s: i for i, s in enumerate(STATE_ORDER)}
CAL_COLORS   = [STATE_COLORS[s] for s in STATE_ORDER]

cal_tids = leaf_shed_df.sort_values('deciduousness_score', ascending=False).index.tolist()

cal_mat = np.full((len(cal_tids), len(OM_IDS)), np.nan)
for r, tree_id in enumerate(cal_tids):
    for c, om_id in enumerate(OM_IDS):
        state = leaf_shed_df.loc[tree_id, 'om_states'][c]
        cal_mat[r, c] = state_to_int[state]

cmap_cal = ListedColormap(CAL_COLORS)
bnorm    = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], cmap_cal.N)

fig, ax = plt.subplots(figsize=(len(OM_IDS) * 1.1 + 3, len(cal_tids) * 0.55 + 2))

im = ax.imshow(cal_mat, cmap=cmap_cal, norm=bnorm, aspect='auto', interpolation='nearest')

ax.set_xticks(range(len(OM_IDS)))
ax.set_xticklabels([f'OM{o}' for o in OM_IDS], fontsize=10)
ax.set_yticks(range(len(cal_tids)))
ax.set_yticklabels(
    [f"T{tid}  DS={leaf_shed_df.loc[tid,'deciduousness_score']:.2f}"
     f"  {'DEC' if leaf_shed_df.loc[tid,'is_deciduous'] else 'EVG'}"
     for tid in cal_tids],
    fontsize=9,
)

# Annotate cell text
for r, tree_id in enumerate(cal_tids):
    for c, om_id in enumerate(OM_IDS):
        state = leaf_shed_df.loc[tree_id, 'om_states'][c]
        abbr  = {'leaf_on': 'ON', 'transitioning': 'TRANS', 'leaf_off': 'OFF', 'stable': 'SBL'}
        tc    = 'black' if state in ('leaf_on', 'transitioning') else 'white'
        ax.text(c, r, abbr[state], ha='center', va='center', fontsize=8,
                fontweight='bold', color=tc)

# Legend
patches = [mpatches.Patch(color=STATE_COLORS[s], label=s.replace('_', ' '))
           for s in STATE_ORDER]
ax.legend(handles=patches, loc='upper right', bbox_to_anchor=(1.22, 1.05),
          fontsize=9, framealpha=0.9)

# Horizontal separator between deciduous and evergreen
n_dec_cal = int(leaf_shed_df.loc[cal_tids, 'is_deciduous'].sum())
if 0 < n_dec_cal < len(cal_tids):
    ax.axhline(n_dec_cal - 0.5, color='white', linewidth=2.5)
    ax.text(len(OM_IDS) - 0.5, n_dec_cal - 0.55,
            '▲ Deciduous  |  Evergreen ▼', ha='right', va='bottom',
            fontsize=9, color='white', fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='black', alpha=0.55))

ax.set_title('Phenophase Calendar — All Trees × OM Dates\n'
             '(sorted by deciduousness score, highest at top)',
             fontsize=12, fontweight='bold', pad=12)
ax.set_xlabel('Orthomosaic Date', fontsize=10)

plt.tight_layout()
cal_png = PHENO_OUTPUT_DIR / 'leafshed_phenophase_calendar.png'
plt.savefig(cal_png, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved: {cal_png}')

# ── Print population summary ──────────────────────────────────────────────────
print('\nPhenophase population counts by OM date:')
state_rows = []
for tree_id, lf_row in leaf_shed_df.iterrows():
    for om_id, state in zip(OM_IDS, lf_row['om_states']):
        state_rows.append({
            'chain_id': int(tree_id), 'om_id': int(om_id),
            'phenophase': state, 'is_deciduous': bool(lf_row['is_deciduous']),
            'deciduousness_score': round(lf_row['deciduousness_score'], 4),
        })
state_long_df = pd.DataFrame(state_rows)
display(
    state_long_df.groupby(['om_id', 'phenophase'])['chain_id']
    .count().unstack(fill_value=0)
)

# ── Save CSV outputs ──────────────────────────────────────────────────────────
summary_out = leaf_shed_df.drop(columns=['om_states']).reset_index()
summary_path = PHENO_OUTPUT_DIR / 'leaf_shed_classification.csv'
summary_out.to_csv(summary_path, index=False)

state_path = PHENO_OUTPUT_DIR / 'leaf_shed_phenophase_per_om.csv'
state_long_df.to_csv(state_path, index=False)

print(f'\nSaved per-tree summary   : {summary_path}')
print(f'Saved per-tree per-OM states: {state_path}')
print(f'\nDeciduous : {n_dec}  |  Evergreen : {n_ever}')
